# TraceGraph — Temporal Graph Intelligence for AML Detection

TraceGraph is a temporal graph-based Anti-Money Laundering (AML) investigation-support system that models financial transactions as directed temporal networks.

The system combines transaction-level, account-level, graph-structural, and temporal features to identify high-risk transaction activity and support investigation through bidirectional tracing and suspicious network-pattern detection.

## Project Objectives

- Model financial transactions as temporal directed graphs.
- Extract transaction, account, graph, and temporal features.
- Evaluate classical machine-learning and deep-learning approaches.
- Generate transaction-level risk scores.
- Trace suspicious activity through incoming and outgoing transaction paths.
- Detect suspicious fan-in, rapid-hop, reverse-funding, forward-flow, and temporal-cycle patterns.
- Provide investigation-oriented outputs through a Streamlit dashboard.

## Dataset

The project is evaluated using the PublicAMLSimData / AMLSim transaction dataset.

The evaluated dataset contains:

- 17,144 transactions
- 100 accounts
- 18 fraud-labelled transactions
- 18 alerts

## Experimental Pipeline

Dataset Validation
→ Graph Construction
→ Temporal Modeling
→ Multi-Channel Graph Representation
→ Baseline ML
→ CNN / ResNet
→ Hybrid Graph + Transaction Features
→ Risk Scoring
→ Bidirectional Tracing
→ Cycle Detection
→ Investigation Reports
→ Streamlit Dashboard
→ Final Evaluation

## Important Note

TraceGraph is designed as an AML investigation-support and risk-prioritization system. A high-risk score is an investigation signal and does not by itself establish that a transaction is fraudulent.


# TraceGraph — Temporal Graph Intelligence for AML Detection

A graph-based machine learning system for detecting suspicious
transaction patterns and enabling explainable money-flow tracing.

## Project Goals

- Represent financial transactions as temporal graphs
- Detect suspicious transaction topologies
- Analyze transactions across multiple time windows
- Build multi-channel adjacency matrices
- Train deep-learning models for risk scoring
- Perform forward and backward transaction tracing
- Provide an investigator-friendly dashboard

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn networkx plotly pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import sklearn
import networkx as nx
import plotly

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NetworkX:", nx.__version__)
print("Plotly:", plotly.__version__)

print("\nAll basic libraries imported successfully! ✅")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

print(os.listdir('/content/drive/MyDrive')[:20])

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

folders = [
    "data/raw",
    "data/processed",
    "data/samples",
    "models",
    "results",
    "notebooks"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

print("TraceGraph project structure created! ✅")
print(PROJECT_DIR)

In [ ]:
for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

In [ ]:
import sys
import torch
import numpy as np
import pandas as pd
import sklearn
import networkx as nx

print("=" * 50)
print("TRACEGRAPH ENVIRONMENT")
print("=" * 50)

print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("Scikit-learn :", sklearn.__version__)
print("NetworkX     :", nx.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("CUDA version :", torch.version.cuda)

print("=" * 50)

In [ ]:
!git clone --depth 1 https://github.com/IBM/AMLSim.git /content/AMLSim

In [ ]:
import os

print(os.listdir("/content/AMLSim"))

In [ ]:
import os
import shutil

# Copy AMLSim's 1K parameter dataset into our project
source = "/content/AMLSim/paramFiles/1K"
destination = "/content/drive/MyDrive/TraceGraph/data/raw/amlsim_1k"

shutil.copytree(source, destination, dirs_exist_ok=True)

print("AMLSim 1K dataset copied successfully! ✅")
print("Location:", destination)

print("\nFiles:")
for file in os.listdir(destination):
    print(" -", file)

In [ ]:
import pandas as pd

RAW_DIR = "/content/drive/MyDrive/TraceGraph/data/raw/amlsim_1k"

patterns = pd.read_csv(f"{RAW_DIR}/alertPatterns.csv")

print("Shape:", patterns.shape)
print("\nColumns:")
print(patterns.columns.tolist())

print("\nAML Patterns:")
display(patterns)

In [ ]:
accounts = pd.read_csv(f"{RAW_DIR}/accounts.csv")

print("Shape:", accounts.shape)

print("\nColumns:")
print(accounts.columns.tolist())

print("\nFirst 10 accounts:")
display(accounts.head(10))

print("\nData types:")
print(accounts.dtypes)

In [ ]:
conf = pd.read_json(f"{RAW_DIR}/conf.json")

print("Configuration:")
display(conf)

In [ ]:
import os

print("AMLSim directory:")
print("/content/AMLSim")

print("\nAvailable sample/config files:")
for root, dirs, files in os.walk("/content/AMLSim"):
    for file in files:
        if file.endswith(".py") or file.endswith(".jar"):
            print(os.path.join(root, file))

In [ ]:
!python /content/AMLSim/scripts/transaction_graph_generator.py --help

In [ ]:
!tail -n 80 /content/AMLSim/scripts/transaction_graph_generator.py

In [ ]:
!python /content/AMLSim/scripts/transaction_graph_generator.py \
"/content/drive/MyDrive/TraceGraph/data/raw/amlsim_1k/conf.json"

In [ ]:
import json

CONF_FILE = "/content/drive/MyDrive/TraceGraph/data/raw/amlsim_1k/conf.json"

with open(CONF_FILE, "r") as f:
    conf = json.load(f)

print("Top-level sections:")
print(list(conf.keys()))

print("\nInput configuration:")
print(json.dumps(conf.get("input", {}), indent=2))

In [ ]:
import json

ORIGINAL_CONF = "/content/AMLSim/paramFiles/1K/conf.json"

with open(ORIGINAL_CONF, "r") as f:
    original_conf = json.load(f)

print("Original AMLSim input configuration:")
print(json.dumps(original_conf.get("input", {}), indent=2))

In [ ]:
%cd /content/AMLSim

!git log -1 --oneline
!git remote -v


In [ ]:
import os

print("Searching AMLSim for transaction datasets...\n")

for root, dirs, files in os.walk("/content/AMLSim"):
    for file in files:
        if file in [
            "transactions.csv",
            "alert_transactions.csv",
            "alert_accounts.csv",
            "sar_accounts.csv"
        ]:
            print(os.path.join(root, file))

In [ ]:
!grep -R '"normal_models"\|normal_models' /content/AMLSim/scripts /content/AMLSim/paramFiles/1K -n

In [ ]:
import os

NORMAL_MODELS = "/content/AMLSim/paramFiles/1K/normal_models.csv"

print("Exists:", os.path.exists(NORMAL_MODELS))

if os.path.exists(NORMAL_MODELS):
    print("Size:", os.path.getsize(NORMAL_MODELS), "bytes")
    print("\nFirst 10 lines:")
    with open(NORMAL_MODELS, "r") as f:
        for i in range(10):
            line = f.readline()
            if not line:
                break
            print(line.rstrip())

In [ ]:
!find /content/AMLSim -type f -iname "*normal*model*" -o -iname "normal_models.csv"

In [ ]:
import shutil
import os

source = "/content/AMLSim/paramFiles/1K/normalModels.csv"
destination = "/content/drive/MyDrive/TraceGraph/data/raw/amlsim_1k/normal_models.csv"

shutil.copy2(source, destination)

print("Copied successfully! ✅")
print("Destination:", destination)
print("Exists:", os.path.exists(destination))

In [ ]:
!python /content/AMLSim/scripts/transaction_graph_generator.py \
"/content/drive/MyDrive/TraceGraph/data/raw/amlsim_1k/conf.json"

In [ ]:
import json

CONF_FILE = "/content/drive/MyDrive/TraceGraph/data/raw/amlsim_1k/conf.json"

with open(CONF_FILE, "r") as f:
    conf = json.load(f)

# Add the normal model definition expected by the generator
conf["input"]["normal_models"] = "normal_models.csv"

with open(CONF_FILE, "w") as f:
    json.dump(conf, f, indent=2)

print("Updated conf.json successfully! ✅")
print("\nInput configuration:")
print(json.dumps(conf["input"], indent=2))

In [ ]:
!python /content/AMLSim/scripts/transaction_graph_generator.py \
"/content/drive/MyDrive/TraceGraph/data/raw/amlsim_1k/conf.json"

In [ ]:
import pandas as pd

BASE = "/content/AMLSim/paramFiles/1K"

normal_models = pd.read_csv(f"{BASE}/normalModels.csv")
accounts_config = pd.read_csv(f"{BASE}/accounts.csv")

print("=== normalModels.csv ===")
print("Shape:", normal_models.shape)
display(normal_models.head(10))

print("\n=== accounts.csv ===")
print("Shape:", accounts_config.shape)
display(accounts_config)

In [ ]:
!sed -n '280,330p' /content/AMLSim/scripts/transaction_graph_generator.py

In [ ]:
!sed -n '470,570p' /content/AMLSim/scripts/transaction_graph_generator.py

In [ ]:
degree_file = "/content/AMLSim/paramFiles/1K/degree.csv"

degree = pd.read_csv(degree_file)

print("Shape:", degree.shape)
print("\nColumns:")
print(degree.columns.tolist())

print("\nFirst 20 rows:")
display(degree.head(20))

In [ ]:
import sys

sys.path.insert(0, "/content/AMLSim/scripts")

from amlsim import *

print("Testing AMLSim degree generation...")

degree_file = "/content/AMLSim/paramFiles/1K/degree.csv"

in_deg, out_deg = get_degrees(degree_file, 1000)

print("Number of accounts:", len(in_deg))
print("In-degree sum:", sum(in_deg))
print("Out-degree sum:", sum(out_deg))
print("First 10 in-degrees:", in_deg[:10])
print("First 10 out-degrees:", out_deg[:10])

In [ ]:
import sys

sys.path.insert(0, "/content/AMLSim/scripts")

from transaction_graph_generator import get_degrees

degree_file = "/content/AMLSim/paramFiles/1K/degree.csv"

in_deg, out_deg = get_degrees(degree_file, 1000)

print("Number of accounts:", len(in_deg))
print("In-degree sum:", sum(in_deg))
print("Out-degree sum:", sum(out_deg))
print("First 10 in-degrees:", in_deg[:10])
print("First 10 out-degrees:", out_deg[:10])


In [ ]:
import pandas as pd

accounts_file = "/content/AMLSim/paramFiles/1K/accounts.csv"

accounts = pd.read_csv(accounts_file)

print("Account configuration:")
display(accounts)

print("\nTotal configured accounts:")
print(accounts["count"].sum())

In [ ]:
degree_file = "/content/AMLSim/paramFiles/1K/degree.csv"

in_deg, out_deg = get_degrees(degree_file, 1446)

print("Number of accounts:", len(in_deg))
print("In-degree sum:", sum(in_deg))
print("Out-degree sum:", sum(out_deg))
print("First 10 in-degrees:", in_deg[:10])
print("First 10 out-degrees:", out_deg[:10])

In [ ]:
!python /content/AMLSim/scripts/transaction_graph_generator.py \
"/content/drive/MyDrive/TraceGraph/data/raw/amlsim_1k/conf.json" \
"1K"

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print("Uploaded files:")
print(os.listdir("/content"))

In [ ]:
!wget -O /content/100vertices-10Kedges.7z \
"https://www.dropbox.com/scl/fo/7g35w7wk7gglve627we3k/AHjP6pnCmV8M62L7RxTFtkU?rlkey=6ksx339ac9117onfx3l0g3fji&dl=1"

In [ ]:
import zipfile

ARCHIVE = "/content/100vertices-10Kedges.7z"

with zipfile.ZipFile(ARCHIVE, "r") as z:
    files = z.namelist()

print("Number of files:", len(files))

print("\nFirst 50 files:")
for f in files[:50]:
    print(f)

In [ ]:
import zipfile

ARCHIVE = "/content/100vertices-10Kedges.7z"

with zipfile.ZipFile(ARCHIVE, "r") as z:
    matches = [
        f for f in z.namelist()
        if f.endswith("100vertices-10Kedges.7z")
    ]

print("Matching dataset files:")
for f in matches:
    print(f)

In [ ]:
import zipfile
import os

ARCHIVE = "/content/100vertices-10Kedges.7z"
EXTRACT_DIR = "/content/TraceGraph_dataset"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ARCHIVE, "r") as z:
    z.extract("100vertices-10Kedges.7z", EXTRACT_DIR)

print("Extracted to:")
print(os.path.join(EXTRACT_DIR, "100vertices-10Kedges.7z"))

print("Size:")
print(
    round(
        os.path.getsize(
            os.path.join(EXTRACT_DIR, "100vertices-10Kedges.7z")
        ) / (1024**2),
        2
    ),
    "MB"
)

In [ ]:
import subprocess

DATASET = "/content/TraceGraph_dataset/100vertices-10Kedges.7z"

result = subprocess.run(
    ["7z", "l", DATASET],
    capture_output=True,
    text=True
)

print(result.stdout)

In [ ]:
import subprocess
import os

DATASET = "/content/TraceGraph_dataset/100vertices-10Kedges.7z"
OUTPUT_DIR = "/content/TraceGraph_dataset/extracted"

os.makedirs(OUTPUT_DIR, exist_ok=True)

result = subprocess.run(
    ["7z", "x", DATASET, f"-o{OUTPUT_DIR}", "-y"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode == 0:
    print("\n✅ Dataset extracted successfully.")
else:
    print("\n❌ Extraction failed:")
    print(result.stderr)

In [ ]:
import os

EXTRACTED_DIR = "/content/TraceGraph_dataset/extracted/100vertices-10Kedges"

for root, dirs, files in os.walk(EXTRACTED_DIR):
    level = root.replace(EXTRACTED_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        print(f"{indent}  {file} ({size:,} bytes)")

In [ ]:
import pandas as pd

BASE_DIR = "/content/TraceGraph_dataset/extracted/100vertices-10Kedges/"

transactions_path = BASE_DIR + "transactions.csv"

transactions = pd.read_csv(transactions_path)

print("Shape:", transactions.shape)

print("\nColumns:")
print(transactions.columns.tolist())

print("\nFirst 5 rows:")
display(transactions.head())

print("\nData types:")
print(transactions.dtypes)

In [ ]:

print("=== Transaction Types ===")
print(transactions["TX_TYPE"].value_counts())

print("\n=== Fraud Labels ===")
print(transactions["IS_FRAUD"].value_counts())

print("\n=== ALERT_ID ===")
print(transactions["ALERT_ID"].value_counts().head(20))

print("\n=== Timestamp Range ===")
print("Min:", transactions["TIMESTAMP"].min())
print("Max:", transactions["TIMESTAMP"].max())

print("\n=== Transaction Amount ===")
print(transactions["TX_AMOUNT"].describe())

In [ ]:
fraud_tx = transactions[transactions["IS_FRAUD"] == True].copy()

print("Number of fraud transactions:", len(fraud_tx))

print("\nFraud transactions:")
display(fraud_tx.sort_values(["ALERT_ID", "TIMESTAMP"]))

print("\nFraud transactions by ALERT_ID:")
print(
    fraud_tx["ALERT_ID"]
    .value_counts()
    .sort_index()
)

In [ ]:
alerts_path = BASE_DIR + "alerts.csv"

alerts = pd.read_csv(alerts_path)

print("Shape:", alerts.shape)

print("\nColumns:")
print(alerts.columns.tolist())

print("\nAlerts:")
display(alerts)

print("\nData types:")
print(alerts.dtypes)

In [ ]:
accounts_path = BASE_DIR + "accounts.csv"

accounts = pd.read_csv(accounts_path)

print("Shape:", accounts.shape)

print("\nColumns:")
print(accounts.columns.tolist())

print("\nFirst 10 accounts:")
display(accounts.head(10))

print("\nData types:")
print(accounts.dtypes)

In [ ]:
print("=== Account Fraud Distribution ===")
print(accounts["IS_FRAUD"].value_counts())

print("\n=== Fraudulent Accounts ===")
fraud_accounts = accounts[accounts["IS_FRAUD"] == True]

display(
    fraud_accounts[
        [
            "ACCOUNT_ID",
            "CUSTOMER_ID",
            "INIT_BALANCE",
            "COUNTRY",
            "ACCOUNT_TYPE",
            "TX_BEHAVIOR_ID"
        ]
    ]
)

print("\nNumber of fraudulent accounts:", len(fraud_accounts))

In [ ]:
account_ids = set(accounts["ACCOUNT_ID"])

sender_ids = set(transactions["SENDER_ACCOUNT_ID"])
receiver_ids = set(transactions["RECEIVER_ACCOUNT_ID"])

missing_senders = sender_ids - account_ids
missing_receivers = receiver_ids - account_ids

print("Total accounts:", len(account_ids))
print("Unique sender accounts:", len(sender_ids))
print("Unique receiver accounts:", len(receiver_ids))

print("\nMissing sender accounts:", missing_senders)
print("Missing receiver accounts:", missing_receivers)

print("\nReferential integrity:",
      "PASS" if not missing_senders and not missing_receivers else "FAIL")

In [ ]:
print("Total transaction rows:", len(transactions))
print("Unique TX_IDs:", transactions["TX_ID"].nunique())

duplicate_tx_ids = transactions[
    transactions["TX_ID"].duplicated(keep=False)
].sort_values("TX_ID")

print("\nDuplicate TX_ID rows:", len(duplicate_tx_ids))

if len(duplicate_tx_ids) > 0:
    display(duplicate_tx_ids.head(20))
else:
    print("No duplicate TX_IDs found.")

print(
    "\nTX_ID uniqueness:",
    "PASS" if transactions["TX_ID"].is_unique else "FAIL"
)

In [ ]:
print("=== TRANSACTIONS ===")
print(transactions.isnull().sum())

print("\n=== ACCOUNTS ===")
print(accounts.isnull().sum())

print("\n=== ALERTS ===")
print(alerts.isnull().sum())


In [ ]:
is_sorted = transactions["TIMESTAMP"].is_monotonic_increasing

print("Transactions sorted by TIMESTAMP:", is_sorted)

print("\nFirst 10 timestamps:")
print(transactions["TIMESTAMP"].head(10).tolist())

print("\nLast 10 timestamps:")
print(transactions["TIMESTAMP"].tail(10).tolist())

print("\nNumber of unique timestamps:")
print(transactions["TIMESTAMP"].nunique())


In [ ]:
time_counts = (
    transactions
    .groupby("TIMESTAMP")
    .size()
    .reset_index(name="TRANSACTION_COUNT")
)

print("Number of time points:", len(time_counts))
print("\nFirst 10 time points:")
display(time_counts.head(10))

print("\nLast 10 time points:")
display(time_counts.tail(10))

print("\nTransaction count statistics per timestamp:")
print(time_counts["TRANSACTION_COUNT"].describe())

In [ ]:
amount_by_fraud = (
    transactions
    .groupby("IS_FRAUD")["TX_AMOUNT"]
    .agg(["count", "mean", "median", "min", "max"])
)

print(amount_by_fraud)

print("\nFraud transaction amounts:")
print(
    transactions.loc[
        transactions["IS_FRAUD"] == True,
        "TX_AMOUNT"
    ].tolist()
)


In [ ]:
tx_order_check = transactions[["TX_ID", "TIMESTAMP"]].copy()

print(
    "TX_ID monotonically increasing:",
    tx_order_check["TX_ID"].is_monotonic_increasing
)

print(
    "TX_ID unique:",
    tx_order_check["TX_ID"].is_unique
)

print("\nTransactions around the first fraud alert:")
display(
    transactions[
        transactions["ALERT_ID"] != -1
    ][["TX_ID", "TIMESTAMP", "ALERT_ID"]].head(10)
)

In [ ]:
# Create a working copy of the raw transaction data
tx = transactions.copy()

# Verify that the copy was created correctly
print("Shape:", tx.shape)
print("Columns:", tx.columns.tolist())

print("\nData types:")
print(tx.dtypes)

In [ ]:
# Enforce the expected data types
tx["TX_ID"] = tx["TX_ID"].astype(int)
tx["SENDER_ACCOUNT_ID"] = tx["SENDER_ACCOUNT_ID"].astype(int)
tx["RECEIVER_ACCOUNT_ID"] = tx["RECEIVER_ACCOUNT_ID"].astype(int)

tx["TX_AMOUNT"] = pd.to_numeric(tx["TX_AMOUNT"], errors="raise")
tx["TIMESTAMP"] = pd.to_numeric(tx["TIMESTAMP"], errors="raise").astype(int)

tx["IS_FRAUD"] = tx["IS_FRAUD"].astype(bool)
tx["ALERT_ID"] = tx["ALERT_ID"].astype(int)

tx["TX_TYPE"] = tx["TX_TYPE"].astype(str)

print(tx.dtypes)

In [ ]:
import numpy as np

print("=== Numeric validity ===")

print("TX_AMOUNT NaN:", tx["TX_AMOUNT"].isna().sum())
print("TX_AMOUNT infinite:", np.isinf(tx["TX_AMOUNT"]).sum())
print("TX_AMOUNT <= 0:", (tx["TX_AMOUNT"] <= 0).sum())

print("\nTIMESTAMP NaN:", tx["TIMESTAMP"].isna().sum())
print("TIMESTAMP < 0:", (tx["TIMESTAMP"] < 0).sum())

print("\nSENDER_ACCOUNT_ID < 0:",
      (tx["SENDER_ACCOUNT_ID"] < 0).sum())

print("RECEIVER_ACCOUNT_ID < 0:",
      (tx["RECEIVER_ACCOUNT_ID"] < 0).sum())

In [ ]:
self_transactions = tx[
    tx["SENDER_ACCOUNT_ID"] == tx["RECEIVER_ACCOUNT_ID"]
]

print("Number of self-transactions:", len(self_transactions))

if len(self_transactions) > 0:
    display(self_transactions)
else:
    print("No self-transactions found.")

In [ ]:
edge_keys = [
    "SENDER_ACCOUNT_ID",
    "RECEIVER_ACCOUNT_ID",
    "TIMESTAMP"
]

duplicate_edges = tx[
    tx.duplicated(subset=edge_keys, keep=False)
].sort_values(edge_keys)

print("Rows involved in duplicate sender-receiver-timestamp combinations:",
      len(duplicate_edges))

if len(duplicate_edges) > 0:
    print("\nNumber of unique duplicated combinations:",
          duplicate_edges[edge_keys].drop_duplicates().shape[0])

    display(duplicate_edges.head(20))
else:
    print("No duplicate sender-receiver-timestamp combinations found.")

In [ ]:
alert_tx = tx[tx["ALERT_ID"] != -1].copy()

alert_accounts = set(alert_tx["SENDER_ACCOUNT_ID"]) | set(
    alert_tx["RECEIVER_ACCOUNT_ID"]
)

fraud_account_ids = set(
    accounts.loc[accounts["IS_FRAUD"] == True, "ACCOUNT_ID"]
)

missing_fraud_labels = alert_accounts - fraud_account_ids

print("Unique accounts involved in alerts:", len(alert_accounts))
print("Accounts labeled fraudulent:", len(fraud_account_ids))

print("\nAlert accounts:")
print(sorted(alert_accounts))

print("\nAlert accounts without account-level fraud label:")
print(sorted(missing_fraud_labels))

print(
    "\nConsistency check:",
    "PASS" if not missing_fraud_labels else "CHECK REQUIRED"
)

In [ ]:
import os

PROCESSED_DIR = "/content/drive/MyDrive/TraceGraph/data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

TX_PROCESSED_PATH = os.path.join(
    PROCESSED_DIR,
    "transactions_validated.csv"
)

tx.to_csv(TX_PROCESSED_PATH, index=False)

print("Saved:", TX_PROCESSED_PATH)
print("Shape:", tx.shape)
print("File exists:", os.path.exists(TX_PROCESSED_PATH))

In [ ]:
ACCOUNTS_PROCESSED_PATH = os.path.join(
    PROCESSED_DIR,
    "accounts_validated.csv"
)

ALERTS_PROCESSED_PATH = os.path.join(
    PROCESSED_DIR,
    "alerts_validated.csv"
)

accounts.to_csv(ACCOUNTS_PROCESSED_PATH, index=False)
alerts.to_csv(ALERTS_PROCESSED_PATH, index=False)

print("Saved account data:")
print(ACCOUNTS_PROCESSED_PATH)

print("\nSaved alert data:")
print(ALERTS_PROCESSED_PATH)

print("\nVerification:")
print("Accounts:", os.path.exists(ACCOUNTS_PROCESSED_PATH))
print("Alerts:", os.path.exists(ALERTS_PROCESSED_PATH))

In [ ]:
import os

processed_files = [
    "transactions_validated.csv",
    "accounts_validated.csv",
    "alerts_validated.csv"
]

for filename in processed_files:
    path = os.path.join(PROCESSED_DIR, filename)
    size_kb = os.path.getsize(path) / 1024

    print(f"{filename}")
    print(f"  Exists: {os.path.exists(path)}")
    print(f"  Size: {size_kb:.2f} KB")


In [ ]:
summary = {
    "transactions": len(tx),
    "accounts": len(accounts),
    "alerts": len(alerts),
    "fraud_transactions": int(tx["IS_FRAUD"].sum()),
    "fraud_accounts": int(accounts["IS_FRAUD"].sum()),
    "unique_timestamps": tx["TIMESTAMP"].nunique(),
    "timestamp_min": int(tx["TIMESTAMP"].min()),
    "timestamp_max": int(tx["TIMESTAMP"].max()),
    "self_transactions": int(
        (tx["SENDER_ACCOUNT_ID"] == tx["RECEIVER_ACCOUNT_ID"]).sum()
    ),
    "missing_transaction_values": int(tx.isnull().sum().sum()),
    "duplicate_tx_ids": int(tx["TX_ID"].duplicated().sum())
}

for key, value in summary.items():
    print(f"{key}: {value}")

In [ ]:
import networkx as nx

G = nx.DiGraph()

# Add all 100 accounts as nodes
for account_id in accounts["ACCOUNT_ID"]:
    G.add_node(account_id)

# Add transactions as directed edges
for _, row in tx.iterrows():
    G.add_edge(
        row["SENDER_ACCOUNT_ID"],
        row["RECEIVER_ACCOUNT_ID"]
    )

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Is directed:", G.is_directed())

In [ ]:
G = nx.DiGraph()

# Add all accounts as nodes
G.add_nodes_from(accounts["ACCOUNT_ID"])

# Aggregate transactions by directed edge
edge_data = (
    tx.groupby(
        ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"],
        as_index=False
    )
    .agg(
        transaction_count=("TX_ID", "count"),
        total_amount=("TX_AMOUNT", "sum")
    )
)

# Add aggregated edges
for _, row in edge_data.iterrows():
    G.add_edge(
        row["SENDER_ACCOUNT_ID"],
        row["RECEIVER_ACCOUNT_ID"],
        transaction_count=int(row["transaction_count"]),
        total_amount=float(row["total_amount"])
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

print("\nSample edges:")
for u, v, data in list(G.edges(data=True))[:10]:
    print(f"{u} → {v} | {data}")

In [ ]:
G = nx.DiGraph()

# Add all accounts as integer node IDs
G.add_nodes_from(accounts["ACCOUNT_ID"].astype(int))

# Aggregate transactions by directed edge
edge_data = (
    tx.groupby(
        ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"],
        as_index=False
    )
    .agg(
        transaction_count=("TX_ID", "count"),
        total_amount=("TX_AMOUNT", "sum")
    )
)

# Add edges while explicitly preserving integer account IDs
for row in edge_data.itertuples(index=False):
    sender = int(row.SENDER_ACCOUNT_ID)
    receiver = int(row.RECEIVER_ACCOUNT_ID)

    G.add_edge(
        sender,
        receiver,
        transaction_count=int(row.transaction_count),
        total_amount=float(row.total_amount)
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

print("\nSample edges:")
for u, v, data in list(G.edges(data=True))[:10]:
    print(f"{u} → {v} | {data}")

In [ ]:
edge_data = (
    tx.groupby(
        ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"],
        as_index=False
    )
    .agg(
        transaction_count=("TX_ID", "count"),
        total_amount=("TX_AMOUNT", "sum"),
        fraud_transaction_count=("IS_FRAUD", "sum"),
        alert_transaction_count=(
            "ALERT_ID",
            lambda x: (x != -1).sum()
        )
    )
)

G = nx.DiGraph()
G.add_nodes_from(accounts["ACCOUNT_ID"].astype(int))

for row in edge_data.itertuples(index=False):
    sender = int(row.SENDER_ACCOUNT_ID)
    receiver = int(row.RECEIVER_ACCOUNT_ID)

    G.add_edge(
        sender,
        receiver,
        transaction_count=int(row.transaction_count),
        total_amount=float(row.total_amount),
        fraud_transaction_count=int(row.fraud_transaction_count),
        alert_transaction_count=int(row.alert_transaction_count)
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

print("\nEdges containing fraud transactions:")
for u, v, data in G.edges(data=True):
    if data["fraud_transaction_count"] > 0:
        print(f"{u} → {v} | {data}")

In [ ]:
graph_fraud_transactions = sum(
    data["fraud_transaction_count"]
    for _, _, data in G.edges(data=True)
)

graph_alert_transactions = sum(
    data["alert_transaction_count"]
    for _, _, data in G.edges(data=True)
)

print("Original fraud transactions:", int(tx["IS_FRAUD"].sum()))
print("Graph fraud transactions:", graph_fraud_transactions)

print("\nOriginal alert transactions:", (tx["ALERT_ID"] != -1).sum())
print("Graph alert transactions:", graph_alert_transactions)

print(
    "\nFraud count preserved:",
    graph_fraud_transactions == int(tx["IS_FRAUD"].sum())
)

print(
    "Alert count preserved:",
    graph_alert_transactions == (tx["ALERT_ID"] != -1).sum()
)

In [ ]:
# Basic graph topology statistics

in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

print("\nAverage in-degree:", sum(in_degrees.values()) / G.number_of_nodes())
print("Average out-degree:", sum(out_degrees.values()) / G.number_of_nodes())

print("\nTop 10 accounts by in-degree:")
for account, degree in sorted(
    in_degrees.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]:
    print(account, "->", degree)

print("\nTop 10 accounts by out-degree:")
for account, degree in sorted(
    out_degrees.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]:
    print(account, "->", degree)

In [ ]:
self_loop_edges = list(nx.selfloop_edges(G))

print("Number of self-loop edges:", len(self_loop_edges))

print("\nFirst 20 self-loop edges:")
for u, v in self_loop_edges[:20]:
    data = G[u][v]
    print(
        f"{u} → {v} | "
        f"transactions={data['transaction_count']} | "
        f"amount={data['total_amount']:.2f}"
    )

In [ ]:
weak_components = list(nx.weakly_connected_components(G))
strong_components = list(nx.strongly_connected_components(G))

print("Weakly connected components:", len(weak_components))
print("Strongly connected components:", len(strong_components))

print("\nLargest weakly connected components:")
for i, component in enumerate(
    sorted(weak_components, key=len, reverse=True)[:10],
    start=1
):
    print(f"Component {i}: {len(component)} nodes")

print("\nLargest strongly connected components:")
for i, component in enumerate(
    sorted(strong_components, key=len, reverse=True)[:10],
    start=1
):
    print(f"Component {i}: {len(component)} nodes")


In [ ]:
# Detect accounts with high incoming/outgoing connectivity

fan_in_candidates = sorted(
    in_degrees.items(),
    key=lambda x: x[1],
    reverse=True
)

fan_out_candidates = sorted(
    out_degrees.items(),
    key=lambda x: x[1],
    reverse=True
)

print("Top 10 fan-in candidates:")
for account, degree in fan_in_candidates[:10]:
    print(f"Account {account}: {degree} incoming connections")

print("\nTop 10 fan-out candidates:")
for account, degree in fan_out_candidates[:10]:
    print(f"Account {account}: {degree} outgoing connections")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

In [ ]:
import pandas as pd

TX_PATH = "/content/drive/MyDrive/TraceGraph/data/processed/transactions_validated.csv"

tx = pd.read_csv(TX_PATH)

print("Transactions loaded:", len(tx))
print("Columns:", tx.columns.tolist())

In [ ]:
# Calculate incoming and outgoing transaction activity for each account

account_flow = (
    tx.groupby("RECEIVER_ACCOUNT_ID")
      .agg(
          incoming_transactions=("TX_ID", "count"),
          incoming_amount=("TX_AMOUNT", "sum")
      )
      .rename_axis("ACCOUNT_ID")
)

outgoing_flow = (
    tx.groupby("SENDER_ACCOUNT_ID")
      .agg(
          outgoing_transactions=("TX_ID", "count"),
          outgoing_amount=("TX_AMOUNT", "sum")
      )
      .rename_axis("ACCOUNT_ID")
)

account_flow = account_flow.join(
    outgoing_flow,
    how="outer"
).fillna(0)

# Ensure account IDs are integers
account_flow.index = account_flow.index.astype(int)

print("Account flow shape:", account_flow.shape)

print("\nTop 10 accounts by incoming transactions:")
print(
    account_flow
    .sort_values("incoming_transactions", ascending=False)
    .head(10)
)

print("\nTop 10 accounts by outgoing transactions:")
print(
    account_flow
    .sort_values("outgoing_transactions", ascending=False)
    .head(10)
)

In [ ]:
# Calculate flow imbalance for each account

account_flow["amount_imbalance"] = (
    account_flow["incoming_amount"]
    - account_flow["outgoing_amount"]
)

account_flow["transaction_imbalance"] = (
    account_flow["incoming_transactions"]
    - account_flow["outgoing_transactions"]
)

print("Top 10 accounts by absolute amount imbalance:")

print(
    account_flow.assign(
        absolute_amount_imbalance=
        account_flow["amount_imbalance"].abs()
    )
    .sort_values(
        "absolute_amount_imbalance",
        ascending=False
    )
    .head(10)
    [
        [
            "incoming_amount",
            "outgoing_amount",
            "amount_imbalance",
            "incoming_transactions",
            "outgoing_transactions",
            "transaction_imbalance"
        ]
    ]
)

In [ ]:
# Calculate incoming/outgoing ratios

account_flow["amount_in_out_ratio"] = (
    account_flow["incoming_amount"]
    / account_flow["outgoing_amount"].replace(0, float("nan"))
)

account_flow["transaction_in_out_ratio"] = (
    account_flow["incoming_transactions"]
    / account_flow["outgoing_transactions"].replace(0, float("nan"))
)

print("Top 10 accounts by amount in/out ratio:")

print(
    account_flow
    .sort_values(
        "amount_in_out_ratio",
        ascending=False
    )
    .head(10)
    [
        [
            "incoming_amount",
            "outgoing_amount",
            "amount_in_out_ratio",
            "incoming_transactions",
            "outgoing_transactions",
            "transaction_in_out_ratio"
        ]
    ]
)

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

print("PROJECT_DIR restored:")
print(PROJECT_DIR)
print("Exists:", os.path.exists(PROJECT_DIR))

In [ ]:
account_flow_path = os.path.join(
    PROJECT_DIR,
    "data/processed/account_flow_features.csv"
)

account_flow.to_csv(account_flow_path)

print("Account flow features saved successfully.")
print("Path:", account_flow_path)
print("Shape:", account_flow.shape)
print("\nColumns:")
print(account_flow.columns.tolist())

In [ ]:
import pandas as pd

accounts_path = os.path.join(
    PROJECT_DIR,
    "data/processed/accounts_validated.csv"
)

accounts = pd.read_csv(accounts_path)

print("Accounts loaded successfully.")
print("Shape:", accounts.shape)
print("Columns:", accounts.columns.tolist())


In [ ]:
# Join graph features with account fraud labels

account_features = account_flow.reset_index().merge(
    accounts[
        ["ACCOUNT_ID", "IS_FRAUD"]
    ],
    on="ACCOUNT_ID",
    how="left"
)

print("Account feature table shape:", account_features.shape)

print("\nFraud vs non-fraud account counts:")
print(account_features["IS_FRAUD"].value_counts())

print("\nAverage features by fraud label:")
print(
    account_features
    .groupby("IS_FRAUD")[
        [
            "incoming_transactions",
            "outgoing_transactions",
            "incoming_amount",
            "outgoing_amount",
            "amount_imbalance",
            "transaction_imbalance",
            "amount_in_out_ratio",
            "transaction_in_out_ratio"
        ]
    ]
    .mean()
)

In [ ]:
# Inspect graph features of fraudulent accounts

fraud_accounts = (
    account_features[
        account_features["IS_FRAUD"] == True
    ]
    .sort_values(
        "amount_in_out_ratio",
        ascending=False
    )
)

print("Fraudulent accounts:", len(fraud_accounts))

print(
    fraud_accounts[
        [
            "ACCOUNT_ID",
            "incoming_transactions",
            "outgoing_transactions",
            "incoming_amount",
            "outgoing_amount",
            "amount_imbalance",
            "transaction_imbalance",
            "amount_in_out_ratio",
            "transaction_in_out_ratio"
        ]
    ].to_string(index=False)
)

In [ ]:
# Create the final account-level feature table

account_model_features = account_features[
    [
        "ACCOUNT_ID",
        "incoming_transactions",
        "outgoing_transactions",
        "incoming_amount",
        "outgoing_amount",
        "amount_imbalance",
        "transaction_imbalance",
        "amount_in_out_ratio",
        "transaction_in_out_ratio",
        "IS_FRAUD"
    ]
].copy()

print("Account model feature table:")
print("Shape:", account_model_features.shape)

print("\nMissing values:")
print(account_model_features.isnull().sum())

print("\nFirst 5 rows:")
print(account_model_features.head())

In [ ]:
# Save finalized account-level modeling features

account_model_features_path = os.path.join(
    PROJECT_DIR,
    "data/processed/account_model_features.csv"
)

account_model_features.to_csv(
    account_model_features_path,
    index=False
)

print("Account model features saved successfully.")
print("Path:", account_model_features_path)
print("Shape:", account_model_features.shape)

In [ ]:
# Correlation analysis of account-level features

correlation_matrix = account_model_features[
    [
        "incoming_transactions",
        "outgoing_transactions",
        "incoming_amount",
        "outgoing_amount",
        "amount_imbalance",
        "transaction_imbalance",
        "amount_in_out_ratio",
        "transaction_in_out_ratio",
        "IS_FRAUD"
    ]
].corr()

print("Correlation with IS_FRAUD:")
print(
    correlation_matrix["IS_FRAUD"]
    .sort_values(ascending=False)
)

print("\nFull correlation matrix:")
print(correlation_matrix.round(3))

In [ ]:
# Inspect temporal distribution of transactions

timestamp_summary = (
    tx.groupby("TIMESTAMP")
      .agg(
          transaction_count=("TX_ID", "count"),
          total_amount=("TX_AMOUNT", "sum"),
          fraud_transactions=("IS_FRAUD", "sum")
      )
      .reset_index()
)

print("Timestamp range:")
print(
    "Minimum:", timestamp_summary["TIMESTAMP"].min(),
    "| Maximum:", timestamp_summary["TIMESTAMP"].max()
)

print("\nNumber of unique timestamps:",
      timestamp_summary["TIMESTAMP"].nunique())

print("\nFirst 10 timestamps:")
print(timestamp_summary.head(10).to_string(index=False))

print("\nLast 10 timestamps:")
print(timestamp_summary.tail(10).to_string(index=False))

print("\nTotal fraud transactions across timestamps:",
      timestamp_summary["fraud_transactions"].sum())

In [ ]:
# Identify timestamps containing fraudulent transactions

fraud_timestamps = (
    tx[tx["IS_FRAUD"] == True]
    .groupby("TIMESTAMP")
    .agg(
        fraud_transactions=("TX_ID", "count"),
        fraud_amount=("TX_AMOUNT", "sum")
    )
    .reset_index()
)

print("Timestamps containing fraud:")
print(fraud_timestamps.to_string(index=False))

print("\nNumber of timestamps containing fraud:",
      len(fraud_timestamps))

print("\nTotal fraud transactions:",
      fraud_timestamps["fraud_transactions"].sum())

In [ ]:
# Define temporal window sizes for TraceGraph

WINDOW_SIZES = {
    "short": 10,
    "medium": 25,
    "long": 50
}

print("TraceGraph temporal windows:")
for name, size in WINDOW_SIZES.items():
    print(f"{name:>6} window: {size} timestamp units")


In [ ]:
import networkx as nx

print("NetworkX restored successfully.")
print("Version:", nx.__version__)

In [ ]:
# Build the first short temporal transaction graph
# Window: timestamps 0-9

SHORT_WINDOW = 10

window_start = 0
window_end = window_start + SHORT_WINDOW - 1

tx_window = tx[
    (tx["TIMESTAMP"] >= window_start) &
    (tx["TIMESTAMP"] <= window_end)
].copy()

G_short_0 = nx.DiGraph()

G_short_0.add_nodes_from(
    accounts["ACCOUNT_ID"].astype(int)
)

window_edges = (
    tx_window
    .groupby(
        ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"],
        as_index=False
    )
    .agg(
        transaction_count=("TX_ID", "count"),
        total_amount=("TX_AMOUNT", "sum")
    )
)

for row in window_edges.itertuples(index=False):
    G_short_0.add_edge(
        int(row.SENDER_ACCOUNT_ID),
        int(row.RECEIVER_ACCOUNT_ID),
        transaction_count=int(row.transaction_count),
        total_amount=float(row.total_amount)
    )

print("Temporal window:", f"{window_start}-{window_end}")
print("Transactions:", len(tx_window))
print("Nodes:", G_short_0.number_of_nodes())
print("Edges:", G_short_0.number_of_edges())
print("Total amount:", round(tx_window["TX_AMOUNT"].sum(), 2))

In [ ]:
# Build a short temporal graph for a window containing fraud
# Window: timestamps 27-36

window_start = 27
window_end = window_start + SHORT_WINDOW - 1

tx_fraud_window = tx[
    (tx["TIMESTAMP"] >= window_start) &
    (tx["TIMESTAMP"] <= window_end)
].copy()

G_short_fraud = nx.DiGraph()

G_short_fraud.add_nodes_from(
    accounts["ACCOUNT_ID"].astype(int)
)

fraud_window_edges = (
    tx_fraud_window
    .groupby(
        ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"],
        as_index=False
    )
    .agg(
        transaction_count=("TX_ID", "count"),
        total_amount=("TX_AMOUNT", "sum"),
        fraud_transaction_count=("IS_FRAUD", "sum")
    )
)

for row in fraud_window_edges.itertuples(index=False):
    G_short_fraud.add_edge(
        int(row.SENDER_ACCOUNT_ID),
        int(row.RECEIVER_ACCOUNT_ID),
        transaction_count=int(row.transaction_count),
        total_amount=float(row.total_amount),
        fraud_transaction_count=int(row.fraud_transaction_count)
    )

print("Temporal window:", f"{window_start}-{window_end}")
print("Transactions:", len(tx_fraud_window))
print("Nodes:", G_short_fraud.number_of_nodes())
print("Edges:", G_short_fraud.number_of_edges())
print("Fraud transactions:", int(tx_fraud_window["IS_FRAUD"].sum()))

print("\nFraud edges in this window:")

fraud_edges = [
    (
        u,
        v,
        data["fraud_transaction_count"],
        data["total_amount"]
    )
    for u, v, data in G_short_fraud.edges(data=True)
    if data["fraud_transaction_count"] > 0
]

for edge in fraud_edges:
    print(edge)

In [ ]:
def build_temporal_graph(tx_data, accounts_data, window_start, window_size):
    """
    Build an aggregated directed transaction graph
    for a specified temporal window.
    """

    window_end = window_start + window_size - 1

    window_tx = tx_data[
        (tx_data["TIMESTAMP"] >= window_start) &
        (tx_data["TIMESTAMP"] <= window_end)
    ].copy()

    G = nx.DiGraph()

    # Add all accounts, including accounts with no
    # transactions inside this window
    G.add_nodes_from(
        accounts_data["ACCOUNT_ID"].astype(int)
    )

    # Aggregate transactions between account pairs
    window_edges = (
        window_tx
        .groupby(
            ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"],
            as_index=False
        )
        .agg(
            transaction_count=("TX_ID", "count"),
            total_amount=("TX_AMOUNT", "sum"),
            fraud_transaction_count=("IS_FRAUD", "sum")
        )
    )

    # Add aggregated edges
    for row in window_edges.itertuples(index=False):
        G.add_edge(
            int(row.SENDER_ACCOUNT_ID),
            int(row.RECEIVER_ACCOUNT_ID),
            transaction_count=int(row.transaction_count),
            total_amount=float(row.total_amount),
            fraud_transaction_count=int(row.fraud_transaction_count)
        )

    return G, window_tx

In [ ]:
# Test the reusable temporal graph function

G_test, tx_test = build_temporal_graph(
    tx,
    accounts,
    window_start=27,
    window_size=10
)

print("Temporal window: 27-36")
print("Transactions:", len(tx_test))
print("Nodes:", G_test.number_of_nodes())
print("Edges:", G_test.number_of_edges())
print("Fraud transactions:", int(tx_test["IS_FRAUD"].sum()))
print("Total amount:", round(tx_test["TX_AMOUNT"].sum(), 2))

In [ ]:
# Build all non-overlapping short temporal graphs

SHORT_WINDOW = 10

short_graphs = {}
short_window_summary = []

for window_start in range(
    tx["TIMESTAMP"].min(),
    tx["TIMESTAMP"].max() + 1,
    SHORT_WINDOW
):
    G_window, tx_window = build_temporal_graph(
        tx,
        accounts,
        window_start=window_start,
        window_size=SHORT_WINDOW
    )

    short_graphs[window_start] = G_window

    short_window_summary.append({
        "window_start": window_start,
        "window_end": window_start + SHORT_WINDOW - 1,
        "transactions": len(tx_window),
        "edges": G_window.number_of_edges(),
        "total_amount": tx_window["TX_AMOUNT"].sum(),
        "fraud_transactions": tx_window["IS_FRAUD"].sum()
    })

short_window_summary = pd.DataFrame(short_window_summary)

print("Number of short temporal graphs:",
      len(short_graphs))

print("\nShort temporal window summary:")
print(short_window_summary.to_string(index=False))

print("\nTotal fraud transactions across windows:",
      int(short_window_summary["fraud_transactions"].sum()))

In [ ]:
# ============================================================
# TRACEGRAPH — MULTI-SCALE TEMPORAL GRAPH CONSTRUCTION
# ============================================================

import os
import pandas as pd
import numpy as np
import networkx as nx

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TX_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/transactions_validated.csv"
)

ACCOUNTS_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/accounts_validated.csv"
)

tx = pd.read_csv(TX_PATH)
accounts = pd.read_csv(ACCOUNTS_PATH)

tx["TIMESTAMP"] = tx["TIMESTAMP"].astype(int)
tx["SENDER_ACCOUNT_ID"] = tx["SENDER_ACCOUNT_ID"].astype(int)
tx["RECEIVER_ACCOUNT_ID"] = tx["RECEIVER_ACCOUNT_ID"].astype(int)
tx["TX_AMOUNT"] = pd.to_numeric(tx["TX_AMOUNT"])
tx["IS_FRAUD"] = tx["IS_FRAUD"].astype(bool)

# ------------------------------------------------------------
# 2. Multi-scale window definitions
# ------------------------------------------------------------

WINDOW_SIZES = {
    "short": 10,
    "medium": 25,
    "long": 50
}

MIN_TIMESTAMP = int(tx["TIMESTAMP"].min())
MAX_TIMESTAMP = int(tx["TIMESTAMP"].max())

print("Timestamp range:", MIN_TIMESTAMP, "to", MAX_TIMESTAMP)
print("Window sizes:", WINDOW_SIZES)

# ------------------------------------------------------------
# 3. Function to build one temporal graph
# ------------------------------------------------------------

def build_temporal_graph(tx_data, accounts_data, window_start, window_size):

    window_end = min(
        window_start + window_size - 1,
        MAX_TIMESTAMP
    )

    window_tx = tx_data[
        (tx_data["TIMESTAMP"] >= window_start) &
        (tx_data["TIMESTAMP"] <= window_end)
    ].copy()

    G = nx.DiGraph()

    # Keep all accounts as nodes, including inactive ones
    G.add_nodes_from(
        accounts_data["ACCOUNT_ID"].astype(int)
    )

    if len(window_tx) > 0:

        window_edges = (
            window_tx
            .groupby(
                ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"],
                as_index=False
            )
            .agg(
                transaction_count=("TX_ID", "count"),
                total_amount=("TX_AMOUNT", "sum"),
                fraud_transaction_count=("IS_FRAUD", "sum")
            )
        )

        for row in window_edges.itertuples(index=False):

            G.add_edge(
                int(row.SENDER_ACCOUNT_ID),
                int(row.RECEIVER_ACCOUNT_ID),

                transaction_count=int(
                    row.transaction_count
                ),

                total_amount=float(
                    row.total_amount
                ),

                fraud_transaction_count=int(
                    row.fraud_transaction_count
                )
            )

    return G, window_tx


# ------------------------------------------------------------
# 4. Build all temporal scales
# ------------------------------------------------------------

temporal_graphs = {}
temporal_summaries = {}

for scale_name, window_size in WINDOW_SIZES.items():

    graphs = []
    summary_rows = []

    # Non-overlapping windows
    for window_start in range(
        MIN_TIMESTAMP,
        MAX_TIMESTAMP + 1,
        window_size
    ):

        window_end = min(
            window_start + window_size - 1,
            MAX_TIMESTAMP
        )

        G, window_tx = build_temporal_graph(
            tx,
            accounts,
            window_start,
            window_size
        )

        number_of_nodes = G.number_of_nodes()
        number_of_edges = G.number_of_edges()

        # Directed graph density
        if number_of_nodes > 1:
            density = nx.density(G)
        else:
            density = 0.0

        # Active accounts in this window
        if len(window_tx) > 0:

            active_accounts = len(
                set(
                    window_tx["SENDER_ACCOUNT_ID"]
                ).union(
                    set(
                        window_tx["RECEIVER_ACCOUNT_ID"]
                    )
                )
            )

        else:
            active_accounts = 0

        # Average degree
        if number_of_nodes > 0:
            average_degree = (
                sum(dict(G.degree()).values())
                / number_of_nodes
            )
        else:
            average_degree = 0.0

        summary_rows.append({

            "scale": scale_name,

            "window_size": window_size,

            "window_start": window_start,

            "window_end": window_end,

            "transactions": len(window_tx),

            "edges": number_of_edges,

            "nodes": number_of_nodes,

            "active_accounts": active_accounts,

            "total_amount": (
                float(window_tx["TX_AMOUNT"].sum())
                if len(window_tx) > 0
                else 0.0
            ),

            "fraud_transactions": int(
                window_tx["IS_FRAUD"].sum()
            ),

            "average_degree": average_degree,

            "density": density
        })

        graphs.append({
            "window_start": window_start,
            "window_end": window_end,
            "graph": G,
            "transactions": window_tx
        })

    temporal_graphs[scale_name] = graphs

    temporal_summaries[scale_name] = pd.DataFrame(
        summary_rows
    )


# ------------------------------------------------------------
# 5. Combine all temporal summaries
# ------------------------------------------------------------

all_temporal_summary = pd.concat(
    temporal_summaries.values(),
    ignore_index=True
)

# ------------------------------------------------------------
# 6. Save summaries
# ------------------------------------------------------------

for scale_name, summary_df in temporal_summaries.items():

    output_path = os.path.join(
        PROJECT_DIR,
        "data/processed",
        f"{scale_name}_temporal_summary.csv"
    )

    summary_df.to_csv(
        output_path,
        index=False
    )

all_summary_path = os.path.join(
    PROJECT_DIR,
    "data/processed",
    "multi_scale_temporal_summary.csv"
)

all_temporal_summary.to_csv(
    all_summary_path,
    index=False
)

# ------------------------------------------------------------
# 7. Validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MULTI-SCALE TEMPORAL GRAPH SUMMARY")
print("=" * 60)

for scale_name in WINDOW_SIZES:

    summary = temporal_summaries[scale_name]

    print(f"\n{scale_name.upper()} WINDOWS")
    print("-" * 60)

    print(
        "Number of windows:",
        len(summary)
    )

    print(
        "Total transactions:",
        int(summary["transactions"].sum())
    )

    print(
        "Total fraud transactions:",
        int(summary["fraud_transactions"].sum())
    )

    print(
        "Total amount:",
        round(summary["total_amount"].sum(), 2)
    )

    print(
        "Average edges/window:",
        round(summary["edges"].mean(), 2)
    )

    print(
        "Average active accounts/window:",
        round(summary["active_accounts"].mean(), 2)
    )

# ------------------------------------------------------------
# 8. Critical integrity check
# ------------------------------------------------------------

original_fraud_count = int(
    tx["IS_FRAUD"].sum()
)

print("\n" + "=" * 60)
print("INTEGRITY CHECK")
print("=" * 60)

print(
    "Original fraud transactions:",
    original_fraud_count
)

for scale_name in WINDOW_SIZES:

    scale_fraud_count = int(
        temporal_summaries[scale_name][
            "fraud_transactions"
        ].sum()
    )

    print(
        f"{scale_name.capitalize()} window fraud transactions:",
        scale_fraud_count
    )

    assert (
        scale_fraud_count == original_fraud_count
    ), (
        f"Fraud count mismatch in {scale_name} windows"
    )

print("\n✓ All multi-scale temporal graphs validated.")
print("✓ All 18 fraud transactions preserved.")
print("\nSaved files:")

for scale_name in WINDOW_SIZES:

    print(
        f" - data/processed/"
        f"{scale_name}_temporal_summary.csv"
    )

print(
    " - data/processed/"
    "multi_scale_temporal_summary.csv"
)

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 5
# GRAPH + TEMPORAL FEATURE ENGINEERING
# ============================================================

import os
import numpy as np
import pandas as pd
import networkx as nx

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TX_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/transactions_validated.csv"
)

ACCOUNTS_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/accounts_validated.csv"
)

tx = pd.read_csv(TX_PATH)
accounts = pd.read_csv(ACCOUNTS_PATH)

# ------------------------------------------------------------
# 1. Basic type preparation
# ------------------------------------------------------------

tx["TX_ID"] = tx["TX_ID"].astype(int)
tx["SENDER_ACCOUNT_ID"] = tx["SENDER_ACCOUNT_ID"].astype(int)
tx["RECEIVER_ACCOUNT_ID"] = tx["RECEIVER_ACCOUNT_ID"].astype(int)
tx["TX_AMOUNT"] = pd.to_numeric(tx["TX_AMOUNT"])
tx["TIMESTAMP"] = tx["TIMESTAMP"].astype(int)
tx["IS_FRAUD"] = tx["IS_FRAUD"].astype(bool)
tx["ALERT_ID"] = tx["ALERT_ID"].astype(int)

account_ids = accounts["ACCOUNT_ID"].astype(int).tolist()

# ============================================================
# PART A — STATIC GRAPH FEATURES
# ============================================================

print("=" * 70)
print("PART A — STATIC GRAPH FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# 2. Build transaction graph
# ------------------------------------------------------------

edge_data = (
    tx.groupby(
        ["SENDER_ACCOUNT_ID", "RECEIVER_ACCOUNT_ID"],
        as_index=False
    )
    .agg(
        transaction_count=("TX_ID", "count"),
        total_amount=("TX_AMOUNT", "sum"),
        average_amount=("TX_AMOUNT", "mean"),
        fraud_transaction_count=("IS_FRAUD", "sum")
    )
)

G = nx.DiGraph()

G.add_nodes_from(account_ids)

for row in edge_data.itertuples(index=False):

    G.add_edge(
        int(row.SENDER_ACCOUNT_ID),
        int(row.RECEIVER_ACCOUNT_ID),

        transaction_count=int(row.transaction_count),

        total_amount=float(row.total_amount),

        average_amount=float(row.average_amount),

        fraud_transaction_count=int(
            row.fraud_transaction_count
        )
    )

# ------------------------------------------------------------
# 3. Degree features
# ------------------------------------------------------------

in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

weighted_in_degree = dict(
    G.in_degree(weight="total_amount")
)

weighted_out_degree = dict(
    G.out_degree(weight="total_amount")
)

# ------------------------------------------------------------
# 4. Account flow features
# ------------------------------------------------------------

incoming = (
    tx.groupby("RECEIVER_ACCOUNT_ID")
    .agg(
        incoming_transactions=("TX_ID", "count"),
        incoming_amount=("TX_AMOUNT", "sum"),
        incoming_avg_amount=("TX_AMOUNT", "mean")
    )
)

outgoing = (
    tx.groupby("SENDER_ACCOUNT_ID")
    .agg(
        outgoing_transactions=("TX_ID", "count"),
        outgoing_amount=("TX_AMOUNT", "sum"),
        outgoing_avg_amount=("TX_AMOUNT", "mean")
    )
)

account_features = pd.DataFrame(
    index=account_ids
)

account_features.index.name = "ACCOUNT_ID"

account_features = account_features.join(
    incoming
)

account_features = account_features.join(
    outgoing
)

account_features = account_features.fillna(0)

# ------------------------------------------------------------
# 5. Degree statistics
# ------------------------------------------------------------

account_features["in_degree"] = (
    pd.Series(in_degree)
)

account_features["out_degree"] = (
    pd.Series(out_degree)
)

account_features["weighted_in_degree"] = (
    pd.Series(weighted_in_degree)
)

account_features["weighted_out_degree"] = (
    pd.Series(weighted_out_degree)
)

account_features = account_features.fillna(0)

# ------------------------------------------------------------
# 6. Flow imbalance
# ------------------------------------------------------------

account_features["amount_imbalance"] = (
    account_features["incoming_amount"]
    -
    account_features["outgoing_amount"]
)

account_features["transaction_imbalance"] = (
    account_features["incoming_transactions"]
    -
    account_features["outgoing_transactions"]
)

account_features["amount_in_out_ratio"] = (
    account_features["incoming_amount"]
    /
    account_features["outgoing_amount"]
    .replace(0, np.nan)
)

account_features["transaction_in_out_ratio"] = (
    account_features["incoming_transactions"]
    /
    account_features["outgoing_transactions"]
    .replace(0, np.nan)
)

# ------------------------------------------------------------
# 7. Fan-in / Fan-out features
# ------------------------------------------------------------

account_features["fan_in"] = (
    account_features["in_degree"]
)

account_features["fan_out"] = (
    account_features["out_degree"]
)

account_features["fan_in_out_ratio"] = (
    account_features["fan_in"]
    /
    account_features["fan_out"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 8. Self-loop features
# ------------------------------------------------------------

self_loop_counts = (
    tx[
        tx["SENDER_ACCOUNT_ID"]
        ==
        tx["RECEIVER_ACCOUNT_ID"]
    ]
    .groupby("SENDER_ACCOUNT_ID")
    .size()
)

self_loop_amounts = (
    tx[
        tx["SENDER_ACCOUNT_ID"]
        ==
        tx["RECEIVER_ACCOUNT_ID"]
    ]
    .groupby("SENDER_ACCOUNT_ID")["TX_AMOUNT"]
    .sum()
)

account_features["self_loop_transactions"] = (
    self_loop_counts
)

account_features["self_loop_amount"] = (
    self_loop_amounts
)

account_features = account_features.fillna(0)

# ============================================================
# PART B — TEMPORAL FEATURES
# ============================================================

print("\n" + "=" * 70)
print("PART B — TEMPORAL FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# 9. Activity span
# ------------------------------------------------------------

first_timestamp = (
    tx.groupby("SENDER_ACCOUNT_ID")["TIMESTAMP"]
    .min()
)

last_timestamp = (
    tx.groupby("SENDER_ACCOUNT_ID")["TIMESTAMP"]
    .max()
)

account_features["first_outgoing_timestamp"] = (
    first_timestamp
)

account_features["last_outgoing_timestamp"] = (
    last_timestamp
)

account_features["activity_span"] = (
    account_features["last_outgoing_timestamp"]
    -
    account_features["first_outgoing_timestamp"]
)

# ------------------------------------------------------------
# 10. Incoming activity span
# ------------------------------------------------------------

first_incoming = (
    tx.groupby("RECEIVER_ACCOUNT_ID")["TIMESTAMP"]
    .min()
)

last_incoming = (
    tx.groupby("RECEIVER_ACCOUNT_ID")["TIMESTAMP"]
    .max()
)

account_features["first_incoming_timestamp"] = (
    first_incoming
)

account_features["last_incoming_timestamp"] = (
    last_incoming
)

account_features["incoming_activity_span"] = (
    account_features["last_incoming_timestamp"]
    -
    account_features["first_incoming_timestamp"]
)

account_features = account_features.fillna(0)

# ------------------------------------------------------------
# 11. Transaction velocity
# ------------------------------------------------------------

# Number of transactions per active timestamp

sender_timestamp_counts = (
    tx.groupby(
        ["SENDER_ACCOUNT_ID", "TIMESTAMP"]
    )
    .size()
)

receiver_timestamp_counts = (
    tx.groupby(
        ["RECEIVER_ACCOUNT_ID", "TIMESTAMP"]
    )
    .size()
)

account_features["max_outgoing_velocity"] = (
    sender_timestamp_counts
    .groupby(level=0)
    .max()
)

account_features["avg_outgoing_velocity"] = (
    sender_timestamp_counts
    .groupby(level=0)
    .mean()
)

account_features["max_incoming_velocity"] = (
    receiver_timestamp_counts
    .groupby(level=0)
    .max()
)

account_features["avg_incoming_velocity"] = (
    receiver_timestamp_counts
    .groupby(level=0)
    .mean()
)

account_features = account_features.fillna(0)

# ------------------------------------------------------------
# 12. Temporal concentration
# ------------------------------------------------------------

# Fraction of transactions occurring in the busiest
# timestamp for each account.

account_features["outgoing_temporal_concentration"] = (
    account_features["max_outgoing_velocity"]
    /
    account_features["outgoing_transactions"]
    .replace(0, np.nan)
)

account_features["incoming_temporal_concentration"] = (
    account_features["max_incoming_velocity"]
    /
    account_features["incoming_transactions"]
    .replace(0, np.nan)
)

# ------------------------------------------------------------
# 13. Amount concentration
# ------------------------------------------------------------

max_outgoing_amount = (
    tx.groupby("SENDER_ACCOUNT_ID")["TX_AMOUNT"]
    .max()
)

max_incoming_amount = (
    tx.groupby("RECEIVER_ACCOUNT_ID")["TX_AMOUNT"]
    .max()
)

account_features["max_outgoing_amount"] = (
    max_outgoing_amount
)

account_features["max_incoming_amount"] = (
    max_incoming_amount
)

account_features["outgoing_amount_concentration"] = (
    account_features["max_outgoing_amount"]
    /
    account_features["outgoing_amount"]
    .replace(0, np.nan)
)

account_features["incoming_amount_concentration"] = (
    account_features["max_incoming_amount"]
    /
    account_features["incoming_amount"]
    .replace(0, np.nan)
)

# ------------------------------------------------------------
# 14. Unique counterparties
# ------------------------------------------------------------

unique_senders = (
    tx.groupby("RECEIVER_ACCOUNT_ID")
    ["SENDER_ACCOUNT_ID"]
    .nunique()
)

unique_receivers = (
    tx.groupby("SENDER_ACCOUNT_ID")
    ["RECEIVER_ACCOUNT_ID"]
    .nunique()
)

account_features["unique_incoming_counterparties"] = (
    unique_senders
)

account_features["unique_outgoing_counterparties"] = (
    unique_receivers
)

account_features = account_features.fillna(0)

# ------------------------------------------------------------
# 15. Account activity ratio
# ------------------------------------------------------------

account_features["total_transactions"] = (
    account_features["incoming_transactions"]
    +
    account_features["outgoing_transactions"]
)

account_features["total_flow_amount"] = (
    account_features["incoming_amount"]
    +
    account_features["outgoing_amount"]
)

# ============================================================
# PART C — LABELS
# ============================================================

print("\n" + "=" * 70)
print("PART C — LABELS")
print("=" * 70)

account_labels = (
    accounts[
        ["ACCOUNT_ID", "IS_FRAUD"]
    ]
    .copy()
)

account_labels["ACCOUNT_ID"] = (
    account_labels["ACCOUNT_ID"].astype(int)
)

account_features = (
    account_features
    .reset_index()
    .merge(
        account_labels,
        on="ACCOUNT_ID",
        how="left"
    )
)

# ============================================================
# PART D — TRANSACTION-LEVEL FEATURES
# ============================================================

print("\n" + "=" * 70)
print("PART D — TRANSACTION-LEVEL FEATURES")
print("=" * 70)

tx_features = tx.copy()

# ------------------------------------------------------------
# 16. Sender account statistics
# ------------------------------------------------------------

sender_stats = (
    account_features[
        [
            "ACCOUNT_ID",
            "outgoing_transactions",
            "outgoing_amount",
            "out_degree",
            "fan_out",
            "max_outgoing_velocity",
            "outgoing_temporal_concentration"
        ]
    ]
    .rename(
        columns={
            "ACCOUNT_ID": "SENDER_ACCOUNT_ID",
            "outgoing_transactions":
                "sender_total_transactions",
            "outgoing_amount":
                "sender_total_amount",
            "out_degree":
                "sender_out_degree",
            "fan_out":
                "sender_fan_out",
            "max_outgoing_velocity":
                "sender_max_velocity",
            "outgoing_temporal_concentration":
                "sender_temporal_concentration"
        }
    )
)

tx_features = tx_features.merge(
    sender_stats,
    on="SENDER_ACCOUNT_ID",
    how="left"
)

# ------------------------------------------------------------
# 17. Receiver account statistics
# ------------------------------------------------------------

receiver_stats = (
    account_features[
        [
            "ACCOUNT_ID",
            "incoming_transactions",
            "incoming_amount",
            "in_degree",
            "fan_in",
            "max_incoming_velocity",
            "incoming_temporal_concentration"
        ]
    ]
    .rename(
        columns={
            "ACCOUNT_ID": "RECEIVER_ACCOUNT_ID",
            "incoming_transactions":
                "receiver_total_transactions",
            "incoming_amount":
                "receiver_total_amount",
            "in_degree":
                "receiver_in_degree",
            "fan_in":
                "receiver_fan_in",
            "max_incoming_velocity":
                "receiver_max_velocity",
            "incoming_temporal_concentration":
                "receiver_temporal_concentration"
        }
    )
)

tx_features = tx_features.merge(
    receiver_stats,
    on="RECEIVER_ACCOUNT_ID",
    how="left"
)

# ------------------------------------------------------------
# 18. Transaction-level flow features
# ------------------------------------------------------------

tx_features["sender_receiver_amount_ratio"] = (
    tx_features["sender_total_amount"]
    /
    tx_features["receiver_total_amount"]
    .replace(0, np.nan)
)

tx_features["amount_vs_sender_average"] = (
    tx_features["TX_AMOUNT"]
    /
    (
        tx_features["sender_total_amount"]
        /
        tx_features["sender_total_transactions"]
    )
    .replace(0, np.nan)
)

tx_features["amount_vs_receiver_average"] = (
    tx_features["TX_AMOUNT"]
    /
    (
        tx_features["receiver_total_amount"]
        /
        tx_features["receiver_total_transactions"]
    )
    .replace(0, np.nan)
)

# ------------------------------------------------------------
# 19. Save feature tables
# ------------------------------------------------------------

account_feature_path = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_account_features.csv"
)

transaction_feature_path = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_transaction_features.csv"
)

account_features.to_csv(
    account_feature_path,
    index=False
)

tx_features.to_csv(
    transaction_feature_path,
    index=False
)

# ============================================================
# PART E — VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("FEATURE ENGINEERING VALIDATION")
print("=" * 70)

print(
    "\nAccount feature shape:",
    account_features.shape
)

print(
    "Transaction feature shape:",
    tx_features.shape
)

print(
    "\nAccount fraud labels:"
)

print(
    account_features["IS_FRAUD"]
    .value_counts()
)

print(
    "\nTransaction fraud labels:"
)

print(
    tx_features["IS_FRAUD"]
    .value_counts()
)

print(
    "\nMissing account feature values:",
    int(account_features.isna().sum().sum())
)

print(
    "Missing transaction feature values:",
    int(tx_features.isna().sum().sum())
)

print(
    "\nAccount features:"
)

print(
    list(account_features.columns)
)

print(
    "\nTransaction features:"
)

print(
    list(tx_features.columns)
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(account_features) == len(accounts)
assert len(tx_features) == len(tx)
assert int(account_features["IS_FRAUD"].sum()) == 19
assert int(tx_features["IS_FRAUD"].sum()) == 18

print("\n✓ Account feature table validated.")
print("✓ Transaction feature table validated.")
print("✓ Fraud labels preserved.")
print("✓ Feature engineering complete.")

print("\nSaved:")
print(
    " -",
    account_feature_path
)
print(
    " -",
    transaction_feature_path
)

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 6A
# BASELINE AML DETECTION
# Logistic Regression + Random Forest + XGBoost
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# 1. Load TraceGraph transaction features
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

FEATURE_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_transaction_features.csv"
)

df = pd.read_csv(FEATURE_PATH)

print("=" * 70)
print("TRACEGRAPH BASELINE AML DETECTION")
print("=" * 70)

print("Dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. Select numerical model features
# ------------------------------------------------------------

FEATURE_COLUMNS = [
    "TX_AMOUNT",
    "TIMESTAMP",

    "sender_total_transactions",
    "sender_total_amount",
    "sender_out_degree",
    "sender_fan_out",
    "sender_max_velocity",
    "sender_temporal_concentration",

    "receiver_total_transactions",
    "receiver_total_amount",
    "receiver_in_degree",
    "receiver_fan_in",
    "receiver_max_velocity",
    "receiver_temporal_concentration",

    "sender_receiver_amount_ratio",
    "amount_vs_sender_average",
    "amount_vs_receiver_average"
]

X = df[FEATURE_COLUMNS].copy()
y = df["IS_FRAUD"].astype(int)

# ------------------------------------------------------------
# 3. Clean numerical values
# ------------------------------------------------------------

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)

print("\nNumber of features:", X.shape[1])

print(
    "Fraud transactions:",
    int(y.sum())
)

print(
    "Normal transactions:",
    int((y == 0).sum())
)

# ------------------------------------------------------------
# 4. Stratified train/test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTrain samples:", len(X_train))
print("Test samples:", len(X_test))

print(
    "Train fraud:",
    int(y_train.sum())
)

print(
    "Test fraud:",
    int(y_test.sum())
)

# ------------------------------------------------------------
# 5. Evaluation function
# ------------------------------------------------------------

def evaluate_model(
    model_name,
    model,
    X_test,
    y_test,
    probabilities
):

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        probabilities
    )

    pr_auc = average_precision_score(
        y_test,
        probabilities
    )

    cm = confusion_matrix(
        y_test,
        predictions
    )

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1 Score  : {f1:.4f}"
    )

    print(
        f"ROC-AUC   : {roc_auc:.4f}"
    )

    print(
        f"PR-AUC    : {pr_auc:.4f}"
    )

    print("\nConfusion Matrix:")
    print(cm)

    return {
        "model": model_name,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "true_negatives": int(cm[0, 0]),
        "false_positives": int(cm[0, 1]),
        "false_negatives": int(cm[1, 0]),
        "true_positives": int(cm[1, 1])
    }


results = []

# ============================================================
# 6. Logistic Regression
# ============================================================

print("\nTraining Logistic Regression...")

logistic_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )
])

logistic_model.fit(
    X_train,
    y_train
)

logistic_probabilities = (
    logistic_model.predict_proba(X_test)[:, 1]
)

results.append(
    evaluate_model(
        "Logistic Regression",
        logistic_model,
        X_test,
        y_test,
        logistic_probabilities
    )
)

# ============================================================
# 7. Random Forest
# ============================================================

print("\nTraining Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_probabilities = (
    rf_model.predict_proba(X_test)[:, 1]
)

results.append(
    evaluate_model(
        "Random Forest",
        rf_model,
        X_test,
        y_test,
        rf_probabilities
    )
)

# ============================================================
# 8. XGBoost
# ============================================================

print("\nChecking XGBoost availability...")

try:

    from xgboost import XGBClassifier

    # Calculate class imbalance
    negative_count = int((y_train == 0).sum())
    positive_count = int((y_train == 1).sum())

    scale_pos_weight = (
        negative_count / positive_count
    )

    print(
        "XGBoost scale_pos_weight:",
        round(scale_pos_weight, 2)
    )

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    xgb_model.fit(
        X_train,
        y_train
    )

    xgb_probabilities = (
        xgb_model.predict_proba(X_test)[:, 1]
    )

    results.append(
        evaluate_model(
            "XGBoost",
            xgb_model,
            X_test,
            y_test,
            xgb_probabilities
        )
    )

    xgb_available = True

except Exception as e:

    print(
        "\nXGBoost training failed:",
        str(e)
    )

    xgb_available = False

# ============================================================
# 9. Results table
# ============================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("BASELINE MODEL COMPARISON")
print("=" * 70)

print(
    results_df[
        [
            "model",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "false_positives",
            "false_negatives",
            "true_positives"
        ]
    ].to_string(index=False)
)

# ============================================================
# 10. Save results
# ============================================================

RESULT_PATH = os.path.join(
    PROJECT_DIR,
    "results/baseline_model_results.csv"
)

results_df.to_csv(
    RESULT_PATH,
    index=False
)

# Save trained models for later comparison
import joblib

joblib.dump(
    logistic_model,
    os.path.join(
        PROJECT_DIR,
        "models/logistic_regression_baseline.pkl"
    )
)

joblib.dump(
    rf_model,
    os.path.join(
        PROJECT_DIR,
        "models/random_forest_baseline.pkl"
    )
)

if xgb_available:

    joblib.dump(
        xgb_model,
        os.path.join(
            PROJECT_DIR,
            "models/xgboost_baseline.pkl"
        )
    )

print("\n" + "=" * 70)
print("BASELINE COMPLETE")
print("=" * 70)

print(
    "Results saved to:",
    RESULT_PATH
)

print("Models saved to:")
print(" - models/logistic_regression_baseline.pkl")
print(" - models/random_forest_baseline.pkl")

if xgb_available:
    print(" - models/xgboost_baseline.pkl")

print("\n✓ Baseline AML models trained successfully.")

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 7
# MULTI-CHANNEL TEMPORAL GRAPH REPRESENTATION
#
# Channels:
#   0 → Transaction Amount
#   1 → Transaction Count
#   2 → Transaction Velocity
# ============================================================

import os
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TX_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/transactions_validated.csv"
)

ACCOUNTS_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/accounts_validated.csv"
)

tx = pd.read_csv(TX_PATH)
accounts = pd.read_csv(ACCOUNTS_PATH)

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------

tx["TX_ID"] = tx["TX_ID"].astype(int)
tx["SENDER_ACCOUNT_ID"] = tx["SENDER_ACCOUNT_ID"].astype(int)
tx["RECEIVER_ACCOUNT_ID"] = tx["RECEIVER_ACCOUNT_ID"].astype(int)
tx["TX_AMOUNT"] = pd.to_numeric(tx["TX_AMOUNT"])
tx["TIMESTAMP"] = tx["TIMESTAMP"].astype(int)
tx["IS_FRAUD"] = tx["IS_FRAUD"].astype(bool)

account_ids = sorted(
    accounts["ACCOUNT_ID"].astype(int).unique()
)

account_to_index = {
    account_id: idx
    for idx, account_id in enumerate(account_ids)
}

N = len(account_ids)

print("=" * 70)
print("TRACEGRAPH MULTI-CHANNEL GRAPH REPRESENTATION")
print("=" * 70)

print("Number of accounts:", N)
print("Matrix dimensions:", f"{N} × {N} × 3")

# ------------------------------------------------------------
# 2. Window configuration
# ------------------------------------------------------------

WINDOW_SIZE = 10

MIN_TIMESTAMP = int(tx["TIMESTAMP"].min())
MAX_TIMESTAMP = int(tx["TIMESTAMP"].max())

window_starts = list(
    range(
        MIN_TIMESTAMP,
        MAX_TIMESTAMP + 1,
        WINDOW_SIZE
    )
)

print(
    "Number of short windows:",
    len(window_starts)
)

# ------------------------------------------------------------
# 3. Function to construct one multi-channel matrix
# ------------------------------------------------------------

def build_multichannel_matrix(
    window_tx,
    num_nodes,
    account_to_index
):

    # --------------------------------------------------------
    # Channel 1: transaction amount
    # --------------------------------------------------------

    amount_matrix = np.zeros(
        (num_nodes, num_nodes),
        dtype=np.float32
    )

    # --------------------------------------------------------
    # Channel 2: transaction count
    # --------------------------------------------------------

    count_matrix = np.zeros(
        (num_nodes, num_nodes),
        dtype=np.float32
    )

    # --------------------------------------------------------
    # Channel 3: transaction velocity
    # --------------------------------------------------------

    velocity_matrix = np.zeros(
        (num_nodes, num_nodes),
        dtype=np.float32
    )

    if len(window_tx) == 0:

        return np.stack(
            [
                amount_matrix,
                count_matrix,
                velocity_matrix
            ],
            axis=0
        )

    # Aggregate sender → receiver pairs
    pair_groups = (
        window_tx
        .groupby(
            [
                "SENDER_ACCOUNT_ID",
                "RECEIVER_ACCOUNT_ID"
            ]
        )
    )

    for (
        sender,
        receiver
    ), group in pair_groups:

        sender = int(sender)
        receiver = int(receiver)

        i = account_to_index[sender]
        j = account_to_index[receiver]

        # ----------------------------------------------------
        # Amount
        # ----------------------------------------------------

        total_amount = float(
            group["TX_AMOUNT"].sum()
        )

        amount_matrix[i, j] = total_amount

        # ----------------------------------------------------
        # Count
        # ----------------------------------------------------

        transaction_count = len(group)

        count_matrix[i, j] = float(
            transaction_count
        )

        # ----------------------------------------------------
        # Velocity
        #
        # Number of transactions divided by the temporal
        # span in which the pair was active.
        # +1 avoids division by zero for same-timestamp events.
        # ----------------------------------------------------

        timestamp_span = (
            int(group["TIMESTAMP"].max())
            -
            int(group["TIMESTAMP"].min())
            +
            1
        )

        velocity = (
            transaction_count
            /
            timestamp_span
        )

        velocity_matrix[i, j] = float(
            velocity
        )

    # --------------------------------------------------------
    # Stack channels
    # Shape = (3, N, N)
    # --------------------------------------------------------

    multi_channel = np.stack(
        [
            amount_matrix,
            count_matrix,
            velocity_matrix
        ],
        axis=0
    )

    return multi_channel


# ------------------------------------------------------------
# 4. Build matrices for every short temporal window
# ------------------------------------------------------------

graph_tensor_list = []
window_metadata = []

for window_start in window_starts:

    window_end = min(
        window_start + WINDOW_SIZE - 1,
        MAX_TIMESTAMP
    )

    window_tx = tx[
        (tx["TIMESTAMP"] >= window_start)
        &
        (tx["TIMESTAMP"] <= window_end)
    ].copy()

    tensor = build_multichannel_matrix(
        window_tx,
        N,
        account_to_index
    )

    graph_tensor_list.append(tensor)

    fraud_count = int(
        window_tx["IS_FRAUD"].sum()
    )

    window_metadata.append({

        "window_start": window_start,

        "window_end": window_end,

        "transactions": len(window_tx),

        "fraud_transactions": fraud_count,

        "window_label": int(
            fraud_count > 0
        )
    })


# ------------------------------------------------------------
# 5. Convert to NumPy tensor
# ------------------------------------------------------------

graph_tensors = np.stack(
    graph_tensor_list,
    axis=0
)

window_metadata = pd.DataFrame(
    window_metadata
)

print("\nTensor shape:")
print(
    graph_tensors.shape
)

print(
    "\nExpected shape:",
    f"(20, 3, {N}, {N})"
)

# ------------------------------------------------------------
# 6. Validate tensor dimensions
# ------------------------------------------------------------

assert graph_tensors.shape == (
    len(window_starts),
    3,
    N,
    N
)

# ------------------------------------------------------------
# 7. Validate channel contents
# ------------------------------------------------------------

amount_channel = graph_tensors[:, 0, :, :]
count_channel = graph_tensors[:, 1, :, :]
velocity_channel = graph_tensors[:, 2, :, :]

print("\nChannel statistics")
print("-" * 70)

print(
    "Amount channel total:",
    round(float(amount_channel.sum()), 2)
)

print(
    "Count channel total:",
    int(count_channel.sum())
)

print(
    "Velocity channel total:",
    round(float(velocity_channel.sum()), 4)
)

# Amount and count should preserve the complete dataset
assert np.isclose(
    amount_channel.sum(),
    tx["TX_AMOUNT"].sum()
)

assert int(
    count_channel.sum()
) == len(tx)

# ------------------------------------------------------------
# 8. Validate fraud-window labels
# ------------------------------------------------------------

print("\nWindow labels")
print("-" * 70)

print(
    window_metadata[
        [
            "window_start",
            "window_end",
            "transactions",
            "fraud_transactions",
            "window_label"
        ]
    ].to_string(index=False)
)

assert (
    int(
        window_metadata["fraud_transactions"].sum()
    )
    ==
    int(tx["IS_FRAUD"].sum())
)

# ------------------------------------------------------------
# 9. Save tensor
# ------------------------------------------------------------

tensor_path = os.path.join(
    PROJECT_DIR,
    "data/processed/short_multichannel_graphs.npy"
)

metadata_path = os.path.join(
    PROJECT_DIR,
    "data/processed/short_multichannel_metadata.csv"
)

np.save(
    tensor_path,
    graph_tensors
)

window_metadata.to_csv(
    metadata_path,
    index=False
)

# ------------------------------------------------------------
# 10. Save account mapping
# ------------------------------------------------------------

mapping_df = pd.DataFrame({
    "matrix_index": range(N),
    "ACCOUNT_ID": account_ids
})

mapping_path = os.path.join(
    PROJECT_DIR,
    "data/processed/account_matrix_mapping.csv"
)

mapping_df.to_csv(
    mapping_path,
    index=False
)

# ------------------------------------------------------------
# 11. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MULTI-CHANNEL REPRESENTATION COMPLETE")
print("=" * 70)

print(
    "Tensor shape:",
    graph_tensors.shape
)

print(
    "Amount channel shape:",
    amount_channel.shape
)

print(
    "Count channel shape:",
    count_channel.shape
)

print(
    "Velocity channel shape:",
    velocity_channel.shape
)

print(
    "\nFraud windows:",
    int(window_metadata["window_label"].sum())
)

print(
    "Normal windows:",
    int(
        (window_metadata["window_label"] == 0).sum()
    )
)

print("\nSaved:")
print(
    " -",
    tensor_path
)
print(
    " -",
    metadata_path
)
print(
    " -",
    mapping_path
)

print("\n✓ Multi-channel graph tensors validated.")
print("✓ Amount totals preserved.")
print("✓ Transaction counts preserved.")
print("✓ Fraud labels preserved.")

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 8
# CNN + RESNET-18 TEMPORAL GRAPH DETECTION
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TENSOR_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/short_multichannel_graphs.npy"
)

METADATA_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/short_multichannel_metadata.csv"
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("TRACEGRAPH DEEP LEARNING")
print("=" * 70)

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load graph tensors
# ------------------------------------------------------------

X = np.load(TENSOR_PATH).astype(
    np.float32
)

metadata = pd.read_csv(
    METADATA_PATH
)

y = metadata[
    "window_label"
].values.astype(np.int64)

print(
    "\nInput tensor shape:",
    X.shape
)

print(
    "Labels:",
    y
)

print(
    "Class distribution:",
    np.bincount(y)
)

# ------------------------------------------------------------
# 3. Normalize channels
# ------------------------------------------------------------

# Normalize each channel globally.
#
# This prevents the amount channel from dominating simply
# because its numerical scale is much larger than count/velocity.

X_normalized = X.copy()

for channel in range(3):

    channel_data = X[:, channel]

    mean = channel_data.mean()

    std = channel_data.std()

    if std > 0:

        X_normalized[:, channel] = (
            channel_data - mean
        ) / std

print(
    "\nChannel means after normalization:",
    X_normalized.mean(
        axis=(0, 2, 3)
    )
)

print(
    "Channel stds after normalization:",
    X_normalized.std(
        axis=(0, 2, 3)
    )
)

# ------------------------------------------------------------
# 4. Stratified train / validation / test split
# ------------------------------------------------------------

indices = np.arange(
    len(X_normalized)
)

# First: 70% train, 30% temporary
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

# Second: split temporary into validation/test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=y[temp_idx]
)

print("\nDataset split:")
print(
    "Train:",
    len(train_idx),
    "samples"
)

print(
    "Validation:",
    len(val_idx),
    "samples"
)

print(
    "Test:",
    len(test_idx),
    "samples"
)

print(
    "Train class distribution:",
    np.bincount(
        y[train_idx],
        minlength=2
    )
)

print(
    "Validation class distribution:",
    np.bincount(
        y[val_idx],
        minlength=2
    )
)

print(
    "Test class distribution:",
    np.bincount(
        y[test_idx],
        minlength=2
    )
)

# ------------------------------------------------------------
# 5. Convert to PyTorch tensors
# ------------------------------------------------------------

X_train = torch.tensor(
    X_normalized[train_idx],
    dtype=torch.float32
)

X_val = torch.tensor(
    X_normalized[val_idx],
    dtype=torch.float32
)

X_test = torch.tensor(
    X_normalized[test_idx],
    dtype=torch.float32
)

y_train = torch.tensor(
    y[train_idx],
    dtype=torch.float32
)

y_val = torch.tensor(
    y[val_idx],
    dtype=torch.float32
)

y_test = torch.tensor(
    y[test_idx],
    dtype=torch.float32
)

# ------------------------------------------------------------
# 6. DataLoaders
# ------------------------------------------------------------

train_dataset = TensorDataset(
    X_train,
    y_train
)

val_dataset = TensorDataset(
    X_val,
    y_val
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

# ============================================================
# 7. Custom CNN
# ============================================================

class TraceGraphCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3,
                16,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(16),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.AdaptiveAvgPool2d(
                (1, 1)
            )
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64,
                32
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                32,
                1
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x.squeeze(1)


# ------------------------------------------------------------
# 8. Training/evaluation helpers
# ------------------------------------------------------------

def train_model(
    model,
    train_loader,
    val_loader,
    epochs=30
):

    model = model.to(DEVICE)

    # Class weighting
    positive_count = y_train.sum().item()
    negative_count = (
        len(y_train)
        -
        positive_count
    )

    pos_weight = torch.tensor(
        [
            negative_count
            /
            max(positive_count, 1)
        ],
        dtype=torch.float32
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    best_val_loss = float("inf")

    best_state = None

    history = []

    for epoch in range(epochs):

        model.train()

        train_loss = 0.0

        for batch_x, batch_y in train_loader:

            batch_x = batch_x.to(DEVICE)

            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()

            logits = model(batch_x)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            optimizer.step()

            train_loss += (
                loss.item()
                *
                len(batch_x)
            )

        train_loss /= len(
            train_loader.dataset
        )

        # Validation
        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for batch_x, batch_y in val_loader:

                batch_x = batch_x.to(DEVICE)

                batch_y = batch_y.to(DEVICE)

                logits = model(batch_x)

                loss = criterion(
                    logits,
                    batch_y
                )

                val_loss += (
                    loss.item()
                    *
                    len(batch_x)
                )

        val_loss /= len(
            val_loader.dataset
        )

        history.append(
            [
                epoch + 1,
                train_loss,
                val_loss
            ]
        )

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = {
                k: v.cpu().clone()
                for k, v in
                model.state_dict().items()
            }

        if (
            epoch == 0
            or
            (epoch + 1) % 5 == 0
        ):

            print(
                f"Epoch {epoch + 1:02d}/{epochs} "
                f"| Train Loss: {train_loss:.4f} "
                f"| Val Loss: {val_loss:.4f}"
            )

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    return model, pd.DataFrame(
        history,
        columns=[
            "epoch",
            "train_loss",
            "val_loss"
        ]
    )


def predict_model(
    model,
    loader
):

    model.eval()

    probabilities = []

    labels = []

    with torch.no_grad():

        for batch_x, batch_y in loader:

            batch_x = batch_x.to(DEVICE)

            logits = model(batch_x)

            probs = torch.sigmoid(
                logits
            )

            probabilities.extend(
                probs.cpu().numpy()
            )

            labels.extend(
                batch_y.numpy()
            )

    return (
        np.array(labels),
        np.array(probabilities)
    )


def evaluate_predictions(
    name,
    labels,
    probabilities
):

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    precision = precision_score(
        labels,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        zero_division=0
    )

    # ROC-AUC requires both classes
    if len(np.unique(labels)) == 2:

        roc_auc = roc_auc_score(
            labels,
            probabilities
        )

        pr_auc = average_precision_score(
            labels,
            probabilities
        )

    else:

        roc_auc = np.nan

        pr_auc = np.nan

    cm = confusion_matrix(
        labels,
        predictions
    )

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1        : {f1:.4f}"
    )

    print(
        f"ROC-AUC   : {roc_auc:.4f}"
    )

    print(
        f"PR-AUC    : {pr_auc:.4f}"
    )

    print("\nConfusion Matrix:")
    print(cm)

    return {
        "model": name,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "false_positives": int(cm[0, 1]),
        "false_negatives": int(cm[1, 0]),
        "true_positives": int(cm[1, 1])
    }


# ============================================================
# 9. Train Custom CNN
# ============================================================

print("\n" + "=" * 70)
print("TRAINING CUSTOM TRACEGRAPH CNN")
print("=" * 70)

cnn_model = TraceGraphCNN()

cnn_model, cnn_history = train_model(
    cnn_model,
    train_loader,
    val_loader,
    epochs=30
)

# ------------------------------------------------------------
# Evaluate CNN
# ------------------------------------------------------------

cnn_labels, cnn_probabilities = predict_model(
    cnn_model,
    test_loader
)

cnn_result = evaluate_predictions(
    "TraceGraph CNN",
    cnn_labels,
    cnn_probabilities
)

# ------------------------------------------------------------
# Save CNN
# ------------------------------------------------------------

cnn_path = os.path.join(
    PROJECT_DIR,
    "models/tracegraph_cnn.pt"
)

torch.save(
    cnn_model.state_dict(),
    cnn_path
)

# ============================================================
# 10. ResNet-18
# ============================================================

print("\n" + "=" * 70)
print("TRAINING RESNET-18")
print("=" * 70)

try:

    import torchvision

    from torchvision.models import resnet18

    resnet_model = resnet18(
        weights=None
    )

    # Replace ImageNet's 3-class output
    resnet_model.fc = nn.Linear(
        resnet_model.fc.in_features,
        1
    )

    resnet_model = resnet_model.to(
        DEVICE
    )

    # Same weighted loss
    positive_count = y_train.sum().item()

    negative_count = (
        len(y_train)
        -
        positive_count
    )

    pos_weight = torch.tensor(
        [
            negative_count
            /
            max(positive_count, 1)
        ],
        dtype=torch.float32
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.Adam(
        resnet_model.parameters(),
        lr=0.0003,
        weight_decay=1e-4
    )

    best_val_loss = float("inf")

    best_state = None

    for epoch in range(20):

        resnet_model.train()

        train_loss = 0.0

        for batch_x, batch_y in train_loader:

            batch_x = batch_x.to(DEVICE)

            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()

            logits = resnet_model(
                batch_x
            ).squeeze(1)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            optimizer.step()

            train_loss += (
                loss.item()
                *
                len(batch_x)
            )

        train_loss /= len(
            train_loader.dataset
        )

        resnet_model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for batch_x, batch_y in val_loader:

                batch_x = batch_x.to(DEVICE)

                batch_y = batch_y.to(DEVICE)

                logits = resnet_model(
                    batch_x
                ).squeeze(1)

                loss = criterion(
                    logits,
                    batch_y
                )

                val_loss += (
                    loss.item()
                    *
                    len(batch_x)
                )

        val_loss /= len(
            val_loader.dataset
        )

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_state = {
                k: v.cpu().clone()
                for k, v in
                resnet_model.state_dict().items()
            }

        if (
            epoch == 0
            or
            (epoch + 1) % 5 == 0
        ):

            print(
                f"Epoch {epoch + 1:02d}/20 "
                f"| Train Loss: {train_loss:.4f} "
                f"| Val Loss: {val_loss:.4f}"
            )

    if best_state is not None:

        resnet_model.load_state_dict(
            best_state
        )

    # --------------------------------------------------------
    # Evaluate ResNet
    # --------------------------------------------------------

    resnet_labels, resnet_probabilities = (
        predict_model(
            resnet_model,
            test_loader
        )
    )

    resnet_result = evaluate_predictions(
        "TraceGraph ResNet-18",
        resnet_labels,
        resnet_probabilities
    )

    resnet_available = True

    resnet_path = os.path.join(
        PROJECT_DIR,
        "models/tracegraph_resnet18.pt"
    )

    torch.save(
        resnet_model.state_dict(),
        resnet_path
    )

except Exception as e:

    print(
        "\nResNet-18 training unavailable:"
    )

    print(str(e))

    resnet_available = False

# ============================================================
# 11. Compare deep-learning models
# ============================================================

deep_learning_results = [
    cnn_result
]

if resnet_available:

    deep_learning_results.append(
        resnet_result
    )

deep_results_df = pd.DataFrame(
    deep_learning_results
)

print("\n" + "=" * 70)
print("DEEP LEARNING MODEL COMPARISON")
print("=" * 70)

print(
    deep_results_df.to_string(
        index=False
    )
)

# ============================================================
# 12. Save results
# ============================================================

deep_results_path = os.path.join(
    PROJECT_DIR,
    "results/deep_learning_results.csv"
)

deep_results_df.to_csv(
    deep_results_path,
    index=False
)

cnn_history_path = os.path.join(
    PROJECT_DIR,
    "results/cnn_training_history.csv"
)

cnn_history.to_csv(
    cnn_history_path,
    index=False
)

print("\nSaved:")
print(
    " -",
    cnn_path
)

if resnet_available:

    print(
        " -",
        resnet_path
    )

print(
    " -",
    deep_results_path
)

print(
    " -",
    cnn_history_path
)

print("\n✓ Deep-learning stage complete.")

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 9
# SLIDING TEMPORAL GRAPH DATASET
#
# Creates overlapping temporal graph samples while preserving
# chronological train/validation/test separation.
# ============================================================

import os
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TX_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/transactions_validated.csv"
)

ACCOUNTS_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/accounts_validated.csv"
)

tx = pd.read_csv(TX_PATH)
accounts = pd.read_csv(ACCOUNTS_PATH)

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------

tx["TX_ID"] = tx["TX_ID"].astype(int)
tx["SENDER_ACCOUNT_ID"] = tx["SENDER_ACCOUNT_ID"].astype(int)
tx["RECEIVER_ACCOUNT_ID"] = tx["RECEIVER_ACCOUNT_ID"].astype(int)
tx["TX_AMOUNT"] = pd.to_numeric(tx["TX_AMOUNT"])
tx["TIMESTAMP"] = tx["TIMESTAMP"].astype(int)
tx["IS_FRAUD"] = tx["IS_FRAUD"].astype(bool)

account_ids = sorted(
    accounts["ACCOUNT_ID"].astype(int).unique()
)

account_to_index = {
    account_id: idx
    for idx, account_id in enumerate(account_ids)
}

N = len(account_ids)

WINDOW_SIZE = 10
STRIDE = 1

MIN_TIMESTAMP = int(
    tx["TIMESTAMP"].min()
)

MAX_TIMESTAMP = int(
    tx["TIMESTAMP"].max()
)

# ------------------------------------------------------------
# 2. Multi-channel graph construction
# ------------------------------------------------------------

def build_multichannel_matrix(
    window_tx,
    num_nodes,
    account_to_index
):

    amount_matrix = np.zeros(
        (num_nodes, num_nodes),
        dtype=np.float32
    )

    count_matrix = np.zeros(
        (num_nodes, num_nodes),
        dtype=np.float32
    )

    velocity_matrix = np.zeros(
        (num_nodes, num_nodes),
        dtype=np.float32
    )

    if len(window_tx) == 0:

        return np.stack(
            [
                amount_matrix,
                count_matrix,
                velocity_matrix
            ],
            axis=0
        )

    pair_groups = window_tx.groupby(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID"
        ]
    )

    for (
        sender,
        receiver
    ), group in pair_groups:

        sender = int(sender)
        receiver = int(receiver)

        i = account_to_index[sender]
        j = account_to_index[receiver]

        # Channel 0
        total_amount = float(
            group["TX_AMOUNT"].sum()
        )

        amount_matrix[i, j] = (
            total_amount
        )

        # Channel 1
        transaction_count = len(group)

        count_matrix[i, j] = float(
            transaction_count
        )

        # Channel 2
        timestamp_span = (
            int(group["TIMESTAMP"].max())
            -
            int(group["TIMESTAMP"].min())
            +
            1
        )

        velocity = (
            transaction_count
            /
            timestamp_span
        )

        velocity_matrix[i, j] = float(
            velocity
        )

    return np.stack(
        [
            amount_matrix,
            count_matrix,
            velocity_matrix
        ],
        axis=0
    )


# ------------------------------------------------------------
# 3. Generate sliding windows
# ------------------------------------------------------------

graph_samples = []
window_records = []

window_starts = range(
    MIN_TIMESTAMP,
    MAX_TIMESTAMP - WINDOW_SIZE + 2,
    STRIDE
)

for window_start in window_starts:

    window_end = (
        window_start
        +
        WINDOW_SIZE
        -
        1
    )

    window_tx = tx[
        (tx["TIMESTAMP"] >= window_start)
        &
        (tx["TIMESTAMP"] <= window_end)
    ].copy()

    tensor = build_multichannel_matrix(
        window_tx,
        N,
        account_to_index
    )

    graph_samples.append(
        tensor
    )

    fraud_count = int(
        window_tx["IS_FRAUD"].sum()
    )

    window_records.append({

        "window_start":
            window_start,

        "window_end":
            window_end,

        "transactions":
            len(window_tx),

        "fraud_transactions":
            fraud_count,

        "window_label":
            int(fraud_count > 0)
    })


graph_samples = np.stack(
    graph_samples,
    axis=0
)

window_metadata = pd.DataFrame(
    window_records
)

print("=" * 70)
print("TRACEGRAPH SLIDING TEMPORAL DATASET")
print("=" * 70)

print(
    "Tensor shape:",
    graph_samples.shape
)

print(
    "Number of windows:",
    len(window_metadata)
)

print(
    "Window size:",
    WINDOW_SIZE
)

print(
    "Stride:",
    STRIDE
)

# ------------------------------------------------------------
# 4. Validate tensor
# ------------------------------------------------------------

amount_total = float(
    graph_samples[:, 0].sum()
)

transaction_total = int(
    graph_samples[:, 1].sum()
)

print("\nValidation")
print("-" * 70)

print(
    "Original amount total:",
    round(float(tx["TX_AMOUNT"].sum()), 2)
)

print(
    "Sliding-window amount total:",
    round(amount_total, 2)
)

print(
    "Note: sliding-window totals are expected to be larger"
    " because transactions appear in multiple overlapping windows."
)

print(
    "Original transactions:",
    len(tx)
)

print(
    "Sliding-window transaction representation:",
    transaction_total
)

print(
    "Total fraud transactions represented:",
    int(
        window_metadata[
            "fraud_transactions"
        ].sum()
    )
)

# ------------------------------------------------------------
# 5. Chronological split
#
# Entire windows must remain inside their temporal region.
#
# TRAIN:      0–119
# VALIDATION: 120–159
# TEST:       160–199
# ------------------------------------------------------------

TRAIN_END = 119
VAL_END = 159

train_mask = (
    window_metadata["window_end"]
    <= TRAIN_END
)

val_mask = (
    (window_metadata["window_start"] >= 120)
    &
    (window_metadata["window_end"] <= VAL_END)
)

test_mask = (
    window_metadata["window_start"] >= 160
)

train_indices = np.where(
    train_mask
)[0]

val_indices = np.where(
    val_mask
)[0]

test_indices = np.where(
    test_mask
)[0]

print("\nChronological split")
print("-" * 70)

print(
    "Train windows:",
    len(train_indices)
)

print(
    "Validation windows:",
    len(val_indices)
)

print(
    "Test windows:",
    len(test_indices)
)

# ------------------------------------------------------------
# 6. Split labels
# ------------------------------------------------------------

labels = window_metadata[
    "window_label"
].values.astype(np.int64)

print("\nClass distributions")

print(
    "Train:",
    np.bincount(
        labels[train_indices],
        minlength=2
    )
)

print(
    "Validation:",
    np.bincount(
        labels[val_indices],
        minlength=2
    )
)

print(
    "Test:",
    np.bincount(
        labels[test_indices],
        minlength=2
    )
)

# ------------------------------------------------------------
# 7. Show suspicious windows
# ------------------------------------------------------------

print("\nSuspicious windows by split")
print("-" * 70)

print(
    window_metadata[
        window_metadata["window_label"] == 1
    ][
        [
            "window_start",
            "window_end",
            "fraud_transactions"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 8. Save expanded dataset
# ------------------------------------------------------------

SLIDING_TENSOR_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/sliding_multichannel_graphs.npy"
)

SLIDING_METADATA_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/sliding_multichannel_metadata.csv"
)

np.save(
    SLIDING_TENSOR_PATH,
    graph_samples
)

window_metadata.to_csv(
    SLIDING_METADATA_PATH,
    index=False
)

# ------------------------------------------------------------
# 9. Save split indices
# ------------------------------------------------------------

split_df = pd.DataFrame({

    "index":
        np.arange(
            len(window_metadata)
        ),

    "split":
        [
            (
                "train"
                if i in set(train_indices)
                else
                "validation"
                if i in set(val_indices)
                else
                "test"
            )
            for i in range(
                len(window_metadata)
            )
        ]
})

SPLIT_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/sliding_temporal_splits.csv"
)

split_df.to_csv(
    SPLIT_PATH,
    index=False
)

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SLIDING TEMPORAL DATASET COMPLETE")
print("=" * 70)

print(
    "Tensor:",
    SLIDING_TENSOR_PATH
)

print(
    "Metadata:",
    SLIDING_METADATA_PATH
)

print(
    "Splits:",
    SPLIT_PATH
)

print(
    "\n✓ Overlapping temporal windows generated."
)

print(
    "✓ Chronological split created."
)

print(
    "✓ No random temporal leakage."
)

print(
    "✓ Multi-channel representation preserved."
)

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 10 CORRECTED
# CNN + RESNET-18
#
# IMPORTANT:
#   Train      : window starts 0–110
#   Validation : window starts 120–150
#   Test       : window starts 160–190
#
# Boundary windows 111–119 and 151–159 are excluded so that
# no graph window crosses a train/validation/test boundary.
# ============================================================

import os
import random
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# ============================================================
# 1. Configuration
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TENSOR_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/sliding_multichannel_graphs.npy"
)

METADATA_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/sliding_multichannel_metadata.csv"
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("TRACEGRAPH — CORRECTED TEMPORAL DEEP LEARNING")
print("=" * 70)

print("Device:", DEVICE)

# ============================================================
# 2. Load data
# ============================================================

X = np.load(
    TENSOR_PATH
).astype(np.float32)

metadata = pd.read_csv(
    METADATA_PATH
)

labels = metadata[
    "window_label"
].values.astype(np.int64)

starts = metadata[
    "window_start"
].values.astype(int)

print("\nOriginal dataset:")
print("Samples:", len(X))
print("Tensor:", X.shape)

# ============================================================
# 3. CORRECT chronological split
# ============================================================

train_mask = (
    (starts >= 0)
    &
    (starts <= 110)
)

val_mask = (
    (starts >= 120)
    &
    (starts <= 150)
)

test_mask = (
    (starts >= 160)
    &
    (starts <= 190)
)

excluded_mask = ~(
    train_mask
    |
    val_mask
    |
    test_mask
)

train_idx = np.where(
    train_mask
)[0]

val_idx = np.where(
    val_mask
)[0]

test_idx = np.where(
    test_mask
)[0]

excluded_idx = np.where(
    excluded_mask
)[0]

print("\nCorrected chronological split")
print("-" * 70)

print(
    "Train:",
    len(train_idx)
)

print(
    "Validation:",
    len(val_idx)
)

print(
    "Test:",
    len(test_idx)
)

print(
    "Excluded boundary windows:",
    len(excluded_idx)
)

print("\nExpected:")
print("Train = 111")
print("Validation = 31")
print("Test = 31")
print("Excluded = 18")

assert len(train_idx) == 111
assert len(val_idx) == 31
assert len(test_idx) == 31
assert len(excluded_idx) == 18

print("\nClass distributions")

print(
    "Train:",
    np.bincount(
        labels[train_idx],
        minlength=2
    )
)

print(
    "Validation:",
    np.bincount(
        labels[val_idx],
        minlength=2
    )
)

print(
    "Test:",
    np.bincount(
        labels[test_idx],
        minlength=2
    )
)

# ============================================================
# 4. Training-only normalization
# ============================================================

X_norm = X.copy()

channel_means = []
channel_stds = []

print("\nTraining-only normalization")
print("-" * 70)

for c in range(3):

    train_values = X[
        train_idx,
        c
    ]

    mean = float(
        train_values.mean()
    )

    std = float(
        train_values.std()
    )

    if std == 0:
        std = 1.0

    channel_means.append(mean)
    channel_stds.append(std)

    X_norm[:, c] = (
        X[:, c] - mean
    ) / std

    print(
        f"Channel {c}: "
        f"mean={mean:.6f}, "
        f"std={std:.6f}"
    )

# ============================================================
# 5. PyTorch tensors
# ============================================================

X_train = torch.tensor(
    X_norm[train_idx],
    dtype=torch.float32
)

X_val = torch.tensor(
    X_norm[val_idx],
    dtype=torch.float32
)

X_test = torch.tensor(
    X_norm[test_idx],
    dtype=torch.float32
)

y_train = torch.tensor(
    labels[train_idx],
    dtype=torch.float32
)

y_val = torch.tensor(
    labels[val_idx],
    dtype=torch.float32
)

y_test = torch.tensor(
    labels[test_idx],
    dtype=torch.float32
)

# ============================================================
# 6. DataLoaders
# ============================================================

train_loader = DataLoader(
    TensorDataset(
        X_train,
        y_train
    ),
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(
        X_val,
        y_val
    ),
    batch_size=8,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(
        X_test,
        y_test
    ),
    batch_size=8,
    shuffle=False
)

# ============================================================
# 7. CNN
# ============================================================

class TraceGraphCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3,
                16,
                3,
                padding=1
            ),

            nn.BatchNorm2d(16),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                16,
                32,
                3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.AdaptiveAvgPool2d(
                (1, 1)
            )
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64,
                32
            ),

            nn.ReLU(),

            nn.Dropout(0.4),

            nn.Linear(
                32,
                1
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x.squeeze(1)


# ============================================================
# 8. ResNet wrapper
# ============================================================

from torchvision.models import resnet18


class TraceGraphResNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = resnet18(
            weights=None
        )

        self.model.fc = nn.Linear(
            self.model.fc.in_features,
            1
        )

    def forward(self, x):

        return self.model(
            x
        ).squeeze(1)


# ============================================================
# 9. Prediction helper
# ============================================================

def predict(
    model,
    loader
):

    model.eval()

    probabilities = []
    actual = []

    with torch.no_grad():

        for batch_x, batch_y in loader:

            batch_x = batch_x.to(
                DEVICE
            )

            logits = model(
                batch_x
            )

            probs = torch.sigmoid(
                logits
            )

            probabilities.extend(
                probs.cpu().numpy()
            )

            actual.extend(
                batch_y.numpy()
            )

    return (
        np.asarray(actual),
        np.asarray(probabilities)
    )


# ============================================================
# 10. Threshold selection
# ============================================================

def find_best_threshold(
    y_true,
    probabilities
):

    thresholds = np.arange(
        0.05,
        0.96,
        0.01
    )

    best_threshold = 0.5
    best_f1 = -1

    for threshold in thresholds:

        predictions = (
            probabilities
            >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            predictions,
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score
            best_threshold = float(
                threshold
            )

    return (
        best_threshold,
        best_f1
    )


# ============================================================
# 11. Evaluation
# ============================================================

def evaluate(
    model_name,
    y_true,
    probabilities,
    threshold
):

    predictions = (
        probabilities
        >= threshold
    ).astype(int)

    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_true,
        probabilities
    )

    pr_auc = average_precision_score(
        y_true,
        probabilities
    )

    cm = confusion_matrix(
        y_true,
        predictions
    )

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    print(
        f"Threshold : {threshold:.2f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1        : {f1:.4f}"
    )

    print(
        f"ROC-AUC   : {roc_auc:.4f}"
    )

    print(
        f"PR-AUC    : {pr_auc:.4f}"
    )

    print("\nConfusion Matrix:")
    print(cm)

    return {
        "model": model_name,
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "true_negatives": int(cm[0, 0]),
        "false_positives": int(cm[0, 1]),
        "false_negatives": int(cm[1, 0]),
        "true_positives": int(cm[1, 1])
    }


# ============================================================
# 12. Training function
# ============================================================

def train_model(
    model,
    name,
    epochs,
    learning_rate,
    patience=8
):

    model = model.to(
        DEVICE
    )

    positive_count = float(
        y_train.sum()
    )

    negative_count = float(
        len(y_train)
        -
        positive_count
    )

    pos_weight_value = (
        negative_count
        /
        positive_count
    )

    pos_weight = torch.tensor(
        [pos_weight_value],
        dtype=torch.float32,
        device=DEVICE
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=3
        )
    )

    best_loss = float("inf")
    best_state = None
    wait = 0

    history = []

    print("\n" + "=" * 70)
    print(
        "Training:",
        name
    )
    print("=" * 70)

    print(
        "Positive class weight:",
        round(
            pos_weight_value,
            4
        )
    )

    for epoch in range(
        1,
        epochs + 1
    ):

        # -------------------------
        # Train
        # -------------------------

        model.train()

        train_total = 0.0

        for batch_x, batch_y in train_loader:

            batch_x = batch_x.to(
                DEVICE
            )

            batch_y = batch_y.to(
                DEVICE
            )

            optimizer.zero_grad()

            logits = model(
                batch_x
            )

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            optimizer.step()

            train_total += (
                loss.item()
                *
                len(batch_x)
            )

        train_loss = (
            train_total
            /
            len(train_loader.dataset)
        )

        # -------------------------
        # Validation
        # -------------------------

        model.eval()

        val_total = 0.0

        with torch.no_grad():

            for batch_x, batch_y in val_loader:

                batch_x = batch_x.to(
                    DEVICE
                )

                batch_y = batch_y.to(
                    DEVICE
                )

                logits = model(
                    batch_x
                )

                loss = criterion(
                    logits,
                    batch_y
                )

                val_total += (
                    loss.item()
                    *
                    len(batch_x)
                )

        val_loss = (
            val_total
            /
            len(val_loader.dataset)
        )

        scheduler.step(
            val_loss
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "learning_rate":
                optimizer.param_groups[0]["lr"]
        })

        # -------------------------
        # Save best
        # -------------------------

        if val_loss < best_loss:

            best_loss = val_loss

            best_state = copy.deepcopy(
                model.state_dict()
            )

            wait = 0

        else:

            wait += 1

        if (
            epoch == 1
            or epoch % 5 == 0
        ):

            print(
                f"Epoch {epoch:02d}/{epochs} "
                f"| Train: {train_loss:.4f} "
                f"| Val: {val_loss:.4f} "
                f"| LR: "
                f"{optimizer.param_groups[0]['lr']:.6f}"
            )

        if wait >= patience:

            print(
                f"Early stopping at "
                f"epoch {epoch}"
            )

            break

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    return (
        model,
        pd.DataFrame(history)
    )


# ============================================================
# 13. Train CNN
# ============================================================

cnn_model = TraceGraphCNN()

cnn_model, cnn_history = train_model(
    cnn_model,
    "TraceGraph CNN",
    epochs=50,
    learning_rate=0.001,
    patience=8
)

# Validation threshold
cnn_val_y, cnn_val_probs = predict(
    cnn_model,
    val_loader
)

cnn_threshold, cnn_val_f1 = (
    find_best_threshold(
        cnn_val_y,
        cnn_val_probs
    )
)

print(
    "\nCNN validation threshold:",
    cnn_threshold
)

print(
    "CNN validation F1:",
    cnn_val_f1
)

# Test
cnn_test_y, cnn_test_probs = predict(
    cnn_model,
    test_loader
)

cnn_result = evaluate(
    "TraceGraph CNN — Test",
    cnn_test_y,
    cnn_test_probs,
    cnn_threshold
)

# ============================================================
# 14. Train ResNet-18
# ============================================================

resnet_model = TraceGraphResNet()

resnet_model, resnet_history = train_model(
    resnet_model,
    "TraceGraph ResNet-18",
    epochs=40,
    learning_rate=0.0003,
    patience=7
)

# Validation threshold
resnet_val_y, resnet_val_probs = predict(
    resnet_model,
    val_loader
)

resnet_threshold, resnet_val_f1 = (
    find_best_threshold(
        resnet_val_y,
        resnet_val_probs
    )
)

print(
    "\nResNet validation threshold:",
    resnet_threshold
)

print(
    "ResNet validation F1:",
    resnet_val_f1
)

# Test
resnet_test_y, resnet_test_probs = predict(
    resnet_model,
    test_loader
)

resnet_result = evaluate(
    "TraceGraph ResNet-18 — Test",
    resnet_test_y,
    resnet_test_probs,
    resnet_threshold
)

# ============================================================
# 15. Compare
# ============================================================

results = pd.DataFrame([
    cnn_result,
    resnet_result
])

print("\n" + "=" * 70)
print("FINAL SLIDING GRAPH DEEP LEARNING COMPARISON")
print("=" * 70)

print(
    results.to_string(
        index=False
    )
)

# ============================================================
# 16. Save models
# ============================================================

cnn_path = os.path.join(
    PROJECT_DIR,
    "models/tracegraph_cnn_sliding.pt"
)

resnet_path = os.path.join(
    PROJECT_DIR,
    "models/tracegraph_resnet18_sliding.pt"
)

torch.save(
    cnn_model.state_dict(),
    cnn_path
)

torch.save(
    resnet_model.state_dict(),
    resnet_path
)

# ============================================================
# 17. Save normalization
# ============================================================

normalization_path = os.path.join(
    PROJECT_DIR,
    "models/tracegraph_channel_normalization.npz"
)

np.savez(
    normalization_path,
    means=np.array(
        channel_means
    ),
    stds=np.array(
        channel_stds
    )
)

# ============================================================
# 18. Save threshold information
# ============================================================

threshold_path = os.path.join(
    PROJECT_DIR,
    "results/deep_model_thresholds.csv"
)

pd.DataFrame([
    {
        "model": "TraceGraph CNN",
        "validation_f1": cnn_val_f1,
        "threshold": cnn_threshold
    },
    {
        "model": "TraceGraph ResNet-18",
        "validation_f1": resnet_val_f1,
        "threshold": resnet_threshold
    }
]).to_csv(
    threshold_path,
    index=False
)

# ============================================================
# 19. Save results and histories
# ============================================================

results_path = os.path.join(
    PROJECT_DIR,
    "results/sliding_deep_learning_results.csv"
)

cnn_history_path = os.path.join(
    PROJECT_DIR,
    "results/sliding_cnn_history.csv"
)

resnet_history_path = os.path.join(
    PROJECT_DIR,
    "results/sliding_resnet_history.csv"
)

results.to_csv(
    results_path,
    index=False
)

cnn_history.to_csv(
    cnn_history_path,
    index=False
)

resnet_history.to_csv(
    resnet_history_path,
    index=False
)

print("\n" + "=" * 70)
print("STAGE 10 CORRECTED — COMPLETE")
print("=" * 70)

print("\nSaved:")
print("-", cnn_path)
print("-", resnet_path)
print("-", normalization_path)
print("-", threshold_path)
print("-", results_path)
print("-", cnn_history_path)
print("-", resnet_history_path)

print("\n✓ Correct chronological split")
print("✓ Boundary windows excluded")
print("✓ Training-only normalization")
print("✓ CNN trained")
print("✓ ResNet-18 trained")
print("✓ Threshold selected using validation only")
print("✓ Final test evaluated once")

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 11A
# GRAPH TOPOLOGY + NETWORK FEATURES
# ============================================================

import os
import numpy as np
import pandas as pd
import networkx as nx

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TX_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/transactions_validated.csv"
)

ACCOUNT_FEATURE_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_account_features.csv"
)

TRANSACTION_FEATURE_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_transaction_features.csv"
)

tx = pd.read_csv(TX_PATH)

account_features = pd.read_csv(
    ACCOUNT_FEATURE_PATH
)

transaction_features = pd.read_csv(
    TRANSACTION_FEATURE_PATH
)

# ------------------------------------------------------------
# 1. Prepare transaction data
# ------------------------------------------------------------

tx["SENDER_ACCOUNT_ID"] = (
    tx["SENDER_ACCOUNT_ID"].astype(int)
)

tx["RECEIVER_ACCOUNT_ID"] = (
    tx["RECEIVER_ACCOUNT_ID"].astype(int)
)

tx["TX_AMOUNT"] = pd.to_numeric(
    tx["TX_AMOUNT"]
)

tx["TIMESTAMP"] = (
    tx["TIMESTAMP"].astype(int)
)

tx["IS_FRAUD"] = (
    tx["IS_FRAUD"].astype(bool)
)

# ------------------------------------------------------------
# 2. Build directed transaction graph
# ------------------------------------------------------------

G = nx.DiGraph()

account_ids = sorted(
    account_features["ACCOUNT_ID"]
    .astype(int)
    .unique()
)

G.add_nodes_from(
    account_ids
)

edge_data = (
    tx.groupby(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID"
        ],
        as_index=False
    )
    .agg(
        transaction_count=(
            "TX_ID",
            "count"
        ),
        total_amount=(
            "TX_AMOUNT",
            "sum"
        ),
        fraud_transaction_count=(
            "IS_FRAUD",
            "sum"
        )
    )
)

for row in edge_data.itertuples(
    index=False
):

    G.add_edge(
        int(row.SENDER_ACCOUNT_ID),
        int(row.RECEIVER_ACCOUNT_ID),
        transaction_count=int(
            row.transaction_count
        ),
        total_amount=float(
            row.total_amount
        ),
        fraud_transaction_count=int(
            row.fraud_transaction_count
        )
    )

print("=" * 70)
print("TRACEGRAPH NETWORK FEATURES")
print("=" * 70)

print(
    "Nodes:",
    G.number_of_nodes()
)

print(
    "Edges:",
    G.number_of_edges()
)

# ============================================================
# 3. Basic topology
# ============================================================

graph_features = pd.DataFrame({
    "ACCOUNT_ID":
        account_ids
})

graph_features[
    "in_degree"
] = graph_features[
    "ACCOUNT_ID"
].map(
    dict(G.in_degree())
).fillna(0)

graph_features[
    "out_degree"
] = graph_features[
    "ACCOUNT_ID"
].map(
    dict(G.out_degree())
).fillna(0)

# Weighted degree

graph_features[
    "weighted_in_degree"
] = graph_features[
    "ACCOUNT_ID"
].map(
    dict(
        G.in_degree(
            weight="total_amount"
        )
    )
).fillna(0)

graph_features[
    "weighted_out_degree"
] = graph_features[
    "ACCOUNT_ID"
].map(
    dict(
        G.out_degree(
            weight="total_amount"
        )
    )
).fillna(0)

# ============================================================
# 4. Centrality
# ============================================================

print(
    "\nCalculating centrality..."
)

pagerank = nx.pagerank(
    G,
    alpha=0.85
)

betweenness = nx.betweenness_centrality(
    G,
    normalized=True
)

closeness = nx.closeness_centrality(
    G
)

graph_features[
    "pagerank"
] = graph_features[
    "ACCOUNT_ID"
].map(
    pagerank
).fillna(0)

graph_features[
    "betweenness_centrality"
] = graph_features[
    "ACCOUNT_ID"
].map(
    betweenness
).fillna(0)

graph_features[
    "closeness_centrality"
] = graph_features[
    "ACCOUNT_ID"
].map(
    closeness
).fillna(0)

# ============================================================
# 5. Local neighborhood features
# ============================================================

graph_features[
    "unique_neighbors"
] = graph_features[
    "ACCOUNT_ID"
].map(
    lambda x:
        len(
            set(
                G.predecessors(x)
            )
            |
            set(
                G.successors(x)
            )
        )
)

graph_features[
    "two_hop_in_neighbors"
] = graph_features[
    "ACCOUNT_ID"
].map(
    lambda x:
        len(
            set(
                node
                for neighbor in G.predecessors(x)
                for node in G.predecessors(neighbor)
                if node != x
            )
        )
)

graph_features[
    "two_hop_out_neighbors"
] = graph_features[
    "ACCOUNT_ID"
].map(
    lambda x:
        len(
            set(
                node
                for neighbor in G.successors(x)
                for node in G.successors(neighbor)
                if node != x
            )
        )
)

# ============================================================
# 6. Fan-in / fan-out
# ============================================================

graph_features[
    "fan_in_ratio"
] = (
    graph_features[
        "in_degree"
    ]
    /
    (
        graph_features[
            "unique_neighbors"
        ] + 1e-6
    )
)

graph_features[
    "fan_out_ratio"
] = (
    graph_features[
        "out_degree"
    ]
    /
    (
        graph_features[
            "unique_neighbors"
        ] + 1e-6
    )
)

# ============================================================
# 7. Cycle participation
# ============================================================

print(
    "\nDetecting cycle participation..."
)

cycle_nodes = set()

cycles = list(
    nx.simple_cycles(G)
)

for cycle in cycles:

    for node in cycle:

        cycle_nodes.add(
            node
        )

graph_features[
    "cycle_participation"
] = graph_features[
    "ACCOUNT_ID"
].apply(
    lambda x:
        int(x in cycle_nodes)
)

graph_features[
    "cycle_count"
] = 0

for cycle in cycles:

    for node in cycle:

        graph_features.loc[
            graph_features[
                "ACCOUNT_ID"
            ] == node,
            "cycle_count"
        ] += 1

print(
    "Number of detected cycles:",
    len(cycles)
)

print(
    "Accounts participating in cycles:",
    len(cycle_nodes)
)

# ============================================================
# 8. Neighbor amount statistics
# ============================================================

amount_map = (
    tx.groupby(
        "SENDER_ACCOUNT_ID"
    )["TX_AMOUNT"]
    .sum()
    .to_dict()
)

incoming_amount_map = (
    tx.groupby(
        "RECEIVER_ACCOUNT_ID"
    )["TX_AMOUNT"]
    .sum()
    .to_dict()
)

graph_features[
    "neighbor_outgoing_amount"
] = graph_features[
    "ACCOUNT_ID"
].apply(
    lambda account:
        sum(
            amount_map.get(
                int(neighbor),
                0
            )
            for neighbor in
            G.predecessors(account)
        )
)

graph_features[
    "neighbor_incoming_amount"
] = graph_features[
    "ACCOUNT_ID"
].apply(
    lambda account:
        sum(
            incoming_amount_map.get(
                int(neighbor),
                0
            )
            for neighbor in
            G.successors(account)
        )
)

# ============================================================
# 9. Merge account features
# ============================================================

hybrid_account_features = (
    account_features
    .merge(
        graph_features,
        on="ACCOUNT_ID",
        how="left",
        suffixes=(
            "",
            "_graph"
        )
    )
)

# Remove duplicated topology columns
# where the account feature table already contains the same
# underlying quantity.

duplicate_columns = [
    "in_degree_graph",
    "out_degree_graph",
    "weighted_in_degree_graph",
    "weighted_out_degree_graph"
]

hybrid_account_features = (
    hybrid_account_features.drop(
        columns=[
            c
            for c in duplicate_columns
            if c in hybrid_account_features.columns
        ]
    )
)

# ============================================================
# 10. Merge account features into transactions
# ============================================================

sender_features = (
    hybrid_account_features
    .add_prefix("sender_account_")
)

receiver_features = (
    hybrid_account_features
    .add_prefix("receiver_account_")
)

sender_features = (
    sender_features.rename(
        columns={
            "sender_account_ACCOUNT_ID":
                "SENDER_ACCOUNT_ID"
        }
    )
)

receiver_features = (
    receiver_features.rename(
        columns={
            "receiver_account_ACCOUNT_ID":
                "RECEIVER_ACCOUNT_ID"
        }
    )
)

hybrid_transaction_features = (
    transaction_features
    .merge(
        sender_features,
        on="SENDER_ACCOUNT_ID",
        how="left"
    )
    .merge(
        receiver_features,
        on="RECEIVER_ACCOUNT_ID",
        how="left"
    )
)

# ============================================================
# 11. Remove duplicate target columns
# ============================================================

for column in [
    "sender_account_IS_FRAUD",
    "receiver_account_IS_FRAUD"
]:

    if column in hybrid_transaction_features.columns:

        hybrid_transaction_features = (
            hybrid_transaction_features.drop(
                columns=[column]
            )
        )

# ============================================================
# 12. Validation
# ============================================================

print("\nFeature tables")
print("-" * 70)

print(
    "Account feature shape:",
    hybrid_account_features.shape
)

print(
    "Transaction feature shape:",
    hybrid_transaction_features.shape
)

missing_account = int(
    hybrid_account_features.isna()
    .sum()
    .sum()
)

missing_transaction = int(
    hybrid_transaction_features.isna()
    .sum()
    .sum()
)

print(
    "Account missing values:",
    missing_account
)

print(
    "Transaction missing values:",
    missing_transaction
)

assert missing_account == 0
assert missing_transaction == 0

# ============================================================
# 13. Save
# ============================================================

ACCOUNT_OUTPUT = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_hybrid_account_features.csv"
)

TRANSACTION_OUTPUT = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_hybrid_transaction_features.csv"
)

hybrid_account_features.to_csv(
    ACCOUNT_OUTPUT,
    index=False
)

hybrid_transaction_features.to_csv(
    TRANSACTION_OUTPUT,
    index=False
)

# ============================================================
# 14. Summary
# ============================================================

print("\n" + "=" * 70)
print("STAGE 11A COMPLETE")
print("=" * 70)

print(
    "Account features:",
    hybrid_account_features.shape
)

print(
    "Transaction features:",
    hybrid_transaction_features.shape
)

print(
    "\nSaved:"
)

print(
    "-",
    ACCOUNT_OUTPUT
)

print(
    "-",
    TRANSACTION_OUTPUT
)

print(
    "\n✓ Graph topology features created."
)

print(
    "✓ Centrality features created."
)

print(
    "✓ Two-hop neighborhood features created."
)

print(
    "✓ Cycle features created."
)

print(
    "✓ Account features merged into transactions."
)

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 11A
# RAM-SAFE GRAPH TOPOLOGY FEATURES
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
import networkx as nx

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TX_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/transactions_validated.csv"
)

ACCOUNT_FEATURE_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_account_features.csv"
)

TRANSACTION_FEATURE_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_transaction_features.csv"
)

ACCOUNT_OUTPUT = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_hybrid_account_features.csv"
)

TRANSACTION_OUTPUT = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_hybrid_transaction_features.csv"
)

print("=" * 70)
print("TRACEGRAPH — RAM-SAFE GRAPH FEATURE ENGINEERING")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

print("\nLoading datasets...")

tx = pd.read_csv(TX_PATH)

account_features = pd.read_csv(
    ACCOUNT_FEATURE_PATH
)

transaction_features = pd.read_csv(
    TRANSACTION_FEATURE_PATH
)

print("Transactions:", tx.shape)
print("Account features:", account_features.shape)
print("Transaction features:", transaction_features.shape)

# ------------------------------------------------------------
# 2. PREPARE TYPES
# ------------------------------------------------------------

tx["TX_ID"] = tx["TX_ID"].astype(int)

tx["SENDER_ACCOUNT_ID"] = (
    tx["SENDER_ACCOUNT_ID"].astype(int)
)

tx["RECEIVER_ACCOUNT_ID"] = (
    tx["RECEIVER_ACCOUNT_ID"].astype(int)
)

tx["TX_AMOUNT"] = pd.to_numeric(
    tx["TX_AMOUNT"],
    errors="raise"
)

tx["TIMESTAMP"] = (
    tx["TIMESTAMP"].astype(int)
)

tx["IS_FRAUD"] = (
    tx["IS_FRAUD"].astype(bool)
)

# ------------------------------------------------------------
# 3. BUILD DIRECTED TRANSACTION GRAPH
# ------------------------------------------------------------

print("\nBuilding transaction graph...")

G = nx.DiGraph()

account_ids = sorted(
    account_features["ACCOUNT_ID"]
    .astype(int)
    .unique()
)

G.add_nodes_from(account_ids)

edge_data = (
    tx.groupby(
        [
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID"
        ],
        as_index=False
    )
    .agg(
        transaction_count=(
            "TX_ID",
            "count"
        ),
        total_amount=(
            "TX_AMOUNT",
            "sum"
        ),
        fraud_transaction_count=(
            "IS_FRAUD",
            "sum"
        )
    )
)

for row in edge_data.itertuples(index=False):

    sender = int(
        row.SENDER_ACCOUNT_ID
    )

    receiver = int(
        row.RECEIVER_ACCOUNT_ID
    )

    G.add_edge(
        sender,
        receiver,
        transaction_count=int(
            row.transaction_count
        ),
        total_amount=float(
            row.total_amount
        ),
        fraud_transaction_count=int(
            row.fraud_transaction_count
        )
    )

print(
    "Nodes:",
    G.number_of_nodes()
)

print(
    "Edges:",
    G.number_of_edges()
)

# Free intermediate dataframe
del edge_data
gc.collect()

# ============================================================
# 4. BASIC GRAPH TOPOLOGY
# ============================================================

print("\nCalculating basic topology...")

graph_features = pd.DataFrame({
    "ACCOUNT_ID": account_ids
})

in_degree_dict = dict(
    G.in_degree()
)

out_degree_dict = dict(
    G.out_degree()
)

weighted_in_dict = dict(
    G.in_degree(
        weight="total_amount"
    )
)

weighted_out_dict = dict(
    G.out_degree(
        weight="total_amount"
    )
)

graph_features["graph_in_degree"] = (
    graph_features["ACCOUNT_ID"]
    .map(in_degree_dict)
    .fillna(0)
)

graph_features["graph_out_degree"] = (
    graph_features["ACCOUNT_ID"]
    .map(out_degree_dict)
    .fillna(0)
)

graph_features["graph_weighted_in_degree"] = (
    graph_features["ACCOUNT_ID"]
    .map(weighted_in_dict)
    .fillna(0)
)

graph_features["graph_weighted_out_degree"] = (
    graph_features["ACCOUNT_ID"]
    .map(weighted_out_dict)
    .fillna(0)
)

# ============================================================
# 5. CENTRALITY FEATURES
# ============================================================

print("\nCalculating PageRank...")

pagerank = nx.pagerank(
    G,
    alpha=0.85
)

print("Calculating betweenness...")

betweenness = nx.betweenness_centrality(
    G,
    normalized=True
)

print("Calculating closeness...")

closeness = nx.closeness_centrality(
    G
)

graph_features["pagerank"] = (
    graph_features["ACCOUNT_ID"]
    .map(pagerank)
    .fillna(0)
)

graph_features["betweenness_centrality"] = (
    graph_features["ACCOUNT_ID"]
    .map(betweenness)
    .fillna(0)
)

graph_features["closeness_centrality"] = (
    graph_features["ACCOUNT_ID"]
    .map(closeness)
    .fillna(0)
)

# ============================================================
# 6. NEIGHBORHOOD FEATURES
# ============================================================

print("\nCalculating neighborhood features...")

unique_neighbors = {}
two_hop_in = {}
two_hop_out = {}

for account in account_ids:

    predecessors = set(
        G.predecessors(account)
    )

    successors = set(
        G.successors(account)
    )

    unique_neighbors[account] = len(
        predecessors | successors
    )

    # 2-hop incoming neighborhood
    incoming_two_hop = set()

    for neighbor in predecessors:

        incoming_two_hop.update(
            G.predecessors(neighbor)
        )

    incoming_two_hop.discard(account)

    two_hop_in[account] = len(
        incoming_two_hop
    )

    # 2-hop outgoing neighborhood
    outgoing_two_hop = set()

    for neighbor in successors:

        outgoing_two_hop.update(
            G.successors(neighbor)
        )

    outgoing_two_hop.discard(account)

    two_hop_out[account] = len(
        outgoing_two_hop
    )

graph_features["unique_neighbors"] = (
    graph_features["ACCOUNT_ID"]
    .map(unique_neighbors)
    .fillna(0)
)

graph_features["two_hop_in_neighbors"] = (
    graph_features["ACCOUNT_ID"]
    .map(two_hop_in)
    .fillna(0)
)

graph_features["two_hop_out_neighbors"] = (
    graph_features["ACCOUNT_ID"]
    .map(two_hop_out)
    .fillna(0)
)

# ============================================================
# 7. FAN-IN / FAN-OUT RATIOS
# ============================================================

graph_features["fan_in_ratio"] = (
    graph_features["graph_in_degree"]
    /
    (
        graph_features["unique_neighbors"]
        + 1e-6
    )
)

graph_features["fan_out_ratio"] = (
    graph_features["graph_out_degree"]
    /
    (
        graph_features["unique_neighbors"]
        + 1e-6
    )
)

# ============================================================
# 8. RAM-SAFE CYCLE FEATURES
# ============================================================
#
# IMPORTANT:
# DO NOT USE:
#
#     list(nx.simple_cycles(G))
#
# The graph is strongly connected and dense, so enumerating
# every simple cycle can consume the entire Colab RAM.
#
# Instead:
# For every node v, find the shortest path from each successor
# of v back to v.
#
# shortest_cycle_length(v)
# =
# shortest_path_length(successor -> v) + 1
#
# This gives the shortest directed cycle containing v.
# ============================================================

print("\nCalculating RAM-safe cycle features...")

# Reverse graph
G_reverse = G.reverse(copy=False)

shortest_cycle_length = {}
short_cycle_count = {}

for account in account_ids:

    # Distance from every node TO account.
    #
    # Running BFS from account on the reversed graph
    # is equivalent to finding distances TO account
    # in the original graph.
    distances_to_account = (
        nx.single_source_shortest_path_length(
            G_reverse,
            account
        )
    )

    best_cycle = np.inf
    cycle_count = 0

    # Every outgoing edge account -> successor
    # can close a cycle if successor can reach account.
    for successor in G.successors(account):

        if successor in distances_to_account:

            distance = (
                distances_to_account[
                    successor
                ]
                + 1
            )

            if distance < best_cycle:
                best_cycle = distance

            if distance <= 5:
                cycle_count += 1

    if np.isinf(best_cycle):
        shortest_cycle_length[account] = 0
    else:
        shortest_cycle_length[account] = int(
            best_cycle
        )

    short_cycle_count[account] = int(
        cycle_count
    )

graph_features[
    "shortest_cycle_length"
] = (
    graph_features["ACCOUNT_ID"]
    .map(shortest_cycle_length)
    .fillna(0)
)

graph_features[
    "short_cycle_count"
] = (
    graph_features["ACCOUNT_ID"]
    .map(short_cycle_count)
    .fillna(0)
)

# Cycle participation:
# A node is considered part of a short directed cycle
# if its shortest cycle is <= 5.

graph_features[
    "cycle_participation"
] = (
    (
        graph_features[
            "shortest_cycle_length"
        ] > 0
    )
    &
    (
        graph_features[
            "shortest_cycle_length"
        ] <= 5
    )
).astype(int)

print(
    "Accounts in short cycles:",
    int(
        graph_features[
            "cycle_participation"
        ].sum()
    )
)

# ============================================================
# 9. NEIGHBOR AMOUNT FEATURES
# ============================================================

print("\nCalculating neighbor flow features...")

sender_amounts = (
    tx.groupby(
        "SENDER_ACCOUNT_ID"
    )["TX_AMOUNT"]
    .sum()
    .to_dict()
)

receiver_amounts = (
    tx.groupby(
        "RECEIVER_ACCOUNT_ID"
    )["TX_AMOUNT"]
    .sum()
    .to_dict()
)

neighbor_outgoing_amount = {}
neighbor_incoming_amount = {}

for account in account_ids:

    # Accounts sending money INTO this account
    incoming_neighbors = G.predecessors(
        account
    )

    # Accounts receiving money FROM this account
    outgoing_neighbors = G.successors(
        account
    )

    neighbor_outgoing_amount[
        account
    ] = sum(
        sender_amounts.get(
            int(neighbor),
            0
        )
        for neighbor in incoming_neighbors
    )

    neighbor_incoming_amount[
        account
    ] = sum(
        receiver_amounts.get(
            int(neighbor),
            0
        )
        for neighbor in outgoing_neighbors
    )

graph_features[
    "neighbor_outgoing_amount"
] = (
    graph_features["ACCOUNT_ID"]
    .map(neighbor_outgoing_amount)
    .fillna(0)
)

graph_features[
    "neighbor_incoming_amount"
] = (
    graph_features["ACCOUNT_ID"]
    .map(neighbor_incoming_amount)
    .fillna(0)
)

# ============================================================
# 10. CLEAN GRAPH FEATURE TABLE
# ============================================================

graph_features = (
    graph_features
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

print("\nGraph feature table:")
print(
    graph_features.shape
)

print("\nGraph features:")

print(
    graph_features.columns.tolist()
)

# ============================================================
# 11. MERGE GRAPH FEATURES WITH ACCOUNT FEATURES
# ============================================================

print("\nMerging graph + account features...")

hybrid_account_features = (
    account_features
    .merge(
        graph_features,
        on="ACCOUNT_ID",
        how="left"
    )
)

hybrid_account_features = (
    hybrid_account_features
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

# ============================================================
# 12. ADD ACCOUNT FEATURES TO TRANSACTION FEATURES
# ============================================================

print(
    "Adding sender/receiver graph intelligence..."
)

sender_features = (
    hybrid_account_features
    .add_prefix("sender_account_")
)

receiver_features = (
    hybrid_account_features
    .add_prefix("receiver_account_")
)

sender_features = (
    sender_features.rename(
        columns={
            "sender_account_ACCOUNT_ID":
                "SENDER_ACCOUNT_ID"
        }
    )
)

receiver_features = (
    receiver_features.rename(
        columns={
            "receiver_account_ACCOUNT_ID":
                "RECEIVER_ACCOUNT_ID"
        }
    )
)

hybrid_transaction_features = (
    transaction_features
    .merge(
        sender_features,
        on="SENDER_ACCOUNT_ID",
        how="left"
    )
    .merge(
        receiver_features,
        on="RECEIVER_ACCOUNT_ID",
        how="left"
    )
)

# ============================================================
# 13. REMOVE ACCOUNT-LEVEL LABELS
# ============================================================
#
# IMPORTANT:
# We must NOT feed the known account fraud label
# directly into the model.
#
# The transaction IS_FRAUD column remains as the target.

label_columns = [
    "sender_account_IS_FRAUD",
    "receiver_account_IS_FRAUD"
]

for column in label_columns:

    if column in hybrid_transaction_features.columns:

        hybrid_transaction_features = (
            hybrid_transaction_features.drop(
                columns=[column]
            )
        )

# ============================================================
# 14. FINAL CLEANING
# ============================================================

hybrid_account_features = (
    hybrid_account_features
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

hybrid_transaction_features = (
    hybrid_transaction_features
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

# ============================================================
# 15. VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(
    "Hybrid account features:",
    hybrid_account_features.shape
)

print(
    "Hybrid transaction features:",
    hybrid_transaction_features.shape
)

account_missing = int(
    hybrid_account_features
    .isna()
    .sum()
    .sum()
)

transaction_missing = int(
    hybrid_transaction_features
    .isna()
    .sum()
    .sum()
)

print(
    "Account missing values:",
    account_missing
)

print(
    "Transaction missing values:",
    transaction_missing
)

assert account_missing == 0
assert transaction_missing == 0

# Verify transaction count
assert len(
    hybrid_transaction_features
) == len(tx)

# Verify fraud count
original_fraud = int(
    tx["IS_FRAUD"].sum()
)

hybrid_fraud = int(
    hybrid_transaction_features[
        "IS_FRAUD"
    ].sum()
)

print(
    "Original fraud transactions:",
    original_fraud
)

print(
    "Hybrid fraud transactions:",
    hybrid_fraud
)

assert original_fraud == hybrid_fraud

# ============================================================
# 16. SAVE
# ============================================================

print("\nSaving feature tables...")

hybrid_account_features.to_csv(
    ACCOUNT_OUTPUT,
    index=False
)

hybrid_transaction_features.to_csv(
    TRANSACTION_OUTPUT,
    index=False
)

# ============================================================
# 17. SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STAGE 11A COMPLETE")
print("=" * 70)

print(
    "Graph:",
    G.number_of_nodes(),
    "nodes |",
    G.number_of_edges(),
    "edges"
)

print(
    "Hybrid account features:",
    hybrid_account_features.shape
)

print(
    "Hybrid transaction features:",
    hybrid_transaction_features.shape
)

print(
    "Fraud transactions preserved:",
    hybrid_fraud
)

print(
    "\nSaved:"
)

print(
    "-",
    ACCOUNT_OUTPUT
)

print(
    "-",
    TRANSACTION_OUTPUT
)

print("\n✓ Centrality features")
print("✓ Neighborhood features")
print("✓ Fan-in / fan-out features")
print("✓ RAM-safe cycle features")
print("✓ Sender graph features")
print("✓ Receiver graph features")
print("✓ No account fraud labels used as model features")
print("✓ Original fraud count preserved")

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 11B
# HYBRID AML MODEL
#
# Transaction + Account + Graph Topology Features
# ============================================================

import os
import gc
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

HYBRID_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_hybrid_transaction_features.csv"
)

BASELINE_RESULTS_PATH = os.path.join(
    PROJECT_DIR,
    "results/baseline_model_results.csv"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "results"
)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print("=" * 70)
print("TRACEGRAPH — STAGE 11B")
print("HYBRID TRANSACTION + GRAPH MODEL")
print("=" * 70)

# ============================================================
# 2. LOAD HYBRID FEATURES
# ============================================================

print("\nLoading hybrid features...")

df = pd.read_csv(
    HYBRID_PATH
)

print(
    "Dataset shape:",
    df.shape
)

print(
    "Fraud transactions:",
    int(df["IS_FRAUD"].sum())
)

# ============================================================
# 3. DEFINE TARGET
# ============================================================

TARGET = "IS_FRAUD"

y = (
    df[TARGET]
    .astype(int)
)

# ============================================================
# 4. REMOVE IDENTIFIERS / LABEL LEAKAGE
# ============================================================
#
# These columns must NOT be predictors:
#
# TX_ID          -> transaction identifier
# ACCOUNT IDs    -> identifiers rather than learned features
# ALERT_ID       -> directly associated with fraud alerts
# IS_FRAUD       -> target
# TX_TYPE        -> categorical transaction type; currently
#                   only TRANSFER in this dataset
#
# ============================================================

EXCLUDE_COLUMNS = [
    "TX_ID",
    "SENDER_ACCOUNT_ID",
    "RECEIVER_ACCOUNT_ID",
    "ALERT_ID",
    "IS_FRAUD",
    "TX_TYPE"
]

feature_columns = [
    column
    for column in df.columns
    if column not in EXCLUDE_COLUMNS
]

X = (
    df[feature_columns]
    .copy()
)

# ------------------------------------------------------------
# Convert everything to numeric
# ------------------------------------------------------------

X = X.apply(
    pd.to_numeric,
    errors="coerce"
)

X = (
    X.replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

print(
    "\nPredictor count:",
    len(feature_columns)
)

print(
    "Predictor matrix:",
    X.shape
)

# ============================================================
# 5. VERIFY NO LABEL LEAKAGE
# ============================================================

print("\nChecking excluded columns...")

for column in EXCLUDE_COLUMNS:

    if column in feature_columns:

        raise ValueError(
            f"Leakage column found in predictors: {column}"
        )

print(
    "✓ Target excluded"
)

print(
    "✓ ALERT_ID excluded"
)

print(
    "✓ Transaction IDs excluded"
)

print(
    "✓ Account IDs excluded"
)

# ============================================================
# 6. TRAIN / TEST SPLIT
# ============================================================
#
# Same split used by the original baseline:
#
# 75% train
# 25% test
# random_state = 42
# stratified
#
# This makes the model comparison consistent.
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.25,
        stratify=y,
        random_state=42
    )
)

print("\n" + "-" * 70)
print("DATA SPLIT")
print("-" * 70)

print(
    "Train:",
    X_train.shape
)

print(
    "Test:",
    X_test.shape
)

print(
    "Train fraud:",
    int(y_train.sum())
)

print(
    "Test fraud:",
    int(y_test.sum())
)

# ============================================================
# 7. MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model_name,
    model,
    X_test,
    y_test,
    threshold=0.5
):

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        probabilities
    )

    pr_auc = average_precision_score(
        y_test,
        probabilities
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions
    ).ravel()

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    print(
        f"Threshold : {threshold:.4f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1        : {f1:.4f}"
    )

    print(
        f"ROC-AUC   : {roc_auc:.4f}"
    )

    print(
        f"PR-AUC    : {pr_auc:.4f}"
    )

    print("\nConfusion Matrix:")

    print(
        confusion_matrix(
            y_test,
            predictions
        )
    )

    return {
        "model": model_name,
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    }


# ============================================================
# 8. HYBRID LOGISTIC REGRESSION
# ============================================================

print("\n" + "=" * 70)
print("TRAINING HYBRID LOGISTIC REGRESSION")
print("=" * 70)

hybrid_logistic = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

hybrid_logistic.fit(
    X_train,
    y_train
)

logistic_result = evaluate_model(
    "TraceGraph Hybrid — Logistic Regression",
    hybrid_logistic,
    X_test,
    y_test
)

# ============================================================
# 9. HYBRID XGBOOST
# ============================================================

print("\n" + "=" * 70)
print("TRAINING HYBRID XGBOOST")
print("=" * 70)

negative_count = int(
    (y_train == 0).sum()
)

positive_count = int(
    (y_train == 1).sum()
)

scale_pos_weight = (
    negative_count /
    max(positive_count, 1)
)

print(
    "Negative training samples:",
    negative_count
)

print(
    "Positive training samples:",
    positive_count
)

print(
    "Scale pos weight:",
    scale_pos_weight
)

hybrid_xgb = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

hybrid_xgb.fit(
    X_train,
    y_train
)

xgb_result = evaluate_model(
    "TraceGraph Hybrid — XGBoost",
    hybrid_xgb,
    X_test,
    y_test
)

# ============================================================
# 10. FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("HYBRID XGBOOST FEATURE IMPORTANCE")
print("=" * 70)

importance_df = pd.DataFrame({
    "feature":
        feature_columns,
    "importance":
        hybrid_xgb.feature_importances_
})

importance_df = (
    importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    importance_df.head(30).to_string(
        index=False
    )
)

# ============================================================
# 11. GROUP FEATURES BY TYPE
# ============================================================
#
# This helps us understand whether the model relies more on:
#
# - transaction features
# - sender account features
# - receiver account features
# - graph topology
# ============================================================

def classify_feature(feature):

    if feature.startswith(
        "sender_account_"
    ):
        return "sender_account"

    if feature.startswith(
        "receiver_account_"
    ):
        return "receiver_account"

    if feature in [
        "TX_AMOUNT",
        "TIMESTAMP",
        "sender_total_transactions",
        "sender_total_amount",
        "sender_out_degree",
        "sender_fan_out",
        "sender_max_velocity",
        "sender_temporal_concentration",
        "receiver_total_transactions",
        "receiver_total_amount",
        "receiver_in_degree",
        "receiver_fan_in",
        "receiver_max_velocity",
        "receiver_temporal_concentration",
        "sender_receiver_amount_ratio",
        "amount_vs_sender_average",
        "amount_vs_receiver_average"
    ]:
        return "transaction"

    return "other"


importance_df[
    "feature_group"
] = importance_df[
    "feature"
].apply(
    classify_feature
)

group_importance = (
    importance_df
    .groupby(
        "feature_group",
        as_index=False
    )["importance"]
    .sum()
    .sort_values(
        "importance",
        ascending=False
    )
)

print("\n" + "-" * 70)
print("FEATURE IMPORTANCE BY GROUP")
print("-" * 70)

print(
    group_importance.to_string(
        index=False
    )
)

# ============================================================
# 12. SAVE FEATURE IMPORTANCE
# ============================================================

IMPORTANCE_PATH = os.path.join(
    RESULT_DIR,
    "hybrid_xgboost_feature_importance.csv"
)

importance_df.to_csv(
    IMPORTANCE_PATH,
    index=False
)

GROUP_IMPORTANCE_PATH = os.path.join(
    RESULT_DIR,
    "hybrid_feature_group_importance.csv"
)

group_importance.to_csv(
    GROUP_IMPORTANCE_PATH,
    index=False
)

# ============================================================
# 13. SAVE MODELS
# ============================================================

LOGISTIC_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "tracegraph_hybrid_logistic.pkl"
)

XGB_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "tracegraph_hybrid_xgboost.pkl"
)

joblib.dump(
    hybrid_logistic,
    LOGISTIC_MODEL_PATH
)

joblib.dump(
    hybrid_xgb,
    XGB_MODEL_PATH
)

# ============================================================
# 14. COMPARISON WITH EXISTING BASELINES
# ============================================================

current_results = pd.DataFrame([
    logistic_result,
    xgb_result
])

if os.path.exists(
    BASELINE_RESULTS_PATH
):

    baseline_results = pd.read_csv(
        BASELINE_RESULTS_PATH
    )

    comparison = pd.concat(
        [
            baseline_results,
            current_results
        ],
        ignore_index=True
    )

else:

    comparison = current_results.copy()

COMPARISON_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_model_comparison.csv"
)

comparison.to_csv(
    COMPARISON_PATH,
    index=False
)

# ============================================================
# 15. PRINT FINAL COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("TRACEGRAPH MODEL COMPARISON")
print("=" * 70)

display_columns = [
    "model",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc"
]

available_columns = [
    column
    for column in display_columns
    if column in comparison.columns
]

print(
    comparison[
        available_columns
    ].to_string(
        index=False
    )
)

# ============================================================
# 16. SAVE TEST PREDICTIONS
# ============================================================

test_predictions = X_test.copy()

test_predictions[
    "TRUE_LABEL"
] = y_test.values

test_predictions[
    "HYBRID_LOGISTIC_PROBABILITY"
] = (
    hybrid_logistic
    .predict_proba(X_test)[:, 1]
)

test_predictions[
    "HYBRID_XGBOOST_PROBABILITY"
] = (
    hybrid_xgb
    .predict_proba(X_test)[:, 1]
)

PREDICTIONS_PATH = os.path.join(
    RESULT_DIR,
    "hybrid_test_predictions.csv"
)

test_predictions.to_csv(
    PREDICTIONS_PATH,
    index=False
)

# ============================================================
# 17. MEMORY CLEANUP
# ============================================================

del X_train
del X_test
del y_train
del y_test

gc.collect()

# ============================================================
# 18. COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("STAGE 11B COMPLETE")
print("=" * 70)

print("\nSaved:")

print(
    "-",
    LOGISTIC_MODEL_PATH
)

print(
    "-",
    XGB_MODEL_PATH
)

print(
    "-",
    IMPORTANCE_PATH
)

print(
    "-",
    GROUP_IMPORTANCE_PATH
)

print(
    "-",
    COMPARISON_PATH
)

print(
    "-",
    PREDICTIONS_PATH
)

print("\n✓ Hybrid Logistic Regression trained")
print("✓ Hybrid XGBoost trained")
print("✓ Feature importance calculated")
print("✓ Feature groups analyzed")
print("✓ Existing baseline comparison generated")
print("✓ Test predictions saved")

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 12
# TEMPORAL AML INVESTIGATION + MONEY FLOW TRACING
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
import networkx as nx
import joblib

from collections import defaultdict, deque

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TX_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/transactions_validated.csv"
)

HYBRID_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/tracegraph_hybrid_transaction_features.csv"
)

MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models/tracegraph_hybrid_xgboost.pkl"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "results"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

print("=" * 70)
print("TRACEGRAPH — STAGE 12")
print("TEMPORAL AML INVESTIGATION ENGINE")
print("=" * 70)

# ============================================================
# 2. LOAD DATA + MODEL
# ============================================================

print("\nLoading transaction data...")

tx = pd.read_csv(
    TX_PATH
)

hybrid_features = pd.read_csv(
    HYBRID_PATH
)

model = joblib.load(
    MODEL_PATH
)

print(
    "Transactions:",
    tx.shape
)

print(
    "Hybrid features:",
    hybrid_features.shape
)

# ============================================================
# 3. PREPARE TRANSACTION DATA
# ============================================================

tx["TX_ID"] = (
    tx["TX_ID"].astype(int)
)

tx["SENDER_ACCOUNT_ID"] = (
    tx["SENDER_ACCOUNT_ID"].astype(int)
)

tx["RECEIVER_ACCOUNT_ID"] = (
    tx["RECEIVER_ACCOUNT_ID"].astype(int)
)

tx["TIMESTAMP"] = (
    tx["TIMESTAMP"].astype(int)
)

tx["TX_AMOUNT"] = (
    pd.to_numeric(
        tx["TX_AMOUNT"]
    )
)

# ============================================================
# 4. GENERATE RISK SCORES
# ============================================================
#
# IMPORTANT:
# This is an investigation/demo score using the exploratory
# hybrid model from Stage 11B.
#
# It is NOT the final leakage-free temporal evaluation.
# ============================================================

EXCLUDE_COLUMNS = [
    "TX_ID",
    "SENDER_ACCOUNT_ID",
    "RECEIVER_ACCOUNT_ID",
    "ALERT_ID",
    "IS_FRAUD",
    "TX_TYPE"
]

feature_columns = [
    c
    for c in hybrid_features.columns
    if c not in EXCLUDE_COLUMNS
]

X_all = (
    hybrid_features[
        feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

risk_scores = (
    model
    .predict_proba(X_all)[:, 1]
)

tx["RISK_SCORE"] = risk_scores

# ============================================================
# 5. RANK SUSPICIOUS TRANSACTIONS
# ============================================================

tx_ranked = (
    tx
    .sort_values(
        "RISK_SCORE",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("TOP SUSPICIOUS TRANSACTIONS")
print("=" * 70)

print(
    tx_ranked[
        [
            "TX_ID",
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TX_AMOUNT",
            "TIMESTAMP",
            "RISK_SCORE"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# ============================================================
# 6. BUILD TEMPORAL TRANSACTION INDEX
# ============================================================
#
# incoming_index[account]
#     = transactions arriving at account
#
# outgoing_index[account]
#     = transactions leaving account
#
# Each list is sorted by timestamp.
# ============================================================

print("\nBuilding temporal transaction index...")

incoming_index = defaultdict(list)
outgoing_index = defaultdict(list)

for row in tx.itertuples(index=False):

    record = {
        "tx_id": int(row.TX_ID),
        "sender": int(
            row.SENDER_ACCOUNT_ID
        ),
        "receiver": int(
            row.RECEIVER_ACCOUNT_ID
        ),
        "amount": float(
            row.TX_AMOUNT
        ),
        "timestamp": int(
            row.TIMESTAMP
        ),
        "risk": float(
            row.RISK_SCORE
        ),
        "is_fraud": bool(
            row.IS_FRAUD
        ),
        "alert_id": int(
            row.ALERT_ID
        )
    }

    outgoing_index[
        record["sender"]
    ].append(record)

    incoming_index[
        record["receiver"]
    ].append(record)

for account in incoming_index:

    incoming_index[
        account
    ].sort(
        key=lambda x: (
            x["timestamp"],
            x["tx_id"]
        )
    )

for account in outgoing_index:

    outgoing_index[
        account
    ].sort(
        key=lambda x: (
            x["timestamp"],
            x["tx_id"]
        )
    )

print(
    "Indexed accounts:",
    len(
        set(
            incoming_index.keys()
        )
        |
        set(
            outgoing_index.keys()
        )
    )
)

# ============================================================
# 7. REVERSE TEMPORAL TRACE
# ============================================================
#
# Starting from the RECEIVER of suspicious transaction:
#
# Find earlier transactions that funded it.
#
# Constraint:
#
# previous_timestamp <= current_timestamp
#
# This prevents tracing through future transactions.
# ============================================================

def reverse_trace(
    start_account,
    start_timestamp,
    max_depth=4,
    max_nodes=50
):

    visited = set(
        [start_account]
    )

    queue = deque()

    queue.append(
        (
            start_account,
            start_timestamp,
            0
        )
    )

    discovered_transactions = []

    while queue:

        current_account, current_time, depth = (
            queue.popleft()
        )

        if depth >= max_depth:
            continue

        candidates = incoming_index.get(
            current_account,
            []
        )

        # Latest earlier transactions first
        candidates = reversed(
            candidates
        )

        for record in candidates:

            # Never travel into the future
            if record["timestamp"] > current_time:
                continue

            # Ignore self-loop for tracing
            if (
                record["sender"]
                ==
                record["receiver"]
            ):
                continue

            discovered_transactions.append(
                record
            )

            sender = record["sender"]

            if (
                sender not in visited
                and
                len(visited) < max_nodes
            ):

                visited.add(
                    sender
                )

                queue.append(
                    (
                        sender,
                        record["timestamp"],
                        depth + 1
                    )
                )

    return {
        "accounts": sorted(
            visited
        ),
        "transactions": discovered_transactions
    }


# ============================================================
# 8. FORWARD TEMPORAL TRACE
# ============================================================
#
# Starting from the SENDER of suspicious transaction:
#
# Follow where the money subsequently moves.
#
# Constraint:
#
# next_timestamp >= current_timestamp
# ============================================================

def forward_trace(
    start_account,
    start_timestamp,
    max_depth=4,
    max_nodes=50
):

    visited = set(
        [start_account]
    )

    queue = deque()

    queue.append(
        (
            start_account,
            start_timestamp,
            0
        )
    )

    discovered_transactions = []

    while queue:

        current_account, current_time, depth = (
            queue.popleft()
        )

        if depth >= max_depth:
            continue

        candidates = outgoing_index.get(
            current_account,
            []
        )

        for record in candidates:

            if record["timestamp"] < current_time:
                continue

            if (
                record["sender"]
                ==
                record["receiver"]
            ):
                continue

            discovered_transactions.append(
                record
            )

            receiver = record[
                "receiver"
            ]

            if (
                receiver not in visited
                and
                len(visited) < max_nodes
            ):

                visited.add(
                    receiver
                )

                queue.append(
                    (
                        receiver,
                        record["timestamp"],
                        depth + 1
                    )
                )

    return {
        "accounts": sorted(
            visited
        ),
        "transactions": discovered_transactions
    }


# ============================================================
# 9. TEMPORAL FAN-IN DETECTION
# ============================================================

def detect_fan_in(
    account,
    timestamp,
    window=10
):

    lower = timestamp - window

    records = [
        record
        for record in incoming_index.get(
            account,
            []
        )
        if lower <= record["timestamp"] <= timestamp
        and record["sender"] != account
    ]

    unique_senders = sorted(
        set(
            record["sender"]
            for record in records
        )
    )

    total_amount = sum(
        record["amount"]
        for record in records
    )

    return {
        "transaction_count": len(records),
        "unique_senders": len(
            unique_senders
        ),
        "total_amount": total_amount,
        "senders": unique_senders
    }


# ============================================================
# 10. TEMPORAL FAN-OUT DETECTION
# ============================================================

def detect_fan_out(
    account,
    timestamp,
    window=10
):

    upper = timestamp + window

    records = [
        record
        for record in outgoing_index.get(
            account,
            []
        )
        if timestamp <= record["timestamp"] <= upper
        and record["receiver"] != account
    ]

    unique_receivers = sorted(
        set(
            record["receiver"]
            for record in records
        )
    )

    total_amount = sum(
        record["amount"]
        for record in records
    )

    return {
        "transaction_count": len(records),
        "unique_receivers": len(
            unique_receivers
        ),
        "total_amount": total_amount,
        "receivers": unique_receivers
    }


# ============================================================
# 11. RAPID-HOP DETECTION
# ============================================================
#
# Detect:
#
# A -> B
#     B -> C
#
# within a small number of timestamp steps.
# ============================================================

def detect_rapid_hops(
    account,
    timestamp,
    direction="forward",
    max_time_gap=5
):

    hops = []

    if direction == "forward":

        first_transactions = (
            outgoing_index.get(
                account,
                []
            )
        )

        for first in first_transactions:

            if first["timestamp"] < timestamp:
                continue

            middle = first[
                "receiver"
            ]

            for second in outgoing_index.get(
                middle,
                []
            ):

                if second["timestamp"] < first["timestamp"]:
                    continue

                gap = (
                    second["timestamp"]
                    -
                    first["timestamp"]
                )

                if gap <= max_time_gap:

                    hops.append(
                        {
                            "from":
                                account,
                            "middle":
                                middle,
                            "to":
                                second["receiver"],
                            "first_tx":
                                first["tx_id"],
                            "second_tx":
                                second["tx_id"],
                            "time_gap":
                                gap
                        }
                    )

    return hops


# ============================================================
# 12. SHORT TEMPORAL CYCLE DETECTION
# ============================================================

def detect_temporal_cycles(
    start_account,
    start_timestamp,
    max_depth=5,
    max_time_gap=20
):

    cycles = []

    path = [
        start_account
    ]

    def dfs(
        current,
        current_time,
        depth
    ):

        if depth >= max_depth:
            return

        for record in outgoing_index.get(
            current,
            []
        ):

            next_node = record[
                "receiver"
            ]

            next_time = record[
                "timestamp"
            ]

            if next_time < current_time:
                continue

            if (
                next_time
                -
                start_timestamp
                >
                max_time_gap
            ):
                continue

            # Found return to starting account
            if next_node == start_account:

                if len(path) >= 2:

                    cycles.append(
                        {
                            "path":
                                path.copy()
                                + [
                                    start_account
                                ],
                            "length":
                                len(path),
                            "start_timestamp":
                                start_timestamp,
                            "end_timestamp":
                                next_time
                        }
                    )

                continue

            # Prevent repeated nodes
            if next_node in path:
                continue

            path.append(
                next_node
            )

            dfs(
                next_node,
                next_time,
                depth + 1
            )

            path.pop()

    dfs(
        start_account,
        start_timestamp,
        0
    )

    return cycles


# ============================================================
# 13. INVESTIGATION REPORT GENERATOR
# ============================================================

def investigate_transaction(
    tx_row
):

    tx_id = int(
        tx_row.TX_ID
    )

    sender = int(
        tx_row.SENDER_ACCOUNT_ID
    )

    receiver = int(
        tx_row.RECEIVER_ACCOUNT_ID
    )

    amount = float(
        tx_row.TX_AMOUNT
    )

    timestamp = int(
        tx_row.TIMESTAMP
    )

    risk = float(
        tx_row.RISK_SCORE
    )

    # -----------------------------------------
    # Reverse
    # -----------------------------------------

    reverse = reverse_trace(
        receiver,
        timestamp,
        max_depth=4,
        max_nodes=50
    )

    # -----------------------------------------
    # Forward
    # -----------------------------------------

    forward = forward_trace(
        sender,
        timestamp,
        max_depth=4,
        max_nodes=50
    )

    # -----------------------------------------
    # Fan-in
    # -----------------------------------------

    fan_in = detect_fan_in(
        receiver,
        timestamp,
        window=10
    )

    # -----------------------------------------
    # Fan-out
    # -----------------------------------------

    fan_out = detect_fan_out(
        sender,
        timestamp,
        window=10
    )

    # -----------------------------------------
    # Rapid hops
    # -----------------------------------------

    rapid_hops = detect_rapid_hops(
        sender,
        timestamp,
        direction="forward",
        max_time_gap=5
    )

    # -----------------------------------------
    # Cycles
    # -----------------------------------------

    cycles = detect_temporal_cycles(
        receiver,
        timestamp,
        max_depth=5,
        max_time_gap=20
    )

    # -----------------------------------------
    # Pattern labels
    # -----------------------------------------

    patterns = []

    if fan_in[
        "unique_senders"
    ] >= 3:

        patterns.append(
            "fan-in"
        )

    if fan_out[
        "unique_receivers"
    ] >= 3:

        patterns.append(
            "fan-out"
        )

    if len(rapid_hops) > 0:

        patterns.append(
            "rapid-hop"
        )

    if len(cycles) > 0:

        patterns.append(
            "temporal-cycle"
        )

    if len(reverse["accounts"]) > 1:

        patterns.append(
            "multi-account-funding"
        )

    if len(forward["accounts"]) > 1:

        patterns.append(
            "multi-account-flow"
        )

    if not patterns:

        patterns.append(
            "no-strong-network-pattern"
        )

    return {
        "TX_ID": tx_id,
        "sender": sender,
        "receiver": receiver,
        "amount": amount,
        "timestamp": timestamp,
        "risk_score": risk,

        "reverse_accounts":
            reverse["accounts"],

        "reverse_transaction_count":
            len(
                reverse[
                    "transactions"
                ]
            ),

        "forward_accounts":
            forward["accounts"],

        "forward_transaction_count":
            len(
                forward[
                    "transactions"
                ]
            ),

        "fan_in_unique_senders":
            fan_in[
                "unique_senders"
            ],

        "fan_in_transaction_count":
            fan_in[
                "transaction_count"
            ],

        "fan_in_total_amount":
            fan_in[
                "total_amount"
            ],

        "fan_out_unique_receivers":
            fan_out[
                "unique_receivers"
            ],

        "fan_out_transaction_count":
            fan_out[
                "transaction_count"
            ],

        "fan_out_total_amount":
            fan_out[
                "total_amount"
            ],

        "rapid_hop_count":
            len(
                rapid_hops
            ),

        "cycle_count":
            len(
                cycles
            ),

        "patterns":
            " | ".join(
                patterns
            )
    }


# ============================================================
# 14. INVESTIGATE TOP SUSPICIOUS TRANSACTIONS
# ============================================================

TOP_N = 25

suspicious_transactions = (
    tx_ranked
    .head(TOP_N)
)

print("\n" + "=" * 70)
print(
    f"INVESTIGATING TOP {TOP_N} TRANSACTIONS"
)
print("=" * 70)

investigation_reports = []

for row in suspicious_transactions.itertuples(
    index=False
):

    report = investigate_transaction(
        row
    )

    investigation_reports.append(
        report
    )

investigation_df = pd.DataFrame(
    investigation_reports
)

# ============================================================
# 15. PRINT INVESTIGATION SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("INVESTIGATION SUMMARY")
print("=" * 70)

print(
    investigation_df[
        [
            "TX_ID",
            "sender",
            "receiver",
            "amount",
            "timestamp",
            "risk_score",
            "reverse_transaction_count",
            "forward_transaction_count",
            "fan_in_unique_senders",
            "fan_out_unique_receivers",
            "rapid_hop_count",
            "cycle_count",
            "patterns"
        ]
    ]
    .to_string(index=False)
)

# ============================================================
# 16. SAVE RISK-SCORED TRANSACTIONS
# ============================================================

RISK_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_transaction_risk_scores.csv"
)

tx_ranked.to_csv(
    RISK_PATH,
    index=False
)

# ============================================================
# 17. SAVE INVESTIGATION REPORT
# ============================================================

INVESTIGATION_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_investigation_reports.csv"
)

investigation_df.to_csv(
    INVESTIGATION_PATH,
    index=False
)

# ============================================================
# 18. SAVE DETAILED JSON REPORTS
# ============================================================

DETAILED_REPORT_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_detailed_investigations.json"
)

detailed_reports = []

for row in suspicious_transactions.itertuples(
    index=False
):

    report = investigate_transaction(
        row
    )

    # Add actual transaction details
    report[
        "original_transaction"
    ] = {
        "TX_ID":
            int(row.TX_ID),
        "sender":
            int(row.SENDER_ACCOUNT_ID),
        "receiver":
            int(row.RECEIVER_ACCOUNT_ID),
        "amount":
            float(row.TX_AMOUNT),
        "timestamp":
            int(row.TIMESTAMP),
        "risk_score":
            float(row.RISK_SCORE)
    }

    # Include detailed traces
    reverse = reverse_trace(
        int(row.RECEIVER_ACCOUNT_ID),
        int(row.TIMESTAMP),
        max_depth=4,
        max_nodes=50
    )

    forward = forward_trace(
        int(row.SENDER_ACCOUNT_ID),
        int(row.TIMESTAMP),
        max_depth=4,
        max_nodes=50
    )

    report[
        "reverse_transactions"
    ] = reverse[
        "transactions"
    ]

    report[
        "forward_transactions"
    ] = forward[
        "transactions"
    ]

    report[
        "temporal_cycles"
    ] = detect_temporal_cycles(
        int(row.RECEIVER_ACCOUNT_ID),
        int(row.TIMESTAMP),
        max_depth=5,
        max_time_gap=20
    )

    detailed_reports.append(
        report
    )

with open(
    DETAILED_REPORT_PATH,
    "w"
) as f:

    json.dump(
        detailed_reports,
        f,
        indent=2
    )

# ============================================================
# 19. PATTERN STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("DETECTED NETWORK PATTERNS")
print("=" * 70)

pattern_counts = defaultdict(int)

for patterns in investigation_df[
    "patterns"
]:

    for pattern in patterns.split(
        " | "
    ):

        pattern_counts[
            pattern
        ] += 1

pattern_df = (
    pd.DataFrame(
        list(
            pattern_counts.items()
        ),
        columns=[
            "pattern",
            "transaction_count"
        ]
    )
    .sort_values(
        "transaction_count",
        ascending=False
    )
)

print(
    pattern_df.to_string(
        index=False
    )
)

PATTERN_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_detected_patterns.csv"
)

pattern_df.to_csv(
    PATTERN_PATH,
    index=False
)

# ============================================================
# 20. EXAMPLE INVESTIGATION
# ============================================================

print("\n" + "=" * 70)
print("HIGHEST-RISK INVESTIGATION")
print("=" * 70)

if len(investigation_df) > 0:

    top = investigation_df.iloc[0]

    print(
        f"Transaction: {int(top['TX_ID'])}"
    )

    print(
        f"Flow: {int(top['sender'])} -> "
        f"{int(top['receiver'])}"
    )

    print(
        f"Amount: {top['amount']:.2f}"
    )

    print(
        f"Timestamp: {int(top['timestamp'])}"
    )

    print(
        f"Risk score: {top['risk_score']:.6f}"
    )

    print(
        "\nDetected patterns:"
    )

    print(
        top["patterns"]
    )

    print(
        "\nReverse trace accounts:"
    )

    print(
        top["reverse_accounts"]
    )

    print(
        "\nForward trace accounts:"
    )

    print(
        top["forward_accounts"]
    )

# ============================================================
# 21. COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("STAGE 12 COMPLETE")
print("=" * 70)

print(
    "\nSaved:"
)

print(
    "-",
    RISK_PATH
)

print(
    "-",
    INVESTIGATION_PATH
)

print(
    "-",
    DETAILED_REPORT_PATH
)

print(
    "-",
    PATTERN_PATH
)

print("\n✓ Risk scores generated")
print("✓ Temporal reverse tracing implemented")
print("✓ Temporal forward tracing implemented")
print("✓ Fan-in detection implemented")
print("✓ Fan-out detection implemented")
print("✓ Rapid-hop detection implemented")
print("✓ Temporal cycle detection implemented")
print("✓ Investigation reports generated")

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 12B
# BOUNDED TEMPORAL INVESTIGATION ENGINE
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from collections import defaultdict, deque

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TX_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/transactions_validated.csv"
)

RISK_PATH = os.path.join(
    PROJECT_DIR,
    "results/tracegraph_transaction_risk_scores.csv"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "results"
)

# ============================================================
# CONFIGURATION
# ============================================================

TRACE_TIME_WINDOW = 10
MAX_TRACE_DEPTH = 3
MAX_TRACE_ACCOUNTS = 20
MAX_TRACE_TRANSACTIONS = 50

MAX_RAPID_HOPS = 20
MAX_CYCLES = 10
MAX_CYCLE_LENGTH = 5
MAX_CYCLE_TIME = 20

TOP_N = 25

print("=" * 70)
print("TRACEGRAPH — STAGE 12B")
print("BOUNDED TEMPORAL INVESTIGATION")
print("=" * 70)

# ============================================================
# 1. LOAD
# ============================================================

tx = pd.read_csv(
    TX_PATH
)

tx_ranked = pd.read_csv(
    RISK_PATH
)

tx["TX_ID"] = tx["TX_ID"].astype(int)
tx["SENDER_ACCOUNT_ID"] = (
    tx["SENDER_ACCOUNT_ID"].astype(int)
)
tx["RECEIVER_ACCOUNT_ID"] = (
    tx["RECEIVER_ACCOUNT_ID"].astype(int)
)
tx["TIMESTAMP"] = tx["TIMESTAMP"].astype(int)
tx["TX_AMOUNT"] = pd.to_numeric(
    tx["TX_AMOUNT"]
)

# ============================================================
# 2. TEMPORAL INDEX
# ============================================================

incoming_index = defaultdict(list)
outgoing_index = defaultdict(list)

for row in tx.itertuples(index=False):

    record = {
        "tx_id": int(row.TX_ID),
        "sender": int(row.SENDER_ACCOUNT_ID),
        "receiver": int(row.RECEIVER_ACCOUNT_ID),
        "amount": float(row.TX_AMOUNT),
        "timestamp": int(row.TIMESTAMP),
        "is_fraud": bool(row.IS_FRAUD),
        "alert_id": int(row.ALERT_ID)
    }

    incoming_index[
        record["receiver"]
    ].append(record)

    outgoing_index[
        record["sender"]
    ].append(record)

for account in incoming_index:

    incoming_index[account].sort(
        key=lambda x: (
            x["timestamp"],
            x["tx_id"]
        )
    )

for account in outgoing_index:

    outgoing_index[account].sort(
        key=lambda x: (
            x["timestamp"],
            x["tx_id"]
        )
    )

# ============================================================
# 3. BOUNDED REVERSE TRACE
# ============================================================

def bounded_reverse_trace(
    start_account,
    start_timestamp
):

    visited_accounts = {
        start_account
    }

    visited_transactions = set()

    queue = deque([
        (
            start_account,
            start_timestamp,
            0
        )
    ])

    trace_transactions = []

    while queue:

        account, current_time, depth = (
            queue.popleft()
        )

        if depth >= MAX_TRACE_DEPTH:
            continue

        lower_time = max(
            0,
            current_time - TRACE_TIME_WINDOW
        )

        candidates = incoming_index.get(
            account,
            []
        )

        for record in reversed(candidates):

            if record["timestamp"] < lower_time:
                continue

            if record["timestamp"] > current_time:
                continue

            if (
                record["sender"]
                ==
                record["receiver"]
            ):
                continue

            if record["tx_id"] in visited_transactions:
                continue

            visited_transactions.add(
                record["tx_id"]
            )

            trace_transactions.append(
                record
            )

            if (
                len(trace_transactions)
                >= MAX_TRACE_TRANSACTIONS
            ):
                break

            sender = record["sender"]

            if (
                sender not in visited_accounts
                and
                len(visited_accounts)
                < MAX_TRACE_ACCOUNTS
            ):

                visited_accounts.add(
                    sender
                )

                queue.append(
                    (
                        sender,
                        record["timestamp"],
                        depth + 1
                    )
                )

        if (
            len(trace_transactions)
            >= MAX_TRACE_TRANSACTIONS
        ):
            break

    return {
        "accounts": sorted(
            visited_accounts
        ),
        "transactions": trace_transactions
    }


# ============================================================
# 4. BOUNDED FORWARD TRACE
# ============================================================

def bounded_forward_trace(
    start_account,
    start_timestamp
):

    visited_accounts = {
        start_account
    }

    visited_transactions = set()

    queue = deque([
        (
            start_account,
            start_timestamp,
            0
        )
    ])

    trace_transactions = []

    while queue:

        account, current_time, depth = (
            queue.popleft()
        )

        if depth >= MAX_TRACE_DEPTH:
            continue

        upper_time = (
            current_time
            +
            TRACE_TIME_WINDOW
        )

        candidates = outgoing_index.get(
            account,
            []
        )

        for record in candidates:

            if record["timestamp"] < current_time:
                continue

            if record["timestamp"] > upper_time:
                continue

            if (
                record["sender"]
                ==
                record["receiver"]
            ):
                continue

            if record["tx_id"] in visited_transactions:
                continue

            visited_transactions.add(
                record["tx_id"]
            )

            trace_transactions.append(
                record
            )

            if (
                len(trace_transactions)
                >= MAX_TRACE_TRANSACTIONS
            ):
                break

            receiver = record["receiver"]

            if (
                receiver not in visited_accounts
                and
                len(visited_accounts)
                < MAX_TRACE_ACCOUNTS
            ):

                visited_accounts.add(
                    receiver
                )

                queue.append(
                    (
                        receiver,
                        record["timestamp"],
                        depth + 1
                    )
                )

        if (
            len(trace_transactions)
            >= MAX_TRACE_TRANSACTIONS
        ):
            break

    return {
        "accounts": sorted(
            visited_accounts
        ),
        "transactions": trace_transactions
    }


# ============================================================
# 5. TARGETED FAN-IN
# ============================================================

def targeted_fan_in(
    account,
    timestamp
):

    lower = max(
        0,
        timestamp - TRACE_TIME_WINDOW
    )

    records = [
        r
        for r in incoming_index.get(
            account,
            []
        )
        if lower <= r["timestamp"] <= timestamp
        and r["sender"] != account
    ]

    senders = sorted(
        set(
            r["sender"]
            for r in records
        )
    )

    return {
        "transaction_count": len(records),
        "unique_senders": len(senders),
        "senders": senders,
        "total_amount": sum(
            r["amount"]
            for r in records
        )
    }


# ============================================================
# 6. TARGETED FAN-OUT
# ============================================================

def targeted_fan_out(
    account,
    timestamp
):

    upper = (
        timestamp
        +
        TRACE_TIME_WINDOW
    )

    records = [
        r
        for r in outgoing_index.get(
            account,
            []
        )
        if timestamp <= r["timestamp"] <= upper
        and r["receiver"] != account
    ]

    receivers = sorted(
        set(
            r["receiver"]
            for r in records
        )
    )

    return {
        "transaction_count": len(records),
        "unique_receivers": len(receivers),
        "receivers": receivers,
        "total_amount": sum(
            r["amount"]
            for r in records
        )
    }


# ============================================================
# 7. TARGETED RAPID-HOP DETECTION
# ============================================================

def targeted_rapid_hops(
    sender,
    receiver,
    timestamp
):

    hops = []

    # Start specifically from the suspicious
    # transaction's receiver.
    #
    # Example:
    #
    # A -> B  [suspicious transaction]
    # B -> C  [rapid next hop]

    first_hops = outgoing_index.get(
        receiver,
        []
    )

    for second in first_hops:

        if second["timestamp"] < timestamp:
            continue

        if (
            second["timestamp"]
            >
            timestamp + TRACE_TIME_WINDOW
        ):
            continue

        if second["receiver"] == receiver:
            continue

        hops.append({
            "from_account":
                receiver,
            "to_account":
                second["receiver"],
            "tx_id":
                second["tx_id"],
            "timestamp":
                second["timestamp"],
            "time_gap":
                second["timestamp"]
                -
                timestamp,
            "amount":
                second["amount"]
        })

        if len(hops) >= MAX_RAPID_HOPS:
            break

    return hops


# ============================================================
# 8. TARGETED TEMPORAL CYCLE
# ============================================================

def targeted_cycles(
    start_account,
    start_timestamp
):

    cycles = []

    path = [
        start_account
    ]

    visited = {
        start_account
    }

    def dfs(
        current,
        current_time
    ):

        if len(cycles) >= MAX_CYCLES:
            return

        if (
            len(path)
            >
            MAX_CYCLE_LENGTH
        ):
            return

        for record in outgoing_index.get(
            current,
            []
        ):

            next_node = record["receiver"]
            next_time = record["timestamp"]

            if next_time < current_time:
                continue

            if (
                next_time
                >
                start_timestamp
                +
                MAX_CYCLE_TIME
            ):
                continue

            # Return to start
            if next_node == start_account:

                if len(path) >= 2:

                    cycles.append({
                        "path":
                            path.copy()
                            + [
                                start_account
                            ],
                        "length":
                            len(path),
                        "start_timestamp":
                            start_timestamp,
                        "end_timestamp":
                            next_time
                    })

                continue

            if next_node in visited:
                continue

            visited.add(
                next_node
            )

            path.append(
                next_node
            )

            dfs(
                next_node,
                next_time
            )

            path.pop()

            visited.remove(
                next_node
            )

            if len(cycles) >= MAX_CYCLES:
                return

    dfs(
        start_account,
        start_timestamp
    )

    return cycles


# ============================================================
# 9. INVESTIGATE ONE TRANSACTION
# ============================================================

def investigate(row):

    tx_id = int(row.TX_ID)

    sender = int(
        row.SENDER_ACCOUNT_ID
    )

    receiver = int(
        row.RECEIVER_ACCOUNT_ID
    )

    timestamp = int(
        row.TIMESTAMP
    )

    amount = float(
        row.TX_AMOUNT
    )

    risk = float(
        row.RISK_SCORE
    )

    reverse = bounded_reverse_trace(
        receiver,
        timestamp
    )

    forward = bounded_forward_trace(
        sender,
        timestamp
    )

    fan_in = targeted_fan_in(
        receiver,
        timestamp
    )

    fan_out = targeted_fan_out(
        sender,
        timestamp
    )

    rapid_hops = targeted_rapid_hops(
        sender,
        receiver,
        timestamp
    )

    cycles = targeted_cycles(
        receiver,
        timestamp
    )

    patterns = []

    if fan_in[
        "unique_senders"
    ] >= 3:

        patterns.append(
            "fan-in"
        )

    if fan_out[
        "unique_receivers"
    ] >= 3:

        patterns.append(
            "fan-out"
        )

    if len(rapid_hops) > 0:

        patterns.append(
            "rapid-hop"
        )

    if len(cycles) > 0:

        patterns.append(
            "temporal-cycle"
        )

    if (
        len(
            reverse["accounts"]
        ) > 1
    ):

        patterns.append(
            "reverse-funding-chain"
        )

    if (
        len(
            forward["accounts"]
        ) > 1
    ):

        patterns.append(
            "forward-money-flow"
        )

    if not patterns:

        patterns.append(
            "no-strong-network-pattern"
        )

    return {
        "TX_ID": tx_id,
        "sender": sender,
        "receiver": receiver,
        "amount": amount,
        "timestamp": timestamp,
        "risk_score": risk,

        "reverse_accounts":
            reverse["accounts"],

        "reverse_transaction_count":
            len(
                reverse[
                    "transactions"
                ]
            ),

        "reverse_transactions":
            reverse["transactions"],

        "forward_accounts":
            forward["accounts"],

        "forward_transaction_count":
            len(
                forward[
                    "transactions"
                ]
            ),

        "forward_transactions":
            forward["transactions"],

        "fan_in_unique_senders":
            fan_in[
                "unique_senders"
            ],

        "fan_in_transaction_count":
            fan_in[
                "transaction_count"
            ],

        "fan_in_total_amount":
            fan_in[
                "total_amount"
            ],

        "fan_out_unique_receivers":
            fan_out[
                "unique_receivers"
            ],

        "fan_out_transaction_count":
            fan_out[
                "transaction_count"
            ],

        "fan_out_total_amount":
            fan_out[
                "total_amount"
            ],

        "rapid_hop_count":
            len(
                rapid_hops
            ),

        "rapid_hops":
            rapid_hops,

        "cycle_count":
            len(
                cycles
            ),

        "temporal_cycles":
            cycles,

        "patterns":
            " | ".join(
                patterns
            )
    }


# ============================================================
# 10. RUN INVESTIGATION
# ============================================================

top_transactions = (
    tx_ranked
    .sort_values(
        "RISK_SCORE",
        ascending=False
    )
    .head(TOP_N)
)

reports = []

for row in top_transactions.itertuples(
    index=False
):

    reports.append(
        investigate(row)
    )

# ============================================================
# 11. SUMMARY TABLE
# ============================================================

summary = pd.DataFrame([
    {
        key: value
        for key, value in report.items()
        if key not in [
            "reverse_accounts",
            "reverse_transactions",
            "forward_accounts",
            "forward_transactions",
            "rapid_hops",
            "temporal_cycles"
        ]
    }
    for report in reports
])

print("\n" + "=" * 70)
print("BOUNDED INVESTIGATION SUMMARY")
print("=" * 70)

print(
    summary[
        [
            "TX_ID",
            "sender",
            "receiver",
            "amount",
            "timestamp",
            "risk_score",
            "reverse_transaction_count",
            "forward_transaction_count",
            "fan_in_unique_senders",
            "fan_out_unique_receivers",
            "rapid_hop_count",
            "cycle_count",
            "patterns"
        ]
    ]
    .to_string(index=False)
)

# ============================================================
# 12. SAVE SUMMARY
# ============================================================

SUMMARY_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_bounded_investigation_summary.csv"
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

# ============================================================
# 13. SAVE DETAILED JSON
# ============================================================

JSON_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_bounded_investigations.json"
)

with open(
    JSON_PATH,
    "w"
) as f:

    json.dump(
        reports,
        f,
        indent=2
    )

# ============================================================
# 14. PATTERN COUNTS
# ============================================================

pattern_counts = defaultdict(int)

for report in reports:

    for pattern in report[
        "patterns"
    ].split(" | "):

        pattern_counts[
            pattern
        ] += 1

pattern_df = pd.DataFrame(
    [
        {
            "pattern": pattern,
            "transaction_count": count
        }
        for pattern, count
        in pattern_counts.items()
    ]
).sort_values(
    "transaction_count",
    ascending=False
)

print("\n" + "=" * 70)
print("BOUNDED PATTERN COUNTS")
print("=" * 70)

print(
    pattern_df.to_string(
        index=False
    )
)

PATTERN_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_bounded_patterns.csv"
)

pattern_df.to_csv(
    PATTERN_PATH,
    index=False
)

# ============================================================
# 15. SHOW HIGHEST-RISK CASE
# ============================================================

if len(reports) > 0:

    report = reports[0]

    print("\n" + "=" * 70)
    print("EXAMPLE AML INVESTIGATION")
    print("=" * 70)

    print(
        f"Transaction: {report['TX_ID']}"
    )

    print(
        f"Flow: {report['sender']} -> "
        f"{report['receiver']}"
    )

    print(
        f"Amount: {report['amount']:.2f}"
    )

    print(
        f"Timestamp: {report['timestamp']}"
    )

    print(
        f"Risk score: {report['risk_score']:.6f}"
    )

    print(
        "\nPatterns:"
    )

    print(
        report["patterns"]
    )

    print(
        "\nReverse accounts:"
    )

    print(
        report["reverse_accounts"]
    )

    print(
        "\nReverse transactions:"
    )

    for r in report[
        "reverse_transactions"
    ][:15]:

        print(
            f"  {r['sender']} -> "
            f"{r['receiver']} | "
            f"amount={r['amount']:.2f} | "
            f"t={r['timestamp']} | "
            f"TX={r['tx_id']}"
        )

    print(
        "\nForward accounts:"
    )

    print(
        report["forward_accounts"]
    )

    print(
        "\nRapid hops:"
    )

    for hop in report[
        "rapid_hops"
    ][:10]:

        print(
            f"  {hop['from_account']} -> "
            f"{hop['to_account']} | "
            f"gap={hop['time_gap']} | "
            f"TX={hop['tx_id']}"
        )

    print(
        "\nTemporal cycles:"
    )

    for cycle in report[
        "temporal_cycles"
    ]:

        print(
            "  ",
            " -> ".join(
                map(
                    str,
                    cycle["path"]
                )
            )
        )

# ============================================================
# 16. COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("STAGE 12B COMPLETE")
print("=" * 70)

print(
    "\nSaved:"
)

print(
    "-",
    SUMMARY_PATH
)

print(
    "-",
    JSON_PATH
)

print(
    "-",
    PATTERN_PATH
)

print("\n✓ Reverse trace bounded by time/depth/accounts/transactions")
print("✓ Forward trace bounded by time/depth/accounts/transactions")
print("✓ Rapid-hop detection localized to suspicious transaction")
print("✓ Cycle detection bounded")
print("✓ Investigation output made interpretable")

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 12C
# RISK-AWARE TEMPORAL AML INVESTIGATION
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from collections import defaultdict, deque

PROJECT_DIR = "/content/drive/MyDrive/TraceGraph"

TX_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/transactions_validated.csv"
)

RISK_PATH = os.path.join(
    PROJECT_DIR,
    "results/tracegraph_transaction_risk_scores.csv"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "results"
)

# ============================================================
# CONFIGURATION
# ============================================================

HIGH_RISK_THRESHOLD = 0.90
MEDIUM_RISK_THRESHOLD = 0.50

TRACE_TIME_WINDOW = 10

MAX_DEPTH = 3
MAX_ACCOUNTS = 15
MAX_TRANSACTIONS = 30

MAX_CONTEXT_TRANSACTIONS = 10

MAX_CYCLES = 5
MAX_CYCLE_LENGTH = 5
MAX_CYCLE_TIME = 20

TOP_N = 25

print("=" * 70)
print("TRACEGRAPH — STAGE 12C")
print("RISK-AWARE TEMPORAL AML INVESTIGATION")
print("=" * 70)

# ============================================================
# 1. LOAD
# ============================================================

tx = pd.read_csv(
    TX_PATH
)

risk_tx = pd.read_csv(
    RISK_PATH
)

tx["TX_ID"] = tx["TX_ID"].astype(int)
tx["SENDER_ACCOUNT_ID"] = (
    tx["SENDER_ACCOUNT_ID"].astype(int)
)
tx["RECEIVER_ACCOUNT_ID"] = (
    tx["RECEIVER_ACCOUNT_ID"].astype(int)
)
tx["TIMESTAMP"] = tx["TIMESTAMP"].astype(int)
tx["TX_AMOUNT"] = pd.to_numeric(
    tx["TX_AMOUNT"]
)

# Risk scores are generated from the
# Stage 11B exploratory hybrid model.

risk_lookup = (
    risk_tx[
        [
            "TX_ID",
            "RISK_SCORE"
        ]
    ]
    .drop_duplicates(
        "TX_ID"
    )
)

tx = tx.merge(
    risk_lookup,
    on="TX_ID",
    how="left"
)

tx["RISK_SCORE"] = (
    tx["RISK_SCORE"]
    .fillna(0)
)

# ============================================================
# 2. RISK CATEGORIES
# ============================================================

def risk_category(score):

    if score >= HIGH_RISK_THRESHOLD:
        return "HIGH"

    if score >= MEDIUM_RISK_THRESHOLD:
        return "MEDIUM"

    return "NORMAL"


tx["RISK_CATEGORY"] = (
    tx["RISK_SCORE"]
    .apply(risk_category)
)

print("\nRisk distribution:")
print(
    tx["RISK_CATEGORY"]
    .value_counts()
)

# ============================================================
# 3. TEMPORAL INDEX
# ============================================================

incoming_index = defaultdict(list)
outgoing_index = defaultdict(list)

for row in tx.itertuples(index=False):

    record = {
        "tx_id":
            int(row.TX_ID),

        "sender":
            int(row.SENDER_ACCOUNT_ID),

        "receiver":
            int(row.RECEIVER_ACCOUNT_ID),

        "amount":
            float(row.TX_AMOUNT),

        "timestamp":
            int(row.TIMESTAMP),

        "risk":
            float(row.RISK_SCORE),

        "risk_category":
            row.RISK_CATEGORY,

        "is_fraud":
            bool(row.IS_FRAUD),

        "alert_id":
            int(row.ALERT_ID)
    }

    incoming_index[
        record["receiver"]
    ].append(record)

    outgoing_index[
        record["sender"]
    ].append(record)

for account in incoming_index:

    incoming_index[
        account
    ].sort(
        key=lambda x: (
            x["timestamp"],
            x["tx_id"]
        )
    )

for account in outgoing_index:

    outgoing_index[
        account
    ].sort(
        key=lambda x: (
            x["timestamp"],
            x["tx_id"]
        )
    )

# ============================================================
# 4. RISK-AWARE REVERSE TRACE
# ============================================================

def risk_aware_reverse_trace(
    start_account,
    start_timestamp
):

    visited_accounts = {
        start_account
    }

    visited_transactions = set()

    suspicious_transactions = []
    context_transactions = []

    queue = deque([
        (
            start_account,
            start_timestamp,
            0
        )
    ])

    while queue:

        account, current_time, depth = (
            queue.popleft()
        )

        if depth >= MAX_DEPTH:
            continue

        lower_time = max(
            0,
            current_time - TRACE_TIME_WINDOW
        )

        candidates = [
            r
            for r in incoming_index.get(
                account,
                []
            )
            if (
                lower_time
                <= r["timestamp"]
                <= current_time
            )
            and
            r["sender"] != account
        ]

        # Highest-risk transactions first
        candidates.sort(
            key=lambda x: (
                x["risk"],
                x["timestamp"]
            ),
            reverse=True
        )

        for record in candidates:

            if (
                record["tx_id"]
                in visited_transactions
            ):
                continue

            # HIGH-RISK transaction
            if (
                record["risk"]
                >= HIGH_RISK_THRESHOLD
            ):

                visited_transactions.add(
                    record["tx_id"]
                )

                suspicious_transactions.append(
                    record
                )

            # MEDIUM risk gets retained only
            # as limited supporting context.
            elif (
                record["risk"]
                >= MEDIUM_RISK_THRESHOLD
                and
                len(context_transactions)
                <
                MAX_CONTEXT_TRANSACTIONS
            ):

                visited_transactions.add(
                    record["tx_id"]
                )

                context_transactions.append(
                    record
                )

            else:
                continue

            if (
                len(suspicious_transactions)
                +
                len(context_transactions)
                >= MAX_TRANSACTIONS
            ):
                break

            sender = record["sender"]

            # Continue only through suspicious
            # network participants.
            if (
                record["risk"]
                >= HIGH_RISK_THRESHOLD
                and
                sender not in visited_accounts
                and
                len(visited_accounts)
                <
                MAX_ACCOUNTS
            ):

                visited_accounts.add(
                    sender
                )

                queue.append(
                    (
                        sender,
                        record["timestamp"],
                        depth + 1
                    )
                )

        if (
            len(suspicious_transactions)
            +
            len(context_transactions)
            >= MAX_TRANSACTIONS
        ):
            break

    return {
        "accounts":
            sorted(
                visited_accounts
            ),

        "suspicious_transactions":
            suspicious_transactions,

        "context_transactions":
            context_transactions
    }


# ============================================================
# 5. RISK-AWARE FORWARD TRACE
# ============================================================

def risk_aware_forward_trace(
    start_account,
    start_timestamp
):

    visited_accounts = {
        start_account
    }

    visited_transactions = set()

    suspicious_transactions = []
    context_transactions = []

    queue = deque([
        (
            start_account,
            start_timestamp,
            0
        )
    ])

    while queue:

        account, current_time, depth = (
            queue.popleft()
        )

        if depth >= MAX_DEPTH:
            continue

        upper_time = (
            current_time
            +
            TRACE_TIME_WINDOW
        )

        candidates = [
            r
            for r in outgoing_index.get(
                account,
                []
            )
            if (
                current_time
                <= r["timestamp"]
                <= upper_time
            )
            and
            r["receiver"] != account
        ]

        candidates.sort(
            key=lambda x: (
                x["risk"],
                -x["timestamp"]
            ),
            reverse=True
        )

        for record in candidates:

            if (
                record["tx_id"]
                in visited_transactions
            ):
                continue

            if (
                record["risk"]
                >= HIGH_RISK_THRESHOLD
            ):

                visited_transactions.add(
                    record["tx_id"]
                )

                suspicious_transactions.append(
                    record
                )

            elif (
                record["risk"]
                >= MEDIUM_RISK_THRESHOLD
                and
                len(context_transactions)
                <
                MAX_CONTEXT_TRANSACTIONS
            ):

                visited_transactions.add(
                    record["tx_id"]
                )

                context_transactions.append(
                    record
                )

            else:
                continue

            if (
                len(suspicious_transactions)
                +
                len(context_transactions)
                >= MAX_TRANSACTIONS
            ):
                break

            receiver = record["receiver"]

            if (
                record["risk"]
                >= HIGH_RISK_THRESHOLD
                and
                receiver not in visited_accounts
                and
                len(visited_accounts)
                <
                MAX_ACCOUNTS
            ):

                visited_accounts.add(
                    receiver
                )

                queue.append(
                    (
                        receiver,
                        record["timestamp"],
                        depth + 1
                    )
                )

        if (
            len(suspicious_transactions)
            +
            len(context_transactions)
            >= MAX_TRANSACTIONS
        ):
            break

    return {
        "accounts":
            sorted(
                visited_accounts
            ),

        "suspicious_transactions":
            suspicious_transactions,

        "context_transactions":
            context_transactions
    }


# ============================================================
# 6. RISK-AWARE FAN-IN
# ============================================================

def risk_aware_fan_in(
    account,
    timestamp
):

    lower_time = max(
        0,
        timestamp - TRACE_TIME_WINDOW
    )

    records = [
        r
        for r in incoming_index.get(
            account,
            []
        )
        if (
            lower_time
            <= r["timestamp"]
            <= timestamp
        )
        and
        r["sender"] != account
        and
        r["risk"]
        >= HIGH_RISK_THRESHOLD
    ]

    senders = sorted(
        set(
            r["sender"]
            for r in records
        )
    )

    return {
        "transactions":
            records,

        "transaction_count":
            len(records),

        "unique_senders":
            len(senders),

        "senders":
            senders,

        "total_amount":
            sum(
                r["amount"]
                for r in records
            )
    }


# ============================================================
# 7. RISK-AWARE FAN-OUT
# ============================================================

def risk_aware_fan_out(
    account,
    timestamp
):

    upper_time = (
        timestamp
        +
        TRACE_TIME_WINDOW
    )

    records = [
        r
        for r in outgoing_index.get(
            account,
            []
        )
        if (
            timestamp
            <= r["timestamp"]
            <= upper_time
        )
        and
        r["receiver"] != account
        and
        r["risk"]
        >= HIGH_RISK_THRESHOLD
    ]

    receivers = sorted(
        set(
            r["receiver"]
            for r in records
        )
    )

    return {
        "transactions":
            records,

        "transaction_count":
            len(records),

        "unique_receivers":
            len(receivers),

        "receivers":
            receivers,

        "total_amount":
            sum(
                r["amount"]
                for r in records
            )
    }


# ============================================================
# 8. RISK-AWARE RAPID HOPS
# ============================================================

def risk_aware_rapid_hops(
    sender,
    receiver,
    timestamp
):

    hops = []

    # Suspicious transaction:
    #
    # sender -> receiver
    #
    # then:
    #
    # receiver -> next_account

    for second in outgoing_index.get(
        receiver,
        []
    ):

        if second["timestamp"] < timestamp:
            continue

        if (
            second["timestamp"]
            >
            timestamp + TRACE_TIME_WINDOW
        ):
            continue

        if (
            second["risk"]
            <
            HIGH_RISK_THRESHOLD
        ):
            continue

        hops.append({
            "from":
                receiver,

            "to":
                second["receiver"],

            "tx_id":
                second["tx_id"],

            "timestamp":
                second["timestamp"],

            "time_gap":
                second["timestamp"]
                -
                timestamp,

            "amount":
                second["amount"],

            "risk":
                second["risk"]
        })

        if len(hops) >= 10:
            break

    return hops


# ============================================================
# 9. RISK-AWARE CYCLE DETECTION
# ============================================================

def risk_aware_cycles(
    start_account,
    start_timestamp
):

    cycles = []

    path = [
        start_account
    ]

    visited = {
        start_account
    }

    def dfs(
        current,
        current_time
    ):

        if (
            len(cycles)
            >= MAX_CYCLES
        ):
            return

        if (
            len(path)
            >
            MAX_CYCLE_LENGTH
        ):
            return

        for record in outgoing_index.get(
            current,
            []
        ):

            # Only suspicious edges
            if (
                record["risk"]
                <
                HIGH_RISK_THRESHOLD
            ):
                continue

            next_node = record[
                "receiver"
            ]

            next_time = record[
                "timestamp"
            ]

            if next_time < current_time:
                continue

            if (
                next_time
                >
                start_timestamp
                +
                MAX_CYCLE_TIME
            ):
                continue

            if (
                next_node
                ==
                start_account
            ):

                if len(path) >= 2:

                    cycles.append({
                        "path":
                            path.copy()
                            +
                            [
                                start_account
                            ],

                        "length":
                            len(path),

                        "start_timestamp":
                            start_timestamp,

                        "end_timestamp":
                            next_time
                    })

                continue

            if next_node in visited:
                continue

            visited.add(
                next_node
            )

            path.append(
                next_node
            )

            dfs(
                next_node,
                next_time
            )

            path.pop()

            visited.remove(
                next_node
            )

            if (
                len(cycles)
                >= MAX_CYCLES
            ):
                return

    dfs(
        start_account,
        start_timestamp
    )

    return cycles


# ============================================================
# 10. NETWORK RISK SCORE
# ============================================================

def calculate_network_risk(
    transaction_risk,
    reverse,
    forward,
    fan_in,
    fan_out,
    rapid_hops,
    cycles
):

    components = []

    # Base transaction risk
    components.append(
        transaction_risk
    )

    # Suspicious reverse edges
    reverse_count = len(
        reverse[
            "suspicious_transactions"
        ]
    )

    if reverse_count > 0:
        components.append(
            min(
                1.0,
                0.7
                +
                0.05 * reverse_count
            )
        )

    # Suspicious forward edges
    forward_count = len(
        forward[
            "suspicious_transactions"
        ]
    )

    if forward_count > 0:
        components.append(
            min(
                1.0,
                0.7
                +
                0.05 * forward_count
            )
        )

    # Fan-in
    if (
        fan_in["unique_senders"]
        >= 3
    ):
        components.append(
            min(
                1.0,
                0.6
                +
                0.1
                *
                fan_in[
                    "unique_senders"
                ]
            )
        )

    # Fan-out
    if (
        fan_out["unique_receivers"]
        >= 3
    ):
        components.append(
            min(
                1.0,
                0.6
                +
                0.1
                *
                fan_out[
                    "unique_receivers"
                ]
            )
        )

    # Rapid hops
    if len(rapid_hops) > 0:
        components.append(
            0.85
        )

    # Cycle
    if len(cycles) > 0:
        components.append(
            0.95
        )

    return float(
        np.mean(
            components
        )
    )


# ============================================================
# 11. INVESTIGATION GENERATOR
# ============================================================

def investigate(row):

    tx_id = int(
        row.TX_ID
    )

    sender = int(
        row.SENDER_ACCOUNT_ID
    )

    receiver = int(
        row.RECEIVER_ACCOUNT_ID
    )

    timestamp = int(
        row.TIMESTAMP
    )

    amount = float(
        row.TX_AMOUNT
    )

    risk = float(
        row.RISK_SCORE
    )

    reverse = risk_aware_reverse_trace(
        receiver,
        timestamp
    )

    forward = risk_aware_forward_trace(
        sender,
        timestamp
    )

    fan_in = risk_aware_fan_in(
        receiver,
        timestamp
    )

    fan_out = risk_aware_fan_out(
        sender,
        timestamp
    )

    rapid_hops = risk_aware_rapid_hops(
        sender,
        receiver,
        timestamp
    )

    cycles = risk_aware_cycles(
        receiver,
        timestamp
    )

    network_risk = calculate_network_risk(
        risk,
        reverse,
        forward,
        fan_in,
        fan_out,
        rapid_hops,
        cycles
    )

    patterns = []

    if (
        fan_in["unique_senders"]
        >= 3
    ):
        patterns.append(
            "risk-aware fan-in"
        )

    if (
        fan_out["unique_receivers"]
        >= 3
    ):
        patterns.append(
            "risk-aware fan-out"
        )

    if len(rapid_hops) > 0:
        patterns.append(
            "risk-aware rapid-hop"
        )

    if len(cycles) > 0:
        patterns.append(
            "risk-aware temporal-cycle"
        )

    if len(
        reverse[
            "suspicious_transactions"
        ]
    ) > 0:

        patterns.append(
            "suspicious reverse funding"
        )

    if len(
        forward[
            "suspicious_transactions"
        ]
    ) > 0:

        patterns.append(
            "suspicious forward flow"
        )

    if not patterns:
        patterns.append(
            "isolated high-risk transaction"
        )

    return {

        "TX_ID":
            tx_id,

        "sender":
            sender,

        "receiver":
            receiver,

        "amount":
            amount,

        "timestamp":
            timestamp,

        "transaction_risk":
            risk,

        "network_risk":
            network_risk,

        "reverse_accounts":
            reverse["accounts"],

        "reverse_suspicious_count":
            len(
                reverse[
                    "suspicious_transactions"
                ]
            ),

        "reverse_context_count":
            len(
                reverse[
                    "context_transactions"
                ]
            ),

        "reverse_suspicious_transactions":
            reverse[
                "suspicious_transactions"
            ],

        "reverse_context_transactions":
            reverse[
                "context_transactions"
            ],

        "forward_accounts":
            forward["accounts"],

        "forward_suspicious_count":
            len(
                forward[
                    "suspicious_transactions"
                ]
            ),

        "forward_context_count":
            len(
                forward[
                    "context_transactions"
                ]
            ),

        "forward_suspicious_transactions":
            forward[
                "suspicious_transactions"
            ],

        "forward_context_transactions":
            forward[
                "context_transactions"
            ],

        "fan_in_unique_senders":
            fan_in[
                "unique_senders"
            ],

        "fan_in_total_amount":
            fan_in[
                "total_amount"
            ],

        "fan_out_unique_receivers":
            fan_out[
                "unique_receivers"
            ],

        "fan_out_total_amount":
            fan_out[
                "total_amount"
            ],

        "rapid_hop_count":
            len(
                rapid_hops
            ),

        "rapid_hops":
            rapid_hops,

        "cycle_count":
            len(cycles),

        "temporal_cycles":
            cycles,

        "patterns":
            " | ".join(
                patterns
            )
    }


# ============================================================
# 12. INVESTIGATE TOP TRANSACTIONS
# ============================================================

top_transactions = (
    tx
    .sort_values(
        "RISK_SCORE",
        ascending=False
    )
    .head(TOP_N)
)

reports = []

for row in top_transactions.itertuples(
    index=False
):

    reports.append(
        investigate(row)
    )

# ============================================================
# 13. SUMMARY TABLE
# ============================================================

summary = pd.DataFrame([
    {
        key: value
        for key, value in report.items()
        if key not in [
            "reverse_accounts",
            "reverse_suspicious_transactions",
            "reverse_context_transactions",
            "forward_accounts",
            "forward_suspicious_transactions",
            "forward_context_transactions",
            "rapid_hops",
            "temporal_cycles"
        ]
    }
    for report in reports
])

print("\n" + "=" * 70)
print("RISK-AWARE INVESTIGATION SUMMARY")
print("=" * 70)

print(
    summary[
        [
            "TX_ID",
            "sender",
            "receiver",
            "amount",
            "timestamp",
            "transaction_risk",
            "network_risk",
            "reverse_suspicious_count",
            "forward_suspicious_count",
            "fan_in_unique_senders",
            "fan_out_unique_receivers",
            "rapid_hop_count",
            "cycle_count",
            "patterns"
        ]
    ]
    .to_string(index=False)
)

# ============================================================
# 14. SAVE SUMMARY
# ============================================================

SUMMARY_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_risk_aware_investigation_summary.csv"
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

# ============================================================
# 15. SAVE DETAILED JSON
# ============================================================

JSON_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_risk_aware_investigations.json"
)

with open(
    JSON_PATH,
    "w"
) as f:

    json.dump(
        reports,
        f,
        indent=2
    )

# ============================================================
# 16. PATTERN SUMMARY
# ============================================================

pattern_counts = defaultdict(int)

for report in reports:

    for pattern in report[
        "patterns"
    ].split(" | "):

        pattern_counts[
            pattern
        ] += 1

pattern_df = pd.DataFrame(
    [
        {
            "pattern":
                pattern,

            "transaction_count":
                count
        }
        for pattern, count
        in pattern_counts.items()
    ]
).sort_values(
    "transaction_count",
    ascending=False
)

print("\n" + "=" * 70)
print("RISK-AWARE PATTERN COUNTS")
print("=" * 70)

print(
    pattern_df.to_string(
        index=False
    )
)

PATTERN_PATH = os.path.join(
    RESULT_DIR,
    "tracegraph_risk_aware_patterns.csv"
)

pattern_df.to_csv(
    PATTERN_PATH,
    index=False
)

# ============================================================
# 17. HIGHEST-RISK CASE
# ============================================================

if len(reports) > 0:

    report = reports[0]

    print("\n" + "=" * 70)
    print("TRACEGRAPH — AML INVESTIGATION CASE")
    print("=" * 70)

    print(
        f"Transaction ID : "
        f"{report['TX_ID']}"
    )

    print(
        f"Flow           : "
        f"{report['sender']} -> "
        f"{report['receiver']}"
    )

    print(
        f"Amount         : "
        f"{report['amount']:.2f}"
    )

    print(
        f"Timestamp      : "
        f"{report['timestamp']}"
    )

    print(
        f"Transaction risk: "
        f"{report['transaction_risk']:.6f}"
    )

    print(
        f"Network risk   : "
        f"{report['network_risk']:.6f}"
    )

    print(
        "\nDetected patterns:"
    )

    print(
        report["patterns"]
    )

    print(
        "\nSuspicious reverse transactions:"
    )

    for r in report[
        "reverse_suspicious_transactions"
    ]:

        print(
            f"  {r['sender']} -> "
            f"{r['receiver']} | "
            f"{r['amount']:.2f} | "
            f"t={r['timestamp']} | "
            f"risk={r['risk']:.4f}"
        )

    print(
        "\nSuspicious forward transactions:"
    )

    for r in report[
        "forward_suspicious_transactions"
    ]:

        print(
            f"  {r['sender']} -> "
            f"{r['receiver']} | "
            f"{r['amount']:.2f} | "
            f"t={r['timestamp']} | "
            f"risk={r['risk']:.4f}"
        )

    print(
        "\nRisk-aware reverse accounts:"
    )

    print(
        report["reverse_accounts"]
    )

    print(
        "\nRisk-aware forward accounts:"
    )

    print(
        report["forward_accounts"]
    )

    print(
        "\nTemporal cycles:"
    )

    for cycle in report[
        "temporal_cycles"
    ]:

        print(
            "  "
            +
            " -> ".join(
                map(
                    str,
                    cycle["path"]
                )
            )
        )

# ============================================================
# 18. COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("STAGE 12C COMPLETE")
print("=" * 70)

print("\nSaved:")
print("-", SUMMARY_PATH)
print("-", JSON_PATH)
print("-", PATTERN_PATH)

print("\n✓ Risk-aware reverse tracing")
print("✓ Risk-aware forward tracing")
print("✓ Suspicious-edge prioritization")
print("✓ Limited normal-context retention")
print("✓ Risk-aware fan-in detection")
print("✓ Risk-aware fan-out detection")
print("✓ Risk-aware rapid-hop detection")
print("✓ Risk-aware temporal-cycle detection")
print("✓ Network risk scoring")
print("✓ Investigator-oriented output")

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 12D
# FINAL RISK-AWARE INVESTIGATION
#
# Goals:
# 1. Keep ONLY genuinely high-risk transactions
# 2. Detect suspicious fan-in / fan-out
# 3. Detect rapid hops
# 4. Detect directed suspicious cycles
# 5. Perform bounded reverse/forward tracing
# 6. Generate final investigation artifacts
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import networkx as nx
from collections import defaultdict

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/TraceGraph"

DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

TX_PATH = os.path.join(
    DATA_DIR,
    "transactions_validated.csv"
)

RISK_PATH = os.path.join(
    RESULTS_DIR,
    "tracegraph_transaction_risk_scores.csv"
)

# ------------------------------------------------------------
# 2. CONFIGURATION
# ------------------------------------------------------------

HIGH_RISK_THRESHOLD = 0.90

# Time window for identifying related suspicious activity.
PATTERN_TIME_WINDOW = 25

# Time window used for reverse / forward investigation.
TRACE_TIME_WINDOW = 25

# Maximum trace depth.
MAX_TRACE_DEPTH = 5

# Maximum transactions returned in a trace.
MAX_TRACE_TRANSACTIONS = 30

# Rapid hop threshold.
RAPID_HOP_WINDOW = 10

# Cycle length constraints.
MIN_CYCLE_LENGTH = 3
MAX_CYCLE_LENGTH = 6

# Maximum temporal span allowed for a suspicious cycle.
MAX_CYCLE_TIME_SPAN = 25


# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

print("=" * 70)
print("TRACEGRAPH STAGE 12D")
print("FINAL RISK-AWARE INVESTIGATION")
print("=" * 70)

tx = pd.read_csv(TX_PATH)
risk = pd.read_csv(RISK_PATH)

print("\nTransactions loaded:", tx.shape)
print("Risk scores loaded:", risk.shape)

# ------------------------------------------------------------
# 4. NORMALIZE COLUMN TYPES
# ------------------------------------------------------------

tx["TX_ID"] = tx["TX_ID"].astype(int)
tx["SENDER_ACCOUNT_ID"] = tx["SENDER_ACCOUNT_ID"].astype(int)
tx["RECEIVER_ACCOUNT_ID"] = tx["RECEIVER_ACCOUNT_ID"].astype(int)
tx["TIMESTAMP"] = tx["TIMESTAMP"].astype(int)
tx["TX_AMOUNT"] = tx["TX_AMOUNT"].astype(float)

risk["TX_ID"] = risk["TX_ID"].astype(int)
risk["RISK_SCORE"] = risk["RISK_SCORE"].astype(float)

# ------------------------------------------------------------
# 5. MERGE RISK SCORES
# ------------------------------------------------------------

tx = tx.merge(
    risk[["TX_ID", "RISK_SCORE"]],
    on="TX_ID",
    how="left"
)

if tx["RISK_SCORE"].isna().any():
    missing = tx["RISK_SCORE"].isna().sum()
    raise ValueError(
        f"{missing} transactions do not have risk scores."
    )

# ------------------------------------------------------------
# 6. ISOLATE HIGH-RISK TRANSACTIONS
# ------------------------------------------------------------

high_risk = tx[
    tx["RISK_SCORE"] >= HIGH_RISK_THRESHOLD
].copy()

high_risk = high_risk.sort_values(
    ["TIMESTAMP", "TX_ID"]
).reset_index(drop=True)

print("\n" + "-" * 70)
print("RISK DISTRIBUTION")
print("-" * 70)

print(
    tx["RISK_SCORE"]
    .ge(HIGH_RISK_THRESHOLD)
    .value_counts()
    .rename({True: "HIGH_RISK", False: "NORMAL"})
)

print(
    f"\nHigh-risk transactions: {len(high_risk)}"
)

if len(high_risk) != 18:
    print(
        "WARNING: Expected 18 high-risk transactions "
        f"based on the injected fraud examples, but found {len(high_risk)}."
    )

# ------------------------------------------------------------
# 7. DISPLAY HIGH-RISK TRANSACTIONS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("HIGH-RISK TRANSACTIONS")
print("-" * 70)

display(
    high_risk[
        [
            "TX_ID",
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TX_AMOUNT",
            "TIMESTAMP",
            "RISK_SCORE"
        ]
    ]
)


# ============================================================
# 8. BUILD SUSPICIOUS HIGH-RISK GRAPH
# ============================================================

# IMPORTANT:
# This graph contains ONLY high-risk transactions.
#
# This prevents the dense normal transaction graph from
# creating artificial cycles/patterns.

G = nx.DiGraph()

for _, row in high_risk.iterrows():

    sender = int(row["SENDER_ACCOUNT_ID"])
    receiver = int(row["RECEIVER_ACCOUNT_ID"])

    G.add_edge(
        sender,
        receiver,
        TX_ID=int(row["TX_ID"]),
        amount=float(row["TX_AMOUNT"]),
        timestamp=int(row["TIMESTAMP"]),
        risk_score=float(row["RISK_SCORE"])
    )

print("\n" + "-" * 70)
print("SUSPICIOUS HIGH-RISK GRAPH")
print("-" * 70)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


# ============================================================
# 9. DETECT DIRECTED SUSPICIOUS CYCLES
# ============================================================

print("\n" + "=" * 70)
print("DETECTING SUSPICIOUS CYCLES")
print("=" * 70)

# NetworkX simple_cycles is appropriate here because the
# suspicious graph contains only the 18 high-risk edges.

raw_cycles = list(
    nx.simple_cycles(G)
)

print(
    "Raw directed cycles found:",
    len(raw_cycles)
)

# ------------------------------------------------------------
# Canonical representation to remove rotational duplicates
# ------------------------------------------------------------

def canonical_cycle(cycle):
    """
    Rotate cycle so the smallest account ID appears first.
    """
    cycle = list(cycle)

    rotations = []

    for i in range(len(cycle)):
        rotated = cycle[i:] + cycle[:i]
        rotations.append(tuple(rotated))

    return min(rotations)


unique_cycles = {}

for cycle in raw_cycles:

    cycle_len = len(cycle)

    if cycle_len < MIN_CYCLE_LENGTH:
        continue

    if cycle_len > MAX_CYCLE_LENGTH:
        continue

    canonical = canonical_cycle(cycle)

    unique_cycles[canonical] = canonical


# ------------------------------------------------------------
# Extract cycle transaction information
# ------------------------------------------------------------

cycle_records = []

for cycle_nodes in unique_cycles.values():

    cycle_nodes = list(cycle_nodes)

    edge_records = []

    valid_cycle = True

    for i in range(len(cycle_nodes)):

        sender = cycle_nodes[i]
        receiver = cycle_nodes[
            (i + 1) % len(cycle_nodes)
        ]

        if not G.has_edge(sender, receiver):
            valid_cycle = False
            break

        edge_data = G[sender][receiver]

        edge_records.append({
            "sender": int(sender),
            "receiver": int(receiver),
            "tx_id": int(edge_data["TX_ID"]),
            "amount": float(edge_data["amount"]),
            "timestamp": int(edge_data["timestamp"]),
            "risk_score": float(edge_data["risk_score"])
        })

    if not valid_cycle:
        continue

    timestamps = [
        e["timestamp"]
        for e in edge_records
    ]

    time_span = max(timestamps) - min(timestamps)

    if time_span > MAX_CYCLE_TIME_SPAN:
        continue

    cycle_records.append({
        "cycle_id": len(cycle_records) + 1,
        "cycle_length": len(edge_records),
        "accounts": cycle_nodes,
        "transactions": edge_records,
        "timestamps": timestamps,
        "time_span": time_span,
        "min_risk_score": min(
            e["risk_score"]
            for e in edge_records
        ),
        "avg_risk_score": np.mean(
            [
                e["risk_score"]
                for e in edge_records
            ]
        )
    })


# ------------------------------------------------------------
# Display cycles
# ------------------------------------------------------------

print(
    "\nSuspicious temporal cycles detected:",
    len(cycle_records)
)

for cycle in cycle_records:

    account_path = cycle["accounts"] + [
        cycle["accounts"][0]
    ]

    print("\nCycle", cycle["cycle_id"])
    print(
        "Account path:",
        " → ".join(
            map(str, account_path)
        )
    )

    print(
        "Cycle length:",
        cycle["cycle_length"]
    )

    print(
        "Timestamps:",
        cycle["timestamps"]
    )

    print(
        "Time span:",
        cycle["time_span"]
    )


# ============================================================
# 10. MAP TRANSACTIONS TO CYCLES
# ============================================================

tx_to_cycles = defaultdict(list)

for cycle in cycle_records:

    for edge in cycle["transactions"]:

        tx_to_cycles[
            edge["tx_id"]
        ].append(
            cycle["cycle_id"]
        )


# ============================================================
# 11. FAN-IN / FAN-OUT DETECTION
# ============================================================

print("\n" + "=" * 70)
print("DETECTING RISK-AWARE FAN-IN / FAN-OUT")
print("=" * 70)


def suspicious_fan_in(tx_id, receiver, timestamp):

    related = high_risk[
        (
            high_risk["RECEIVER_ACCOUNT_ID"]
            == receiver
        )
        &
        (
            (high_risk["TIMESTAMP"] - timestamp).abs()
            <= PATTERN_TIME_WINDOW
        )
    ]

    senders = sorted(
        related[
            "SENDER_ACCOUNT_ID"
        ].unique()
        .tolist()
    )

    return related, senders


def suspicious_fan_out(tx_id, sender, timestamp):

    related = high_risk[
        (
            high_risk["SENDER_ACCOUNT_ID"]
            == sender
        )
        &
        (
            (high_risk["TIMESTAMP"] - timestamp).abs()
            <= PATTERN_TIME_WINDOW
        )
    ]

    receivers = sorted(
        related[
            "RECEIVER_ACCOUNT_ID"
        ].unique()
        .tolist()
    )

    return related, receivers


# ============================================================
# 12. RAPID-HOP DETECTION
# ============================================================

def find_rapid_hops(sender, receiver, timestamp):

    # A rapid hop occurs when the destination account
    # subsequently sends a high-risk transaction soon after.

    candidates = high_risk[
        (
            high_risk["SENDER_ACCOUNT_ID"]
            == receiver
        )
        &
        (
            high_risk["TIMESTAMP"]
            >= timestamp
        )
        &
        (
            high_risk["TIMESTAMP"]
            <= timestamp + RAPID_HOP_WINDOW
        )
    ].copy()

    return candidates


# ============================================================
# 13. BOUNDED REVERSE TRACE
# ============================================================

def reverse_trace(
    start_account,
    start_timestamp
):

    visited = {
        start_account
    }

    queue = [
        (
            start_account,
            start_timestamp,
            0
        )
    ]

    trace_edges = []

    while queue:

        current_account, current_time, depth = queue.pop(0)

        if depth >= MAX_TRACE_DEPTH:
            continue

        incoming = high_risk[
            (
                high_risk["RECEIVER_ACCOUNT_ID"]
                == current_account
            )
            &
            (
                high_risk["TIMESTAMP"]
                <= current_time
            )
            &
            (
                high_risk["TIMESTAMP"]
                >= current_time - TRACE_TIME_WINDOW
            )
        ].sort_values(
            "TIMESTAMP",
            ascending=False
        )

        for _, row in incoming.iterrows():

            sender = int(
                row["SENDER_ACCOUNT_ID"]
            )

            if len(trace_edges) >= MAX_TRACE_TRANSACTIONS:
                break

            trace_edges.append({
                "TX_ID": int(row["TX_ID"]),
                "sender": sender,
                "receiver": int(
                    row["RECEIVER_ACCOUNT_ID"]
                ),
                "amount": float(
                    row["TX_AMOUNT"]
                ),
                "timestamp": int(
                    row["TIMESTAMP"]
                ),
                "risk_score": float(
                    row["RISK_SCORE"]
                )
            })

            if sender not in visited:

                visited.add(sender)

                queue.append(
                    (
                        sender,
                        int(row["TIMESTAMP"]),
                        depth + 1
                    )
                )

    return trace_edges


# ============================================================
# 14. BOUNDED FORWARD TRACE
# ============================================================

def forward_trace(
    start_account,
    start_timestamp
):

    visited = {
        start_account
    }

    queue = [
        (
            start_account,
            start_timestamp,
            0
        )
    ]

    trace_edges = []

    while queue:

        current_account, current_time, depth = queue.pop(0)

        if depth >= MAX_TRACE_DEPTH:
            continue

        outgoing = high_risk[
            (
                high_risk["SENDER_ACCOUNT_ID"]
                == current_account
            )
            &
            (
                high_risk["TIMESTAMP"]
                >= current_time
            )
            &
            (
                high_risk["TIMESTAMP"]
                <= current_time + TRACE_TIME_WINDOW
            )
        ].sort_values(
            "TIMESTAMP"
        )

        for _, row in outgoing.iterrows():

            receiver = int(
                row["RECEIVER_ACCOUNT_ID"]
            )

            if len(trace_edges) >= MAX_TRACE_TRANSACTIONS:
                break

            trace_edges.append({
                "TX_ID": int(row["TX_ID"]),
                "sender": int(
                    row["SENDER_ACCOUNT_ID"]
                ),
                "receiver": receiver,
                "amount": float(
                    row["TX_AMOUNT"]
                ),
                "timestamp": int(
                    row["TIMESTAMP"]
                ),
                "risk_score": float(
                    row["RISK_SCORE"]
                )
            })

            if receiver not in visited:

                visited.add(receiver)

                queue.append(
                    (
                        receiver,
                        int(row["TIMESTAMP"]),
                        depth + 1
                    )
                )

    return trace_edges


# ============================================================
# 15. BUILD FINAL INVESTIGATION RECORDS
# ============================================================

investigations = []

for _, row in high_risk.iterrows():

    tx_id = int(row["TX_ID"])
    sender = int(row["SENDER_ACCOUNT_ID"])
    receiver = int(row["RECEIVER_ACCOUNT_ID"])
    timestamp = int(row["TIMESTAMP"])

    patterns = []

    # --------------------------------------------------------
    # FAN-IN
    # --------------------------------------------------------

    fan_in_df, fan_in_accounts = suspicious_fan_in(
        tx_id,
        receiver,
        timestamp
    )

    if len(fan_in_accounts) >= 3:

        patterns.append(
            "risk-aware fan-in"
        )

    # --------------------------------------------------------
    # FAN-OUT
    # --------------------------------------------------------

    fan_out_df, fan_out_accounts = suspicious_fan_out(
        tx_id,
        sender,
        timestamp
    )

    if len(fan_out_accounts) >= 3:

        patterns.append(
            "risk-aware fan-out"
        )

    # --------------------------------------------------------
    # REVERSE TRACE
    # --------------------------------------------------------

    reverse_edges = reverse_trace(
        receiver,
        timestamp
    )

    # --------------------------------------------------------
    # FORWARD TRACE
    # --------------------------------------------------------

    forward_edges = forward_trace(
        sender,
        timestamp
    )

    # --------------------------------------------------------
    # SUSPICIOUS REVERSE FUNDING
    # --------------------------------------------------------

    reverse_tx_ids = {
        e["TX_ID"]
        for e in reverse_edges
    }

    if len(reverse_tx_ids) > 1:

        patterns.append(
            "suspicious reverse funding"
        )

    # --------------------------------------------------------
    # SUSPICIOUS FORWARD FLOW
    # --------------------------------------------------------

    forward_tx_ids = {
        e["TX_ID"]
        for e in forward_edges
    }

    if len(forward_tx_ids) > 1:

        patterns.append(
            "suspicious forward flow"
        )

    # --------------------------------------------------------
    # RAPID HOP
    # --------------------------------------------------------

    rapid_hops_df = find_rapid_hops(
        sender,
        receiver,
        timestamp
    )

    rapid_hop_records = []

    for _, hop in rapid_hops_df.iterrows():

        if int(hop["TX_ID"]) == tx_id:
            continue

        rapid_hop_records.append({
            "TX_ID": int(hop["TX_ID"]),
            "sender": int(
                hop["SENDER_ACCOUNT_ID"]
            ),
            "receiver": int(
                hop["RECEIVER_ACCOUNT_ID"]
            ),
            "timestamp": int(
                hop["TIMESTAMP"]
            ),
            "amount": float(
                hop["TX_AMOUNT"]
            )
        })

    if rapid_hop_records:

        patterns.append(
            "risk-aware rapid-hop"
        )

    # --------------------------------------------------------
    # CYCLE
    # --------------------------------------------------------

    cycle_ids = tx_to_cycles.get(
        tx_id,
        []
    )

    if cycle_ids:

        patterns.append(
            "suspicious temporal cycle"
        )

    # --------------------------------------------------------
    # REMOVE DUPLICATES
    # --------------------------------------------------------

    patterns = list(
        dict.fromkeys(patterns)
    )

    # --------------------------------------------------------
    # FINAL CLASSIFICATION
    # --------------------------------------------------------

    if patterns:

        investigation_status = (
            "SUSPICIOUS_NETWORK_ACTIVITY"
        )

    else:

        investigation_status = (
            "HIGH_RISK_TRANSACTION"
        )

    # --------------------------------------------------------
    # SAVE RECORD
    # --------------------------------------------------------

    investigations.append({

        "TX_ID": tx_id,

        "SENDER_ACCOUNT_ID": sender,

        "RECEIVER_ACCOUNT_ID": receiver,

        "TX_AMOUNT": float(
            row["TX_AMOUNT"]
        ),

        "TIMESTAMP": timestamp,

        "RISK_SCORE": float(
            row["RISK_SCORE"]
        ),

        "INVESTIGATION_STATUS":
            investigation_status,

        "PATTERNS":
            patterns,

        "PATTERN_COUNT":
            len(patterns),

        "FAN_IN_ACCOUNT_COUNT":
            len(fan_in_accounts),

        "FAN_IN_ACCOUNTS":
            fan_in_accounts,

        "FAN_OUT_ACCOUNT_COUNT":
            len(fan_out_accounts),

        "FAN_OUT_ACCOUNTS":
            fan_out_accounts,

        "RAPID_HOP_COUNT":
            len(rapid_hop_records),

        "RAPID_HOPS":
            rapid_hop_records,

        "CYCLE_IDS":
            cycle_ids,

        "REVERSE_TRACE":
            reverse_edges,

        "FORWARD_TRACE":
            forward_edges
    })


# ============================================================
# 16. SAVE FINAL INVESTIGATION JSON
# ============================================================

investigation_json_path = os.path.join(
    RESULTS_DIR,
    "tracegraph_final_investigations.json"
)

with open(
    investigation_json_path,
    "w"
) as f:

    json.dump(
        investigations,
        f,
        indent=2
    )


# ============================================================
# 17. CREATE FLAT SUMMARY TABLE
# ============================================================

summary_records = []

for record in investigations:

    summary_records.append({

        "TX_ID":
            record["TX_ID"],

        "SENDER_ACCOUNT_ID":
            record["SENDER_ACCOUNT_ID"],

        "RECEIVER_ACCOUNT_ID":
            record["RECEIVER_ACCOUNT_ID"],

        "TX_AMOUNT":
            record["TX_AMOUNT"],

        "TIMESTAMP":
            record["TIMESTAMP"],

        "RISK_SCORE":
            record["RISK_SCORE"],

        "INVESTIGATION_STATUS":
            record["INVESTIGATION_STATUS"],

        "PATTERN_COUNT":
            record["PATTERN_COUNT"],

        "PATTERNS":
            " | ".join(
                record["PATTERNS"]
            ),

        "FAN_IN_ACCOUNT_COUNT":
            record["FAN_IN_ACCOUNT_COUNT"],

        "FAN_OUT_ACCOUNT_COUNT":
            record["FAN_OUT_ACCOUNT_COUNT"],

        "RAPID_HOP_COUNT":
            record["RAPID_HOP_COUNT"],

        "CYCLE_IDS":
            ",".join(
                map(
                    str,
                    record["CYCLE_IDS"]
                )
            )
    })


summary_df = pd.DataFrame(
    summary_records
)

summary_df = summary_df.sort_values(
    [
        "PATTERN_COUNT",
        "RISK_SCORE"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)


summary_path = os.path.join(
    RESULTS_DIR,
    "tracegraph_final_investigation_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 18. SAVE CYCLE TABLE
# ============================================================

cycle_rows = []

for cycle in cycle_records:

    account_path = cycle["accounts"] + [
        cycle["accounts"][0]
    ]

    tx_ids = [
        e["tx_id"]
        for e in cycle["transactions"]
    ]

    amounts = [
        e["amount"]
        for e in cycle["transactions"]
    ]

    timestamps = [
        e["timestamp"]
        for e in cycle["transactions"]
    ]

    cycle_rows.append({

        "cycle_id":
            cycle["cycle_id"],

        "cycle_length":
            cycle["cycle_length"],

        "account_path":
            " -> ".join(
                map(str, account_path)
            ),

        "transaction_ids":
            ",".join(
                map(str, tx_ids)
            ),

        "amounts":
            ",".join(
                f"{x:.2f}"
                for x in amounts
            ),

        "timestamps":
            ",".join(
                map(str, timestamps)
            ),

        "time_span":
            cycle["time_span"],

        "min_risk_score":
            cycle["min_risk_score"],

        "avg_risk_score":
            cycle["avg_risk_score"]
    })


cycles_df = pd.DataFrame(
    cycle_rows
)

cycles_path = os.path.join(
    RESULTS_DIR,
    "tracegraph_final_suspicious_cycles.csv"
)

cycles_df.to_csv(
    cycles_path,
    index=False
)


# ============================================================
# 19. PATTERN SUMMARY
# ============================================================

pattern_counter = defaultdict(int)

for record in investigations:

    for pattern in record["PATTERNS"]:

        pattern_counter[
            pattern
        ] += 1


pattern_summary = pd.DataFrame(
    [
        {
            "PATTERN": pattern,
            "TRANSACTION_COUNT": count
        }
        for pattern, count
        in pattern_counter.items()
    ]
).sort_values(
    "TRANSACTION_COUNT",
    ascending=False
)


pattern_path = os.path.join(
    RESULTS_DIR,
    "tracegraph_final_pattern_summary.csv"
)

pattern_summary.to_csv(
    pattern_path,
    index=False
)


# ============================================================
# 20. FINAL VERIFICATION
# ============================================================

print("\n")
print("=" * 70)
print("STAGE 12D FINAL VERIFICATION")
print("=" * 70)

print(
    "\nTotal transactions:",
    len(tx)
)

print(
    "High-risk transactions:",
    len(high_risk)
)

print(
    "Suspicious cycles:",
    len(cycle_records)
)

print(
    "Investigation records:",
    len(investigations)
)


# ------------------------------------------------------------
# Verify expected cycles
# ------------------------------------------------------------

expected_cycles = [
    {55, 66, 67, 90, 53},
    {81, 82, 43, 20, 39}
]

detected_cycle_sets = [
    set(cycle["accounts"])
    for cycle in cycle_records
]


for expected in expected_cycles:

    found = expected in detected_cycle_sets

    print(
        "\nExpected cycle accounts:",
        sorted(expected)
    )

    print(
        "Detected:",
        found
    )


# ============================================================
# 21. DISPLAY CYCLE RESULTS
# ============================================================

print("\n" + "=" * 70)
print("DETECTED SUSPICIOUS CYCLES")
print("=" * 70)

if len(cycles_df) > 0:

    display(cycles_df)

else:

    print(
        "No suspicious cycles detected."
    )


# ============================================================
# 22. DISPLAY PATTERN SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PATTERN SUMMARY")
print("=" * 70)

if len(pattern_summary) > 0:

    display(pattern_summary)

else:

    print(
        "No additional suspicious patterns detected."
    )


# ============================================================
# 23. DISPLAY FINAL INVESTIGATION TABLE
# ============================================================

print("\n" + "=" * 70)
print("FINAL INVESTIGATION SUMMARY")
print("=" * 70)

display(
    summary_df[
        [
            "TX_ID",
            "SENDER_ACCOUNT_ID",
            "RECEIVER_ACCOUNT_ID",
            "TX_AMOUNT",
            "TIMESTAMP",
            "RISK_SCORE",
            "PATTERN_COUNT",
            "PATTERNS",
            "CYCLE_IDS"
        ]
    ]
)


# ============================================================
# 24. SAVE HIGH-RISK TRANSACTION TABLE
# ============================================================

high_risk_path = os.path.join(
    RESULTS_DIR,
    "tracegraph_suspicious_transactions.csv"
)

high_risk.to_csv(
    high_risk_path,
    index=False
)


# ============================================================
# 25. FINAL OUTPUT PATHS
# ============================================================

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(
    "\n1.",
    summary_path
)

print(
    "2.",
    investigation_json_path
)

print(
    "3.",
    cycles_path
)

print(
    "4.",
    pattern_path
)

print(
    "5.",
    high_risk_path
)

print("\n" + "=" * 70)
print("STAGE 12D COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 13
# STREAMLIT AML INVESTIGATION DASHBOARD
# ============================================================

import os
import textwrap

BASE_DIR = "/content/drive/MyDrive/TraceGraph"
RESULTS_DIR = os.path.join(BASE_DIR, "results")
DASHBOARD_DIR = os.path.join(BASE_DIR, "dashboard")

os.makedirs(DASHBOARD_DIR, exist_ok=True)

APP_PATH = os.path.join(DASHBOARD_DIR, "app.py")
REQ_PATH = os.path.join(DASHBOARD_DIR, "requirements.txt")


# ============================================================
# 1. CREATE STREAMLIT APPLICATION
# ============================================================

app_code = r'''
import os
import json
import pandas as pd
import numpy as np
import networkx as nx
import streamlit as st
import plotly.graph_objects as go


# ============================================================
# CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="TraceGraph AML",
    page_icon="TG",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/TraceGraph"

DATA_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

RESULTS_DIR = os.path.join(
    BASE_DIR,
    "results"
)


# ============================================================
# FILE PATHS
# ============================================================

SUMMARY_PATH = os.path.join(
    RESULTS_DIR,
    "tracegraph_final_investigation_summary.csv"
)

INVESTIGATION_PATH = os.path.join(
    RESULTS_DIR,
    "tracegraph_final_investigations.json"
)

CYCLES_PATH = os.path.join(
    RESULTS_DIR,
    "tracegraph_final_suspicious_cycles.csv"
)

PATTERN_PATH = os.path.join(
    RESULTS_DIR,
    "tracegraph_final_pattern_summary.csv"
)

SUSPICIOUS_TX_PATH = os.path.join(
    RESULTS_DIR,
    "tracegraph_suspicious_transactions.csv"
)

TRANSACTIONS_PATH = os.path.join(
    DATA_DIR,
    "transactions_validated.csv"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown(
    """
    <style>

    .main-title {
        font-size: 38px;
        font-weight: 700;
        margin-bottom: 0px;
    }

    .subtitle {
        font-size: 17px;
        opacity: 0.75;
        margin-bottom: 25px;
    }

    .risk-card {
        padding: 20px;
        border-radius: 12px;
        border: 1px solid rgba(128,128,128,0.25);
        text-align: center;
    }

    .section-title {
        font-size: 24px;
        font-weight: 650;
        margin-top: 20px;
    }

    .small-label {
        font-size: 13px;
        opacity: 0.65;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# LOAD DATA
# ============================================================

@st.cache_data
def load_data():

    summary = pd.read_csv(
        SUMMARY_PATH
    )

    cycles = pd.read_csv(
        CYCLES_PATH
    )

    patterns = pd.read_csv(
        PATTERN_PATH
    )

    suspicious = pd.read_csv(
        SUSPICIOUS_TX_PATH
    )

    transactions = pd.read_csv(
        TRANSACTIONS_PATH
    )

    with open(
        INVESTIGATION_PATH,
        "r"
    ) as f:

        investigations = json.load(f)

    return (
        summary,
        cycles,
        patterns,
        suspicious,
        transactions,
        investigations
    )


try:

    (
        summary_df,
        cycles_df,
        pattern_df,
        suspicious_df,
        transactions_df,
        investigations
    ) = load_data()

except Exception as e:

    st.error(
        "Unable to load TraceGraph result files."
    )

    st.exception(e)

    st.stop()


# ============================================================
# BUILD INVESTIGATION LOOKUP
# ============================================================

investigation_lookup = {
    int(item["TX_ID"]): item
    for item in investigations
}


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("TraceGraph")

st.sidebar.caption(
    "Temporal Graph Intelligence for AML Investigation"
)

st.sidebar.markdown("---")

page = st.sidebar.radio(
    "Navigation",
    [
        "Overview",
        "Suspicious Transactions",
        "Transaction Investigation",
        "Network Graph",
        "Cycle Detection",
        "Investigation Report"
    ]
)

st.sidebar.markdown("---")

st.sidebar.caption(
    "Dataset: PublicAMLSimData"
)

st.sidebar.caption(
    f"Transactions: {len(transactions_df):,}"
)

st.sidebar.caption(
    f"High-risk transactions: {len(suspicious_df):,}"
)


# ============================================================
# HEADER
# ============================================================

st.markdown(
    '<div class="main-title">TraceGraph AML</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Temporal Graph Intelligence for Anti-Money Laundering Investigation'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def make_network_figure(
    edges,
    selected_tx=None,
    cycle_edges=None
):

    G = nx.DiGraph()

    cycle_edges = cycle_edges or set()

    for edge in edges:

        sender = int(edge["sender"])
        receiver = int(edge["receiver"])

        G.add_edge(
            sender,
            receiver,
            tx_id=int(edge["TX_ID"]),
            amount=float(edge["amount"]),
            timestamp=int(edge["timestamp"]),
            risk=float(edge["risk_score"])
        )

    if len(G.nodes) == 0:

        return go.Figure()

    # Spring layout
    pos = nx.spring_layout(
        G,
        seed=42,
        k=1.4
    )

    edge_x = []
    edge_y = []

    normal_edge_x = []
    normal_edge_y = []

    cycle_edge_x = []
    cycle_edge_y = []

    for u, v, data in G.edges(data=True):

        x0, y0 = pos[u]
        x1, y1 = pos[v]

        pair = (
            int(data["tx_id"]),
            int(u),
            int(v)
        )

        if pair in cycle_edges:

            cycle_edge_x += [
                x0, x1, None
            ]

            cycle_edge_y += [
                y0, y1, None
            ]

        else:

            normal_edge_x += [
                x0, x1, None
            ]

            normal_edge_y += [
                y0, y1, None
            ]

    # Normal edges
    edge_trace = go.Scatter(
        x=normal_edge_x,
        y=normal_edge_y,
        mode="lines",
        line=dict(
            width=1.5
        ),
        hoverinfo="none"
    )

    # Cycle edges
    cycle_trace = go.Scatter(
        x=cycle_edge_x,
        y=cycle_edge_y,
        mode="lines",
        line=dict(
            width=4
        ),
        hoverinfo="none"
    )

    # Nodes
    node_x = []
    node_y = []
    node_text = []

    for node in G.nodes():

        x, y = pos[node]

        node_x.append(x)
        node_y.append(y)

        node_text.append(
            f"Account {node}"
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=[
            str(node)
            for node in G.nodes()
        ],
        textposition="top center",
        marker=dict(
            size=22
        ),
        hovertext=node_text,
        hoverinfo="text"
    )

    fig = go.Figure(
        data=[
            edge_trace,
            cycle_trace,
            node_trace
        ]
    )

    fig.update_layout(
        showlegend=False,
        height=600,
        margin=dict(
            l=10,
            r=10,
            t=10,
            b=10
        ),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            showticklabels=False
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            showticklabels=False
        )
    )

    return fig


def flatten_trace(trace):

    rows = []

    for item in trace:

        rows.append({

            "TX_ID":
                item["TX_ID"],

            "Sender":
                item["sender"],

            "Receiver":
                item["receiver"],

            "Amount":
                item["amount"],

            "Timestamp":
                item["timestamp"],

            "Risk Score":
                item["risk_score"]
        })

    return pd.DataFrame(rows)


# ============================================================
# PAGE 1 — OVERVIEW
# ============================================================

if page == "Overview":

    st.markdown(
        '<div class="section-title">'
        'Risk Overview'
        '</div>',
        unsafe_allow_html=True
    )

    total_transactions = len(
        transactions_df
    )

    high_risk_count = len(
        suspicious_df
    )

    normal_count = (
        total_transactions
        - high_risk_count
    )

    cycle_count = len(
        cycles_df
    )

    pattern_count = len(
        pattern_df
    )

    col1, col2, col3, col4 = st.columns(4)

    with col1:

        st.metric(
            "Total Transactions",
            f"{total_transactions:,}"
        )

    with col2:

        st.metric(
            "High-Risk Transactions",
            f"{high_risk_count:,}"
        )

    with col3:

        st.metric(
            "Detected Cycles",
            f"{cycle_count:,}"
        )

    with col4:

        st.metric(
            "Pattern Types",
            f"{pattern_count:,}"
        )

    st.markdown("---")

    # --------------------------------------------------------
    # Risk distribution
    # --------------------------------------------------------

    col1, col2 = st.columns(2)

    with col1:

        st.subheader(
            "Risk Distribution"
        )

        risk_counts = pd.DataFrame({

            "Category": [
                "Normal",
                "High Risk"
            ],

            "Transactions": [
                normal_count,
                high_risk_count
            ]
        })

        fig = go.Figure(
            data=[
                go.Bar(
                    x=risk_counts["Category"],
                    y=risk_counts["Transactions"]
                )
            ]
        )

        fig.update_layout(
            height=400,
            yaxis_title="Transactions",
            xaxis_title=""
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    with col2:

        st.subheader(
            "Detected AML Patterns"
        )

        fig = go.Figure(
            data=[
                go.Bar(
                    x=pattern_df["TRANSACTION_COUNT"],
                    y=pattern_df["PATTERN"],
                    orientation="h"
                )
            ]
        )

        fig.update_layout(
            height=400,
            xaxis_title="Transactions",
            yaxis_title=""
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    # --------------------------------------------------------
    # Highest risk transactions
    # --------------------------------------------------------

    st.subheader(
        "Highest-Risk Transactions"
    )

    display_cols = [
        "TX_ID",
        "SENDER_ACCOUNT_ID",
        "RECEIVER_ACCOUNT_ID",
        "TX_AMOUNT",
        "TIMESTAMP",
        "RISK_SCORE",
        "PATTERNS"
    ]

    available_cols = [
        c
        for c in display_cols
        if c in summary_df.columns
    ]

    st.dataframe(
        summary_df.sort_values(
            "RISK_SCORE",
            ascending=False
        )[available_cols],
        use_container_width=True,
        hide_index=True
    )

    st.info(
        "Risk scores indicate model-generated transaction risk. "
        "They are investigation signals and do not by themselves "
        "establish that a transaction is illicit."
    )


# ============================================================
# PAGE 2 — SUSPICIOUS TRANSACTIONS
# ============================================================

elif page == "Suspicious Transactions":

    st.markdown(
        '<div class="section-title">'
        'Suspicious Transactions'
        '</div>',
        unsafe_allow_html=True
    )

    st.write(
        "Transactions whose model risk score exceeds the "
        "investigation threshold."
    )

    # --------------------------------------------------------
    # Filters
    # --------------------------------------------------------

    col1, col2 = st.columns(2)

    with col1:

        min_risk = st.slider(
            "Minimum Risk Score",
            min_value=0.90,
            max_value=1.00,
            value=0.90,
            step=0.001
        )

    with col2:

        pattern_options = sorted(
            set(
                pattern.strip()
                for patterns in summary_df["PATTERNS"].dropna()
                for pattern in patterns.split("|")
                if pattern.strip()
            )
        )

        selected_pattern = st.selectbox(
            "Pattern Filter",
            ["All"] + pattern_options
        )

    filtered = summary_df[
        summary_df["RISK_SCORE"] >= min_risk
    ].copy()

    if selected_pattern != "All":

        filtered = filtered[
            filtered["PATTERNS"].str.contains(
                selected_pattern,
                na=False
            )
        ]

    st.metric(
        "Matching Transactions",
        len(filtered)
    )

    st.dataframe(
        filtered,
        use_container_width=True,
        hide_index=True
    )


# ============================================================
# PAGE 3 — TRANSACTION INVESTIGATION
# ============================================================

elif page == "Transaction Investigation":

    st.markdown(
        '<div class="section-title">'
        'Transaction Investigation'
        '</div>',
        unsafe_allow_html=True
    )

    tx_ids = sorted(
        suspicious_df["TX_ID"].astype(int).tolist()
    )

    selected_tx = st.selectbox(
        "Select Transaction ID",
        tx_ids
    )

    record = investigation_lookup[
        int(selected_tx)
    ]

    # --------------------------------------------------------
    # Basic information
    # --------------------------------------------------------

    st.subheader(
        "Transaction Details"
    )

    col1, col2, col3, col4, col5 = st.columns(5)

    with col1:

        st.metric(
            "Transaction ID",
            record["TX_ID"]
        )

    with col2:

        st.metric(
            "Sender",
            record["SENDER_ACCOUNT_ID"]
        )

    with col3:

        st.metric(
            "Receiver",
            record["RECEIVER_ACCOUNT_ID"]
        )

    with col4:

        st.metric(
            "Amount",
            f'{record["TX_AMOUNT"]:.2f}'
        )

    with col5:

        st.metric(
            "Risk Score",
            f'{record["RISK_SCORE"]:.6f}'
        )

    st.markdown("---")

    # --------------------------------------------------------
    # Patterns
    # --------------------------------------------------------

    st.subheader(
        "Detected Patterns"
    )

    if record["PATTERNS"]:

        for pattern in record["PATTERNS"]:

            st.write(
                f"• {pattern}"
            )

    else:

        st.write(
            "No additional network pattern detected."
        )

    # --------------------------------------------------------
    # Fan-in
    # --------------------------------------------------------

    col1, col2 = st.columns(2)

    with col1:

        st.subheader(
            "Risk-Aware Fan-In"
        )

        st.write(
            "Suspicious senders connected to the "
            "transaction receiver within the pattern window."
        )

        st.write(
            f'Account count: {record["FAN_IN_ACCOUNT_COUNT"]}'
        )

        if record["FAN_IN_ACCOUNTS"]:

            st.write(
                record["FAN_IN_ACCOUNTS"]
            )

    with col2:

        st.subheader(
            "Risk-Aware Fan-Out"
        )

        st.write(
            "Suspicious receivers connected to the "
            "transaction sender within the pattern window."
        )

        st.write(
            f'Account count: {record["FAN_OUT_ACCOUNT_COUNT"]}'
        )

        if record["FAN_OUT_ACCOUNTS"]:

            st.write(
                record["FAN_OUT_ACCOUNTS"]
            )

    # --------------------------------------------------------
    # Rapid hops
    # --------------------------------------------------------

    if record["RAPID_HOPS"]:

        st.subheader(
            "Rapid-Hop Evidence"
        )

        st.dataframe(
            pd.DataFrame(
                record["RAPID_HOPS"]
            ),
            use_container_width=True,
            hide_index=True
        )

    # --------------------------------------------------------
    # Cycle membership
    # --------------------------------------------------------

    if record["CYCLE_IDS"]:

        st.subheader(
            "Cycle Membership"
        )

        for cycle_id in record["CYCLE_IDS"]:

            cycle = cycles_df[
                cycles_df["cycle_id"]
                == cycle_id
            ]

            if len(cycle) > 0:

                st.dataframe(
                    cycle,
                    use_container_width=True,
                    hide_index=True
                )

    # --------------------------------------------------------
    # Reverse and forward traces
    # --------------------------------------------------------

    col1, col2 = st.columns(2)

    with col1:

        st.subheader(
            "Reverse Trace"
        )

        reverse_df = flatten_trace(
            record["REVERSE_TRACE"]
        )

        if len(reverse_df) > 0:

            st.dataframe(
                reverse_df,
                use_container_width=True,
                hide_index=True
            )

        else:

            st.info(
                "No suspicious upstream trace found."
            )

    with col2:

        st.subheader(
            "Forward Trace"
        )

        forward_df = flatten_trace(
            record["FORWARD_TRACE"]
        )

        if len(forward_df) > 0:

            st.dataframe(
                forward_df,
                use_container_width=True,
                hide_index=True
            )

        else:

            st.info(
                "No suspicious downstream trace found."
            )


# ============================================================
# PAGE 4 — NETWORK GRAPH
# ============================================================

elif page == "Network Graph":

    st.markdown(
        '<div class="section-title">'
        'Suspicious Network Graph'
        '</div>',
        unsafe_allow_html=True
    )

    st.write(
        "This visualization uses only the high-risk "
        "transactions identified during investigation."
    )

    # --------------------------------------------------------
    # Select transaction / network
    # --------------------------------------------------------

    tx_ids = sorted(
        suspicious_df["TX_ID"].astype(int).tolist()
    )

    selected_tx = st.selectbox(
        "Select suspicious transaction",
        tx_ids
    )

    record = investigation_lookup[
        int(selected_tx)
    ]

    # Combine reverse and forward evidence.
    all_edges = (
        record["REVERSE_TRACE"]
        + record["FORWARD_TRACE"]
    )

    # Remove duplicate transactions.
    unique_edges = {}

    for edge in all_edges:

        unique_edges[
            int(edge["TX_ID"])
        ] = edge

    all_edges = list(
        unique_edges.values()
    )

    # Add cycle edges for this transaction.
    cycle_edges = set()

    for cycle_id in record["CYCLE_IDS"]:

        cycle_rows = cycles_df[
            cycles_df["cycle_id"]
            == cycle_id
        ]

        if len(cycle_rows) == 0:
            continue

        row = cycle_rows.iloc[0]

        tx_id_list = [
            int(x)
            for x in str(
                row["transaction_ids"]
            ).split(",")
        ]

        account_list = [
            int(x)
            for x in str(
                row["account_path"]
            ).split(" -> ")
        ]

        for i, cycle_tx in enumerate(
            tx_id_list
        ):

            if i >= len(account_list) - 1:
                break

            cycle_edges.add(
                (
                    cycle_tx,
                    account_list[i],
                    account_list[i + 1]
                )
            )

    if len(all_edges) > 0:

        fig = make_network_figure(
            all_edges,
            selected_tx,
            cycle_edges
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

    else:

        st.info(
            "No network edges available for this transaction."
        )

    st.subheader(
        "Network Evidence"
    )

    network_table = pd.DataFrame(
        all_edges
    )

    if len(network_table) > 0:

        st.dataframe(
            network_table,
            use_container_width=True,
            hide_index=True
        )


# ============================================================
# PAGE 5 — CYCLE DETECTION
# ============================================================

elif page == "Cycle Detection":

    st.markdown(
        '<div class="section-title">'
        'Suspicious Temporal Cycles'
        '</div>',
        unsafe_allow_html=True
    )

    st.write(
        "Cycles detected in the high-risk transaction "
        "subgraph. Cycle membership is an investigation "
        "signal rather than a determination of illicit activity."
    )

    st.metric(
        "Detected Cycles",
        len(cycles_df)
    )

    for _, cycle in cycles_df.iterrows():

        st.markdown("---")

        st.subheader(
            f'Cycle {int(cycle["cycle_id"])}'
        )

        st.write(
            f'**Account path:** '
            f'{cycle["account_path"]}'
        )

        col1, col2, col3 = st.columns(3)

        with col1:

            st.metric(
                "Cycle Length",
                int(cycle["cycle_length"])
            )

        with col2:

            st.metric(
                "Time Span",
                int(cycle["time_span"])
            )

        with col3:

            st.metric(
                "Minimum Risk",
                f'{cycle["min_risk_score"]:.6f}'
            )

        st.write(
            f'**Transaction IDs:** '
            f'{cycle["transaction_ids"]}'
        )

        st.write(
            f'**Timestamps:** '
            f'{cycle["timestamps"]}'
        )

        st.write(
            f'**Amounts:** '
            f'{cycle["amounts"]}'
        )


# ============================================================
# PAGE 6 — INVESTIGATION REPORT
# ============================================================

elif page == "Investigation Report":

    st.markdown(
        '<div class="section-title">'
        'Investigation Report'
        '</div>',
        unsafe_allow_html=True
    )

    tx_ids = sorted(
        suspicious_df["TX_ID"].astype(int).tolist()
    )

    selected_tx = st.selectbox(
        "Select Transaction",
        tx_ids
    )

    record = investigation_lookup[
        int(selected_tx)
    ]

    st.subheader(
        "Case Summary"
    )

    st.markdown(
        f"""
        **Transaction ID:** {record["TX_ID"]}

        **Sender Account:** {record["SENDER_ACCOUNT_ID"]}

        **Receiver Account:** {record["RECEIVER_ACCOUNT_ID"]}

        **Transaction Amount:** {record["TX_AMOUNT"]:.2f}

        **Timestamp:** {record["TIMESTAMP"]}

        **Model Risk Score:** {record["RISK_SCORE"]:.6f}

        **Investigation Status:** {record["INVESTIGATION_STATUS"]}
        """
    )

    st.markdown("---")

    st.subheader(
        "Observed Network Evidence"
    )

    if record["PATTERNS"]:

        for pattern in record["PATTERNS"]:

            st.markdown(
                f"- **{pattern}**"
            )

    else:

        st.write(
            "No additional network pattern was detected."
        )

    # --------------------------------------------------------
    # Cycle evidence
    # --------------------------------------------------------

    if record["CYCLE_IDS"]:

        st.subheader(
            "Cycle Evidence"
        )

        for cycle_id in record["CYCLE_IDS"]:

            rows = cycles_df[
                cycles_df["cycle_id"]
                == cycle_id
            ]

            for _, cycle in rows.iterrows():

                st.markdown(
                    f"""
                    - Account path: `{cycle["account_path"]}`
                    - Transactions: `{cycle["transaction_ids"]}`
                    - Time span: `{cycle["time_span"]}`
                    - Minimum risk score: `{cycle["min_risk_score"]:.6f}`
                    """
                )

    # --------------------------------------------------------
    # Trace evidence
    # --------------------------------------------------------

    st.subheader(
        "Reverse Trace Evidence"
    )

    reverse_df = flatten_trace(
        record["REVERSE_TRACE"]
    )

    if len(reverse_df) > 0:

        st.dataframe(
            reverse_df,
            use_container_width=True,
            hide_index=True
        )

    else:

        st.write(
            "No suspicious reverse trace identified."
        )

    st.subheader(
        "Forward Trace Evidence"
    )

    forward_df = flatten_trace(
        record["FORWARD_TRACE"]
    )

    if len(forward_df) > 0:

        st.dataframe(
            forward_df,
            use_container_width=True,
            hide_index=True
        )

    else:

        st.write(
            "No suspicious forward trace identified."
        )

    st.markdown("---")

    st.warning(
        "This report summarizes model and graph-derived "
        "evidence for investigation. It does not establish "
        "that a person, account, or transaction committed "
        "money laundering. Further review and independent "
        "evidence are required."
    )


# ============================================================
# FOOTER
# ============================================================

st.markdown("---")

st.caption(
    "TraceGraph — Temporal Graph Intelligence for AML Detection"
)

st.caption(
    "Research prototype | PublicAMLSimData"
)
'''


# ============================================================
# 2. WRITE APP.PY
# ============================================================

with open(
    APP_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(app_code)


# ============================================================
# 3. CREATE REQUIREMENTS
# ============================================================

requirements = """streamlit
pandas
numpy
networkx
plotly
"""

with open(
    REQ_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(requirements)


# ============================================================
# 4. INSTALL STREAMLIT DEPENDENCIES
# ============================================================

print("=" * 70)
print("TRACEGRAPH STAGE 13")
print("STREAMLIT DASHBOARD SETUP")
print("=" * 70)

print("\nInstalling dashboard dependencies...")

!pip -q install streamlit plotly networkx


# ============================================================
# 5. VERIFY RESULT FILES
# ============================================================

print("\n" + "-" * 70)
print("VERIFYING RESULT FILES")
print("-" * 70)

required_files = [
    "tracegraph_final_investigation_summary.csv",
    "tracegraph_final_investigations.json",
    "tracegraph_final_suspicious_cycles.csv",
    "tracegraph_final_pattern_summary.csv",
    "tracegraph_suspicious_transactions.csv"
]

all_present = True

for filename in required_files:

    path = os.path.join(
        RESULTS_DIR,
        filename
    )

    exists = os.path.exists(path)

    print(
        f"{'OK' if exists else 'MISSING'}  {filename}"
    )

    if not exists:
        all_present = False


# ============================================================
# 6. VERIFY DASHBOARD FILE
# ============================================================

print("\n" + "-" * 70)
print("DASHBOARD FILES")
print("-" * 70)

print(
    "app.py:",
    APP_PATH
)

print(
    "requirements.txt:",
    REQ_PATH
)

print(
    "\napp.py exists:",
    os.path.exists(APP_PATH)
)

print(
    "requirements.txt exists:",
    os.path.exists(REQ_PATH)
)


# ============================================================
# 7. BASIC DATA VALIDATION
# ============================================================

if all_present:

    import json

    summary_test = pd.read_csv(
        os.path.join(
            RESULTS_DIR,
            "tracegraph_final_investigation_summary.csv"
        )
    )

    cycles_test = pd.read_csv(
        os.path.join(
            RESULTS_DIR,
            "tracegraph_final_suspicious_cycles.csv"
        )
    )

    patterns_test = pd.read_csv(
        os.path.join(
            RESULTS_DIR,
            "tracegraph_final_pattern_summary.csv"
        )
    )

    with open(
        os.path.join(
            RESULTS_DIR,
            "tracegraph_final_investigations.json"
        ),
        "r"
    ) as f:

        investigation_test = json.load(f)

    print("\n" + "-" * 70)
    print("DATA VALIDATION")
    print("-" * 70)

    print(
        "Investigation records:",
        len(investigation_test)
    )

    print(
        "Suspicious cycles:",
        len(cycles_test)
    )

    print(
        "Pattern types:",
        len(patterns_test)
    )

    print(
        "Summary rows:",
        len(summary_test)
    )


# ============================================================
# 8. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)

if all_present:

    print(
        "STAGE 13 DASHBOARD CREATED SUCCESSFULLY"
    )

else:

    print(
        "STAGE 13 CREATED, BUT SOME RESULT FILES ARE MISSING"
    )

print("=" * 70)

print(
    "\nDashboard location:"
)

print(APP_PATH)

print(
    "\nNext command to launch the dashboard:"
)

print(
    f"streamlit run {APP_PATH} --server.port 8501"
)

print(
    "\nAfter this cell finishes, send me the output."
)

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 13A
# LAUNCH STREAMLIT DASHBOARD IN GOOGLE COLAB
# ============================================================

import subprocess
import time
import os

APP_PATH = "/content/drive/MyDrive/TraceGraph/dashboard/app.py"
PORT = 8501

print("=" * 70)
print("TRACEGRAPH DASHBOARD")
print("=" * 70)

# ------------------------------------------------------------
# Stop any previous Streamlit process
# ------------------------------------------------------------

!pkill -f "streamlit run" 2>/dev/null || true

time.sleep(2)

# ------------------------------------------------------------
# Launch Streamlit
# ------------------------------------------------------------

log_path = "/content/tracegraph_streamlit.log"

log_file = open(
    log_path,
    "w"
)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port",
        str(PORT),
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
        "--browser.gatherUsageStats",
        "false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("\nStreamlit process started.")
print("PID:", process.pid)

# ------------------------------------------------------------
# Give Streamlit time to start
# ------------------------------------------------------------

time.sleep(8)

# ------------------------------------------------------------
# Check process
# ------------------------------------------------------------

if process.poll() is None:

    print("\nStreamlit is running successfully.")

else:

    print("\nStreamlit failed to start.")

    log_file.close()

    print("\n--- STREAMLIT LOG ---")

    with open(
        log_path,
        "r"
    ) as f:

        print(
            f.read()[-5000:]
        )

    raise RuntimeError(
        "Streamlit process exited."
    )


# ------------------------------------------------------------
# Create Colab proxy URL
# ------------------------------------------------------------

from google.colab import output

try:

    public_url = output.eval_js(
        f"google.colab.kernel.proxyPort({PORT})"
    )

    print("\n" + "=" * 70)
    print("TRACEGRAPH DASHBOARD READY")
    print("=" * 70)

    print("\nOpen this URL in your browser:\n")

    print(public_url)

    print("\n" + "=" * 70)

except Exception as e:

    print(
        "\nCould not automatically generate the Colab URL."
    )

    print(
        "The Streamlit server is still running on port",
        PORT
    )

    print("\nError:", e)

print(
    "\nDashboard log:",
    log_path
)

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 13B
# DASHBOARD HEALTH CHECK
# ============================================================

import os
import time
import requests
import subprocess

PORT = 8501
LOG_PATH = "/content/tracegraph_streamlit.log"

print("=" * 70)
print("TRACEGRAPH DASHBOARD HEALTH CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check Streamlit process
# ------------------------------------------------------------

print("\n[1] Streamlit process")

result = subprocess.run(
    ["pgrep", "-af", "streamlit"],
    capture_output=True,
    text=True
)

if result.stdout.strip():

    print("Streamlit process detected:")
    print(result.stdout)

else:

    print("WARNING: No Streamlit process detected.")


# ------------------------------------------------------------
# 2. Check localhost
# ------------------------------------------------------------

print("\n[2] Local server")

try:

    response = requests.get(
        f"http://127.0.0.1:{PORT}",
        timeout=10
    )

    print(
        "HTTP status:",
        response.status_code
    )

    print(
        "Response size:",
        len(response.text),
        "bytes"
    )

    if response.status_code == 200:

        print(
            "Local Streamlit server is responding successfully."
        )

    else:

        print(
            "Streamlit responded, but with status:",
            response.status_code
        )

except Exception as e:

    print(
        "Could not connect to local Streamlit server."
    )

    print(
        "Error:",
        repr(e)
    )


# ------------------------------------------------------------
# 3. Read Streamlit log
# ------------------------------------------------------------

print("\n[3] Streamlit log")

if os.path.exists(LOG_PATH):

    with open(
        LOG_PATH,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:

        log = f.read()

    if log.strip():

        print(log[-8000:])

    else:

        print(
            "Log file exists but is empty."
        )

else:

    print(
        "Log file not found."
    )


# ------------------------------------------------------------
# 4. Check dashboard files
# ------------------------------------------------------------

print("\n[4] Dashboard files")

APP_PATH = (
    "/content/drive/MyDrive/"
    "TraceGraph/dashboard/app.py"
)

print(
    "app.py exists:",
    os.path.exists(APP_PATH)
)

if os.path.exists(APP_PATH):

    print(
        "app.py size:",
        os.path.getsize(APP_PATH),
        "bytes"
    )


# ------------------------------------------------------------
# 5. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HEALTH CHECK COMPLETE")
print("=" * 70)

print(
    "\nIf HTTP status is 200 and there are no Python exceptions "
    "in the log, refresh the Streamlit browser tab once."
)

In [ ]:
# ============================================================
# TRACEGRAPH — STAGE 13C
# FIX STREAMLIT + COLAB WEBSOCKET
# ============================================================

import subprocess
import time
import os

APP_PATH = "/content/drive/MyDrive/TraceGraph/dashboard/app.py"
PORT = 8501
LOG_PATH = "/content/tracegraph_streamlit.log"

print("=" * 70)
print("TRACEGRAPH DASHBOARD — WEBSOCKET FIX")
print("=" * 70)

# ------------------------------------------------------------
# 1. Stop previous Streamlit instance
# ------------------------------------------------------------

print("\nStopping previous Streamlit process...")

subprocess.run(
    ["pkill", "-f", "streamlit run"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

# ------------------------------------------------------------
# 2. Start Streamlit with Colab-compatible settings
# ------------------------------------------------------------

print("Starting Streamlit with WebSocket compatibility settings...")

log_file = open(
    LOG_PATH,
    "w"
)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,

        "--server.port",
        str(PORT),

        "--server.address",
        "0.0.0.0",

        "--server.headless",
        "true",

        # Important for Google Colab proxy
        "--server.enableCORS",
        "false",

        "--server.enableXsrfProtection",
        "false",

        "--browser.gatherUsageStats",
        "false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print(
    "Streamlit PID:",
    process.pid
)

# ------------------------------------------------------------
# 3. Wait for startup
# ------------------------------------------------------------

time.sleep(8)

# ------------------------------------------------------------
# 4. Check process
# ------------------------------------------------------------

if process.poll() is not None:

    log_file.close()

    print("\nERROR: Streamlit stopped unexpectedly.\n")

    if os.path.exists(LOG_PATH):

        with open(
            LOG_PATH,
            "r",
            encoding="utf-8",
            errors="replace"
        ) as f:

            print(f.read()[-10000:])

    raise RuntimeError(
        "Streamlit failed to start."
    )


print(
    "Streamlit process is running."
)

# ------------------------------------------------------------
# 5. Test local HTTP server
# ------------------------------------------------------------

import requests

try:

    response = requests.get(
        f"http://127.0.0.1:{PORT}",
        timeout=10
    )

    print(
        "Local HTTP status:",
        response.status_code
    )

except Exception as e:

    print(
        "Local HTTP test failed:",
        repr(e)
    )


# ------------------------------------------------------------
# 6. Generate Colab proxy URL
# ------------------------------------------------------------

from google.colab import output

public_url = output.eval_js(
    f"google.colab.kernel.proxyPort({PORT})"
)

print("\n" + "=" * 70)
print("TRACEGRAPH DASHBOARD READY")
print("=" * 70)

print("\nOpen this URL:")
print(public_url)

print("\n" + "=" * 70)

print(
    "\nIMPORTANT:"
)

print(
    "Use the NEW URL above rather than the previous Streamlit tab."
)

print(
    "\nDashboard log:",
    LOG_PATH
)

In [ ]:
# ============================================================
# TRACEGRAPH STAGE 13D — SYNTAX DIAGNOSTIC
# ============================================================

from pathlib import Path
import ast
import traceback

APP_PATH = Path("/content/drive/MyDrive/TraceGraph/dashboard/app.py")

print("=" * 75)
print("TRACEGRAPH STAGE 13D — DASHBOARD SYNTAX DIAGNOSTIC")
print("=" * 75)

if not APP_PATH.exists():
    raise FileNotFoundError(f"Dashboard not found: {APP_PATH}")

lines = APP_PATH.read_text(encoding="utf-8").splitlines()

print(f"\nApp: {APP_PATH}")
print(f"Total lines: {len(lines)}")
print(f"Total characters: {APP_PATH.stat().st_size:,}")

# ------------------------------------------------------------
# AST CHECK
# ------------------------------------------------------------

print("\n" + "-" * 75)
print("PYTHON SYNTAX CHECK")
print("-" * 75)

try:
    ast.parse("\n".join(lines))
    print("SYNTAX OK")
except SyntaxError as e:
    print("SYNTAX ERROR FOUND")
    print(f"Line     : {e.lineno}")
    print(f"Column   : {e.offset}")
    print(f"Message  : {e.msg}")
    print(f"Text     : {e.text.strip() if e.text else 'N/A'}")

    error_line = e.lineno or 1279

    start = max(0, error_line - 45)
    end = min(len(lines), error_line + 25)

    print("\n" + "=" * 75)
    print(f"SOURCE CONTEXT — LINES {start + 1} TO {end}")
    print("=" * 75)

    for i in range(start, end):
        marker = " >>> " if i + 1 == error_line else "     "
        print(f"{marker}{i + 1:4d} | {lines[i]}")

# ------------------------------------------------------------
# FIND PAGE CONTROL FLOW
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PAGE CONTROL-FLOW LINES")
print("=" * 75)

for i, line in enumerate(lines, start=1):
    stripped = line.strip()

    if (
        stripped.startswith("if page ")
        or stripped.startswith("elif page ")
        or stripped.startswith("else:")
    ):
        print(f"{i:4d} | indent={len(line) - len(line.lstrip())} | {stripped}")

# ------------------------------------------------------------
# FIND INVESTIGATION WORKSPACE
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("INVESTIGATION WORKSPACE OCCURRENCES")
print("=" * 75)

for i, line in enumerate(lines, start=1):
    if "Investigation Workspace" in line:
        print(f"{i:4d} | {repr(line)}")

print("\n" + "=" * 75)
print("END DIAGNOSTIC")
print("=" * 75)

In [ ]:
# ============================================================================
# TRACEGRAPH STAGE 13D — DASHBOARD SYNTAX REPAIR
# ============================================================================
# Repairs the page-routing issue without changing dashboard functionality.
# ============================================================================

from pathlib import Path
import ast
import shutil
import subprocess
import sys

APP_PATH = Path("/content/drive/MyDrive/TraceGraph/dashboard/app.py")
BACKUP_PATH = Path("/content/drive/MyDrive/TraceGraph/dashboard/app_stage13d_broken_backup.py")

print("=" * 75)
print("TRACEGRAPH STAGE 13D — DASHBOARD SYNTAX REPAIR")
print("=" * 75)

if not APP_PATH.exists():
    raise FileNotFoundError(f"Dashboard not found: {APP_PATH}")

# --------------------------------------------------------------------------
# 1. BACKUP CURRENT BROKEN VERSION
# --------------------------------------------------------------------------

print("\n" + "-" * 75)
print("BACKING UP CURRENT VERSION")
print("-" * 75)

shutil.copy2(APP_PATH, BACKUP_PATH)

print(f"Broken version backed up to:")
print(BACKUP_PATH)

# --------------------------------------------------------------------------
# 2. READ APP
# --------------------------------------------------------------------------

source = APP_PATH.read_text(encoding="utf-8")

print("\nOriginal size:")
print(f"  {len(source):,} characters")

# --------------------------------------------------------------------------
# 3. EXACT PAGE ROUTING REPAIR
# --------------------------------------------------------------------------
#
# The following branches occur after helper functions and therefore cannot
# legally use "elif". They must become independent top-level "if" blocks.
#
# We deliberately replace only the exact page-routing statements.
# --------------------------------------------------------------------------

replacements = {
    'elif page == "Investigation Workspace":':
        'if page == "Investigation Workspace":',

    'elif page == "Network Intelligence":':
        'if page == "Network Intelligence":',

    'elif page == "Cycle Intelligence":':
        'if page == "Cycle Intelligence":',

    'elif page == "Case Report":':
        'if page == "Case Report":',
}

replacement_count = {}

for old, new in replacements.items():

    count = source.count(old)
    replacement_count[old] = count

    if count != 1:
        raise RuntimeError(
            f"Expected exactly one occurrence of:\n{old}\n"
            f"but found {count}"
        )

    source = source.replace(old, new)

# --------------------------------------------------------------------------
# 4. WRITE REPAIRED APP
# --------------------------------------------------------------------------

APP_PATH.write_text(source, encoding="utf-8")

print("\n" + "-" * 75)
print("PAGE ROUTING REPAIRED")
print("-" * 75)

for old, new in replacements.items():
    print(f"OK  {old}")
    print(f" -> {new}")

# --------------------------------------------------------------------------
# 5. AST / PYTHON SYNTAX VALIDATION
# --------------------------------------------------------------------------

print("\n" + "-" * 75)
print("PYTHON SYNTAX VALIDATION")
print("-" * 75)

repaired_source = APP_PATH.read_text(encoding="utf-8")

try:

    ast.parse(repaired_source)

    print("SUCCESS — app.py is syntactically valid.")

except SyntaxError as e:

    print("SYNTAX ERROR STILL EXISTS")
    print(f"Line   : {e.lineno}")
    print(f"Column : {e.offset}")
    print(f"Error  : {e.msg}")

    if e.text:
        print(f"Text   : {e.text.strip()}")

    raise RuntimeError(
        "Dashboard syntax validation failed after repair."
    )

# --------------------------------------------------------------------------
# 6. COMPILE VALIDATION
# --------------------------------------------------------------------------

print("\n" + "-" * 75)
print("PYTHON COMPILE VALIDATION")
print("-" * 75)

try:

    compile(
        repaired_source,
        str(APP_PATH),
        "exec"
    )

    print("SUCCESS — app.py compiles successfully.")

except Exception as e:

    print("COMPILE ERROR")
    print(e)
    raise

# --------------------------------------------------------------------------
# 7. VERIFY IMPORTANT PAGE ROUTES
# --------------------------------------------------------------------------

print("\n" + "-" * 75)
print("VERIFYING PAGE ROUTES")
print("-" * 75)

required_routes = [
    'if page == "Executive Overview":',
    'elif page == "Alert Explorer":',
    'if page == "Investigation Workspace":',
    'if page == "Network Intelligence":',
    'if page == "Cycle Intelligence":',
    'if page == "Case Report":',
]

for route in required_routes:

    if route in repaired_source:
        print(f"OK  {route}")
    else:
        print(f"MISSING  {route}")

# --------------------------------------------------------------------------
# 8. SHOW PAGE ROUTING SUMMARY
# --------------------------------------------------------------------------

print("\n" + "-" * 75)
print("PAGE ROUTING SUMMARY")
print("-" * 75)

for i, line in enumerate(
    repaired_source.splitlines(),
    start=1
):

    stripped = line.strip()

    if (
        stripped.startswith("if page ")
        or stripped.startswith("elif page ")
    ):

        print(
            f"{i:4d} | "
            f"indent={len(line) - len(line.lstrip())} | "
            f"{stripped}"
        )

# --------------------------------------------------------------------------
# 9. FINAL FILE INFORMATION
# --------------------------------------------------------------------------

print("\n" + "=" * 75)
print("STAGE 13D SYNTAX REPAIR COMPLETE")
print("=" * 75)

print("\nDashboard:")
print(APP_PATH)

print("\nBackup:")
print(BACKUP_PATH)

print("\nSize:")
print(f"{APP_PATH.stat().st_size:,} bytes")

print("\nStatus:")
print("  [OK] Syntax valid")
print("  [OK] Compile valid")
print("  [OK] Page routing repaired")

print("\n" + "=" * 75)
print("DO NOT START STREAMLIT YET")
print("=" * 75)

print("""
Next step:
Run the dashboard launch cell after this repair completes.

The repair only changes the invalid page-routing `elif` statements
into independent `if` statements. No dataset, model, investigation,
cycle-detection, or dashboard artifact was modified.
""")

In [ ]:
# ============================================================================
# TRACEGRAPH STAGE 13D — PROFESSIONAL DASHBOARD LAUNCH
# ============================================================================

import os
import subprocess
import time
import signal
import requests

APP_PATH = "/content/drive/MyDrive/TraceGraph/dashboard/app.py"
LOG_PATH = "/content/tracegraph_streamlit.log"
PORT = 8501

print("=" * 75)
print("TRACEGRAPH STAGE 13D — PROFESSIONAL AML DASHBOARD")
print("=" * 75)

# --------------------------------------------------------------------------
# 1. VERIFY APP
# --------------------------------------------------------------------------

if not os.path.exists(APP_PATH):
    raise FileNotFoundError(APP_PATH)

print("\nDashboard:")
print(APP_PATH)

# --------------------------------------------------------------------------
# 2. STOP PREVIOUS STREAMLIT PROCESSES
# --------------------------------------------------------------------------

print("\n" + "-" * 75)
print("STOPPING PREVIOUS STREAMLIT PROCESSES")
print("-" * 75)

subprocess.run(
    ["pkill", "-f", "streamlit"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

print("Previous Streamlit processes stopped.")

# --------------------------------------------------------------------------
# 3. START PROFESSIONAL DASHBOARD
# --------------------------------------------------------------------------

print("\n" + "-" * 75)
print("STARTING PROFESSIONAL DASHBOARD")
print("-" * 75)

log_file = open(LOG_PATH, "w")

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        APP_PATH,
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print(f"Streamlit PID: {process.pid}")

# --------------------------------------------------------------------------
# 4. WAIT FOR SERVER
# --------------------------------------------------------------------------

print("\nWaiting for Streamlit server...")

server_ready = False

for attempt in range(30):

    time.sleep(1)

    try:
        response = requests.get(
            f"http://127.0.0.1:{PORT}",
            timeout=2
        )

        if response.status_code == 200:
            server_ready = True
            print(
                f"Server ready after approximately {attempt + 1} seconds."
            )
            break

    except Exception:
        pass

if not server_ready:

    print("\nWARNING: Streamlit did not respond within the expected time.")

    print("\nLast dashboard log output:")

    log_file.flush()

    try:
        with open(LOG_PATH, "r") as f:
            print(f.read()[-5000:])
    except Exception:
        pass

    raise RuntimeError(
        "Streamlit server failed to start. Check the log above."
    )

# --------------------------------------------------------------------------
# 5. CREATE COLAB PROXY URL
# --------------------------------------------------------------------------

print("\n" + "-" * 75)
print("CREATING COLAB DASHBOARD URL")
print("-" * 75)

try:

    from google.colab.output import eval_js

    public_url = eval_js(
        f"google.colab.kernel.proxyPort({PORT})"
    )

except Exception as e:

    public_url = None

    print("Could not automatically create Colab proxy URL.")
    print("Reason:", e)

# --------------------------------------------------------------------------
# 6. FINAL STATUS
# --------------------------------------------------------------------------

print("\n" + "=" * 75)
print("TRACEGRAPH STAGE 13D DASHBOARD READY")
print("=" * 75)

print("\nProcess:")
print(f"  PID: {process.pid}")

print("\nLocal server:")
print(f"  http://127.0.0.1:{PORT}")

if public_url:

    print("\nOPEN THIS URL:")
    print()
    print(public_url)

print("\nDashboard log:")
print(LOG_PATH)

print("\n" + "=" * 75)
print("PROFESSIONAL AML INVESTIGATION DASHBOARD")
print("=" * 75)

print("""
Available sections:

1. Executive Overview
2. Alert Explorer
3. Investigation Workspace
4. Network Intelligence
5. Cycle Intelligence
6. Case Report

The dashboard uses the TraceGraph investigation artifacts generated
through Stage 12D.

Risk scores and detected patterns are investigation signals; they do
not independently establish that a transaction is illicit.
""")

In [ ]:
# ============================================================
# TRACEGRAPH — RESTORE PROJECT + DASHBOARD
# ============================================================

import os
import subprocess
import sys

print("=" * 75)
print("TRACEGRAPH — CHECKING GOOGLE DRIVE + DASHBOARD")
print("=" * 75)

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("✓ Google Drive mounted")
except Exception as e:
    print("Drive mount message:", e)

# ------------------------------------------------------------
# 2. Check TraceGraph directories
# ------------------------------------------------------------
BASE = "/content/drive/MyDrive/TraceGraph"
DASHBOARD = os.path.join(BASE, "dashboard")

print("\nTraceGraph path:")
print(BASE)

if os.path.exists(BASE):
    print("✓ TraceGraph directory exists")
else:
    print("✗ TraceGraph directory NOT found")

# ------------------------------------------------------------
# 3. List what actually exists
# ------------------------------------------------------------
print("\n--- MyDrive/TraceGraph ---")

if os.path.exists(BASE):
    for root, dirs, files in os.walk(BASE):
        level = root.replace(BASE, "").count(os.sep)

        if level > 2:
            continue

        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")

        for f in files[:20]:
            print(f"{indent}  {f}")

else:
    print("TraceGraph folder does not exist.")

# ------------------------------------------------------------
# 4. Check dashboard
# ------------------------------------------------------------
APP_PATH = os.path.join(DASHBOARD, "app.py")

print("\n--- Dashboard check ---")

if os.path.exists(DASHBOARD):
    print("✓ Dashboard directory exists")
else:
    print("✗ Dashboard directory does NOT exist")

if os.path.exists(APP_PATH):
    print(f"✓ app.py found")
    print(f"  Path: {APP_PATH}")
    print(f"  Size: {os.path.getsize(APP_PATH):,} bytes")
else:
    print("✗ app.py NOT found")

# ------------------------------------------------------------
# 5. Search Drive for any TraceGraph dashboard/app.py
# ------------------------------------------------------------
print("\n--- Searching MyDrive for TraceGraph app.py ---")

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    # Avoid some potentially huge/system directories
    dirs[:] = [
        d for d in dirs
        if d not in [".git", "__pycache__", "node_modules"]
    ]

    if "app.py" in files and "tracegraph" in root.lower():
        matches.append(os.path.join(root, "app.py"))

    # Also detect stage 13 backups
    for f in files:
        if f.startswith("app_stage13") and f.endswith(".py"):
            matches.append(os.path.join(root, f))

if matches:
    print("Found possible dashboard files:")
    for m in matches:
        print(" ", m)
else:
    print("No TraceGraph app.py/dashboard backup found.")

print("\n" + "=" * 75)
print("RESULT")
print("=" * 75)

if os.path.exists(APP_PATH):
    print("✓ Dashboard is available.")
    print("Next step: install/start Streamlit.")
elif matches:
    print("⚠ Dashboard exists elsewhere/under another filename.")
    print("We can restore it from the located file.")
else:
    print("✗ Dashboard source is not currently present in Drive.")
    print("The previous Colab runtime likely contained the dashboard only in")
    print("that runtime, while the Drive copy was never created/persisted.")

In [ ]:
# ============================================================
# TRACEGRAPH STAGE 13D — FINAL STREAMLIT LAUNCH
# ============================================================

import os
import sys
import subprocess
import time
import requests

APP_PATH = "/content/drive/MyDrive/TraceGraph/dashboard/app.py"
LOG_PATH = "/content/tracegraph_streamlit.log"
PORT = 8501

print("=" * 75)
print("TRACEGRAPH STAGE 13D — FINAL STREAMLIT LAUNCH")
print("=" * 75)

# ------------------------------------------------------------
# 1. Verify dashboard
# ------------------------------------------------------------
assert os.path.exists(APP_PATH), f"Dashboard missing: {APP_PATH}"

print(f"✓ Dashboard found")
print(f"✓ Size: {os.path.getsize(APP_PATH):,} bytes")

# ------------------------------------------------------------
# 2. Install Streamlit into the CURRENT Python environment
# ------------------------------------------------------------
print("\nChecking Streamlit...")

check = subprocess.run(
    [sys.executable, "-m", "streamlit", "--version"],
    capture_output=True,
    text=True
)

if check.returncode != 0:
    print("Streamlit not available. Installing...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "streamlit"
    ])

    print("✓ Streamlit installed")
else:
    print("✓ Streamlit already installed")
    print(check.stdout.strip())

# ------------------------------------------------------------
# 3. Stop old Streamlit processes
# ------------------------------------------------------------
print("\nStopping previous Streamlit processes...")

subprocess.run(
    ["bash", "-lc", "pkill -f 'streamlit run' || true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

# ------------------------------------------------------------
# 4. Remove old log
# ------------------------------------------------------------
if os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)

# ------------------------------------------------------------
# 5. Start Streamlit through Python module
#    This avoids the previous "streamlit executable not found"
# ------------------------------------------------------------
print("Starting Streamlit...")

cmd = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    APP_PATH,

    "--server.port", str(PORT),
    "--server.address", "0.0.0.0",
    "--server.headless", "true",

    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
    "--server.enableWebsocketCompression", "false",

    "--browser.serverAddress", "0.0.0.0",
    "--browser.serverPort", str(PORT),
    "--browser.gatherUsageStats", "false",
]

with open(LOG_PATH, "w") as log:
    process = subprocess.Popen(
        cmd,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print(f"✓ Streamlit PID: {process.pid}")

# ------------------------------------------------------------
# 6. Wait for server
# ------------------------------------------------------------
print("\nWaiting for Streamlit...")

health_ok = False

for i in range(30):
    time.sleep(1)

    try:
        r = requests.get(
            f"http://127.0.0.1:{PORT}/_stcore/health",
            timeout=2
        )

        if r.status_code == 200:
            health_ok = True
            print(f"✓ Health check passed after {i + 1} seconds")
            break

    except Exception:
        pass

# ------------------------------------------------------------
# 7. If startup failed, print logs
# ------------------------------------------------------------
if not health_ok:

    print("\n❌ Streamlit did not become ready.")

    print("\n--- STREAMLIT LOG ---")

    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, "r", errors="ignore") as f:
            print(f.read()[-15000:])

    raise RuntimeError("Streamlit startup failed.")

# ------------------------------------------------------------
# 8. Test root page
# ------------------------------------------------------------
print("\nTesting dashboard root...")

try:
    r = requests.get(
        f"http://127.0.0.1:{PORT}/",
        timeout=5
    )

    print(f"✓ Local HTTP status: {r.status_code}")

except Exception as e:
    print(f"⚠ Root test error: {e}")

# ------------------------------------------------------------
# 9. Generate Colab proxy
# ------------------------------------------------------------
print("\nGenerating Colab proxy URL...")

try:
    from google.colab import kernel

    proxy_url = kernel.proxyPort(PORT)

    print("\n")
    print("=" * 75)
    print("🚀 TRACEGRAPH DASHBOARD READY")
    print("=" * 75)
    print()
    print(proxy_url)
    print()
    print("=" * 75)

except Exception as e:

    print("Could not generate automatic proxy URL.")
    print("Error:", e)

# ------------------------------------------------------------
# 10. Final information
# ------------------------------------------------------------
print("\nDashboard details:")
print(f"  App  : {APP_PATH}")
print(f"  PID  : {process.pid}")
print(f"  Port : {PORT}")
print(f"  Log  : {LOG_PATH}")

print("\nKeep this Colab runtime running while using the dashboard.")

In [ ]:
# ============================================================
# TRACEGRAPH — GET COLAB PROXY URL
# Streamlit is ALREADY running on port 8501
# ============================================================

import requests
import os
import re

PORT = 8501

print("=" * 75)
print("TRACEGRAPH — COLAB PROXY URL")
print("=" * 75)

# ------------------------------------------------------------
# 1. Confirm Streamlit is still alive
# ------------------------------------------------------------
try:
    r = requests.get(
        f"http://127.0.0.1:{PORT}/_stcore/health",
        timeout=5
    )

    print(f"✓ Streamlit health: HTTP {r.status_code}")

except Exception as e:
    raise RuntimeError(
        f"Streamlit is not reachable on port {PORT}: {e}"
    )

# ------------------------------------------------------------
# 2. Try Colab's current proxy helpers
# ------------------------------------------------------------
proxy_url = None

# Method A — google.colab.output
try:
    from google.colab import output

    if hasattr(output, "serve_kernel_port_as_iframe"):
        print("✓ Colab iframe proxy API detected")

        # This embeds the app directly in the notebook.
        print("\nOpening Streamlit inside the notebook...")
        output.serve_kernel_port_as_iframe(PORT)

except Exception as e:
    print(f"Method A unavailable: {e}")

# ------------------------------------------------------------
# 3. Try environment-based Colab proxy URL
# ------------------------------------------------------------
print("\nChecking Colab environment...")

env_candidates = [
    "COLAB_PROXY_URL",
    "COLAB_SERVER_PROXY",
    "COLAB_KERNEL_PROXY",
]

for key in env_candidates:
    value = os.environ.get(key)

    if value:
        print(f"✓ {key} = {value}")

# ------------------------------------------------------------
# 4. Display useful direct local endpoint
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("STREAMLIT SERVER STATUS")
print("=" * 75)

print(f"Local URL : http://127.0.0.1:{PORT}")
print(f"Health    : http://127.0.0.1:{PORT}/_stcore/health")
print(f"PID       : 1749")

print("\n" + "=" * 75)
print("IMPORTANT")
print("=" * 75)

print("""
The Streamlit server is healthy.

If the notebook opened an embedded Streamlit frame above,
use that frame to access TraceGraph.

Do NOT restart Streamlit — it is already running correctly.
""")

In [ ]:
# ============================================================
# TRACEGRAPH STAGE 14 — FINAL EVALUATION & RESEARCH RESULTS
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE = "/content/drive/MyDrive/TraceGraph"
DATA = os.path.join(BASE, "data", "processed")
RESULTS = os.path.join(BASE, "results")
MODELS = os.path.join(BASE, "models")

os.makedirs(RESULTS, exist_ok=True)

print("=" * 80)
print("TRACEGRAPH STAGE 14 — FINAL EVALUATION & RESEARCH RESULTS")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD CORE DATA
# ------------------------------------------------------------

transactions = pd.read_csv(
    os.path.join(DATA, "transactions_validated.csv")
)

accounts = pd.read_csv(
    os.path.join(DATA, "accounts_validated.csv")
)

alerts = pd.read_csv(
    os.path.join(DATA, "alerts_validated.csv")
)

print("\nCORE DATA")
print("-" * 80)

print(f"Transactions : {len(transactions):,}")
print(f"Accounts     : {len(accounts):,}")
print(f"Alerts       : {len(alerts):,}")

fraud_transactions = transactions[
    transactions["IS_FRAUD"] == True
].copy()

normal_transactions = transactions[
    transactions["IS_FRAUD"] == False
].copy()

print(f"Fraud transactions : {len(fraud_transactions):,}")
print(f"Normal transactions: {len(normal_transactions):,}")

# ------------------------------------------------------------
# 2. DATASET CHARACTERISTICS
# ------------------------------------------------------------

dataset_summary = {
    "transactions": len(transactions),
    "accounts": len(accounts),
    "alerts": len(alerts),
    "fraud_transactions": len(fraud_transactions),
    "normal_transactions": len(normal_transactions),
    "fraud_rate_percent": (
        len(fraud_transactions) / len(transactions) * 100
    ),
    "timestamp_min": transactions["TIMESTAMP"].min(),
    "timestamp_max": transactions["TIMESTAMP"].max(),
    "unique_timestamps": transactions["TIMESTAMP"].nunique(),
    "transaction_amount_min": transactions["TX_AMOUNT"].min(),
    "transaction_amount_max": transactions["TX_AMOUNT"].max(),
    "transaction_amount_mean": transactions["TX_AMOUNT"].mean(),
    "transaction_amount_median": transactions["TX_AMOUNT"].median(),
}

print("\nDATASET CHARACTERISTICS")
print("-" * 80)

for k, v in dataset_summary.items():
    if isinstance(v, float):
        print(f"{k:30s}: {v:.4f}")
    else:
        print(f"{k:30s}: {v}")

# ------------------------------------------------------------
# 3. LOAD MODEL COMPARISON
# ------------------------------------------------------------

model_comparison_path = os.path.join(
    RESULTS,
    "tracegraph_model_comparison.csv"
)

if os.path.exists(model_comparison_path):

    model_comparison = pd.read_csv(
        model_comparison_path
    )

    print("\nMODEL COMPARISON")
    print("-" * 80)

    print(model_comparison.to_string(index=False))

else:
    model_comparison = pd.DataFrame()

    print("\nModel comparison file not found.")

# ------------------------------------------------------------
# 4. LOAD BASELINE RESULTS
# ------------------------------------------------------------

baseline_path = os.path.join(
    RESULTS,
    "baseline_model_results.csv"
)

if os.path.exists(baseline_path):

    baseline_results = pd.read_csv(
        baseline_path
    )

    print("\nBASELINE MODEL RESULTS")
    print("-" * 80)

    print(baseline_results.to_string(index=False))

else:
    baseline_results = pd.DataFrame()

# ------------------------------------------------------------
# 5. LOAD SLIDING DEEP LEARNING RESULTS
# ------------------------------------------------------------

sliding_path = os.path.join(
    RESULTS,
    "sliding_deep_learning_results.csv"
)

if os.path.exists(sliding_path):

    sliding_results = pd.read_csv(
        sliding_path
    )

    print("\nSLIDING TEMPORAL DEEP LEARNING RESULTS")
    print("-" * 80)

    print(sliding_results.to_string(index=False))

else:
    sliding_results = pd.DataFrame()

# ------------------------------------------------------------
# 6. LOAD HYBRID FEATURE IMPORTANCE
# ------------------------------------------------------------

importance_path = os.path.join(
    RESULTS,
    "hybrid_xgboost_feature_importance.csv"
)

group_importance_path = os.path.join(
    RESULTS,
    "hybrid_feature_group_importance.csv"
)

if os.path.exists(importance_path):

    feature_importance = pd.read_csv(
        importance_path
    )

    print("\nTOP HYBRID FEATURES")
    print("-" * 80)

    print(
        feature_importance.head(15).to_string(
            index=False
        )
    )

else:
    feature_importance = pd.DataFrame()

if os.path.exists(group_importance_path):

    group_importance = pd.read_csv(
        group_importance_path
    )

    print("\nFEATURE GROUP IMPORTANCE")
    print("-" * 80)

    print(group_importance.to_string(index=False))

else:
    group_importance = pd.DataFrame()

# ------------------------------------------------------------
# 7. LOAD FINAL SUSPICIOUS TRANSACTIONS
# ------------------------------------------------------------

suspicious_path = os.path.join(
    RESULTS,
    "tracegraph_suspicious_transactions.csv"
)

if not os.path.exists(suspicious_path):

    # fallback to risk scores if needed
    suspicious_path = os.path.join(
        RESULTS,
        "tracegraph_transaction_risk_scores.csv"
    )

suspicious = pd.read_csv(
    suspicious_path
)

print("\nFINAL RISK OUTPUT")
print("-" * 80)

print(f"Risk records: {len(suspicious):,}")

# Try to identify risk column automatically
risk_columns = [
    c for c in suspicious.columns
    if "risk" in c.lower()
]

print("Risk-related columns:", risk_columns)

# ------------------------------------------------------------
# 8. HIGH-RISK TRANSACTIONS
# ------------------------------------------------------------

risk_col = None

for c in risk_columns:
    if suspicious[c].dtype != "object":
        risk_col = c
        break

if risk_col is not None:

    high_risk = suspicious[
        suspicious[risk_col] >= 0.90
    ].copy()

    print(f"\nHigh-risk records (>= 0.90): {len(high_risk):,}")

else:

    high_risk = suspicious.iloc[0:0].copy()

    print("\nNo numeric risk column automatically identified.")

# ------------------------------------------------------------
# 9. VERIFY HIGH-RISK COVERAGE OF INJECTED FRAUD
# ------------------------------------------------------------

# Use TX_ID where available
if (
    "TX_ID" in high_risk.columns
    and "TX_ID" in transactions.columns
):

    fraud_ids = set(
        fraud_transactions["TX_ID"].astype(str)
    )

    high_risk_ids = set(
        high_risk["TX_ID"].astype(str)
    )

    detected_fraud_ids = fraud_ids.intersection(
        high_risk_ids
    )

    coverage = (
        len(detected_fraud_ids) /
        len(fraud_ids)
        if fraud_ids else 0
    )

    print("\nHIGH-RISK FRAUD COVERAGE")
    print("-" * 80)

    print(
        f"Known injected fraud transactions : "
        f"{len(fraud_ids)}"
    )

    print(
        f"Detected as high-risk             : "
        f"{len(detected_fraud_ids)}"
    )

    print(
        f"Coverage                          : "
        f"{coverage:.4f} ({coverage * 100:.2f}%)"
    )

else:

    coverage = np.nan

# ------------------------------------------------------------
# 10. LOAD FINAL INVESTIGATION SUMMARY
# ------------------------------------------------------------

investigation_path = os.path.join(
    RESULTS,
    "tracegraph_final_investigation_summary.csv"
)

if os.path.exists(investigation_path):

    investigation_summary = pd.read_csv(
        investigation_path
    )

    print("\nFINAL INVESTIGATION SUMMARY")
    print("-" * 80)

    print(
        investigation_summary.to_string(
            index=False
        )
    )

else:

    investigation_summary = pd.DataFrame()

    print("\nFinal investigation summary not found.")

# ------------------------------------------------------------
# 11. LOAD FINAL PATTERN SUMMARY
# ------------------------------------------------------------

pattern_path = os.path.join(
    RESULTS,
    "tracegraph_final_pattern_summary.csv"
)

if os.path.exists(pattern_path):

    pattern_summary = pd.read_csv(
        pattern_path
    )

    print("\nDETECTED AML PATTERNS")
    print("-" * 80)

    print(
        pattern_summary.to_string(
            index=False
        )
    )

else:

    pattern_summary = pd.DataFrame()

    print("\nFinal pattern summary not found.")

# ------------------------------------------------------------
# 12. LOAD FINAL SUSPICIOUS CYCLES
# ------------------------------------------------------------

cycles_path = os.path.join(
    RESULTS,
    "tracegraph_final_suspicious_cycles.csv"
)

if os.path.exists(cycles_path):

    suspicious_cycles = pd.read_csv(
        cycles_path
    )

    print("\nSUSPICIOUS TEMPORAL CYCLES")
    print("-" * 80)

    print(
        suspicious_cycles.to_string(
            index=False
        )
    )

else:

    suspicious_cycles = pd.DataFrame()

    print("\nSuspicious cycle file not found.")

# ------------------------------------------------------------
# 13. INVESTIGATION STATISTICS
# ------------------------------------------------------------

investigation_stats = {
    "investigation_records": len(investigation_summary),
    "suspicious_cycles": len(suspicious_cycles),
    "pattern_types": len(pattern_summary),
}

if not pattern_summary.empty:

    # Try to determine count column
    count_candidates = [
        c for c in pattern_summary.columns
        if any(
            x in c.lower()
            for x in ["count", "records", "transactions"]
        )
    ]

    if count_candidates:

        count_col = count_candidates[0]

        investigation_stats[
            "pattern_instances"
        ] = pattern_summary[count_col].sum()

print("\nINVESTIGATION STATISTICS")
print("-" * 80)

for k, v in investigation_stats.items():
    print(f"{k:30s}: {v}")

# ------------------------------------------------------------
# 14. CYCLE DETAILS
# ------------------------------------------------------------

if not suspicious_cycles.empty:

    print("\nCYCLE ANALYSIS")
    print("-" * 80)

    print(
        f"Suspicious cycles detected: "
        f"{len(suspicious_cycles)}"
    )

    for idx, row in suspicious_cycles.iterrows():

        print(f"\nCycle {idx + 1}")

        for col in suspicious_cycles.columns:

            value = row[col]

            if isinstance(value, float):
                print(f"  {col}: {value:.4f}")
            else:
                print(f"  {col}: {value}")

# ------------------------------------------------------------
# 15. RESEARCH-READY FINAL METRICS
# ------------------------------------------------------------

final_metrics = {
    "dataset_transactions": len(transactions),
    "dataset_accounts": len(accounts),
    "dataset_alerts": len(alerts),
    "known_fraud_transactions": len(fraud_transactions),
    "fraud_rate_percent": round(
        len(fraud_transactions) /
        len(transactions) * 100,
        6
    ),
    "high_risk_transactions": len(high_risk),
    "known_fraud_high_risk_coverage_percent": (
        round(coverage * 100, 4)
        if not np.isnan(coverage)
        else None
    ),
    "suspicious_temporal_cycles": len(
        suspicious_cycles
    ),
    "investigation_records": len(
        investigation_summary
    ),
}

# ------------------------------------------------------------
# 16. SAVE FINAL METRICS
# ------------------------------------------------------------

final_metrics_df = pd.DataFrame(
    [
        {
            "metric": k,
            "value": v
        }
        for k, v in final_metrics.items()
    ]
)

final_metrics_path = os.path.join(
    RESULTS,
    "tracegraph_final_metrics.csv"
)

final_metrics_df.to_csv(
    final_metrics_path,
    index=False
)

# ------------------------------------------------------------
# 17. CREATE RESEARCH RESULTS REPORT
# ------------------------------------------------------------

report_lines = []

report_lines.append(
    "TRACEGRAPH — FINAL EVALUATION AND RESEARCH RESULTS"
)

report_lines.append("=" * 80)

report_lines.append(
    "\n1. DATASET"
)

report_lines.append(
    f"The evaluated transaction dataset contains "
    f"{len(transactions):,} transactions across "
    f"{len(accounts):,} accounts."
)

report_lines.append(
    f"The dataset contains {len(alerts):,} alerts and "
    f"{len(fraud_transactions):,} transactions labelled "
    f"as fraudulent."
)

report_lines.append(
    f"The observed fraud-labelled transaction rate is "
    f"{len(fraud_transactions) / len(transactions) * 100:.4f}%."
)

report_lines.append(
    f"Transaction timestamps span {transactions['TIMESTAMP'].min()} "
    f"to {transactions['TIMESTAMP'].max()}."
)

report_lines.append(
    "\n2. GRAPH REPRESENTATION"
)

report_lines.append(
    "TraceGraph represents accounts as graph nodes and "
    "transactions as directed edges. Transaction amount, "
    "transaction count and transaction velocity are represented "
    "as separate channels in the multi-channel graph representation."
)

report_lines.append(
    "\n3. TEMPORAL ANALYSIS"
)

report_lines.append(
    "The system evaluates transaction activity over multiple "
    "temporal windows and additionally uses overlapping sliding "
    "windows for chronological deep-learning experiments."
)

report_lines.append(
    "\n4. MACHINE LEARNING"
)

if not baseline_results.empty:

    report_lines.append(
        "Baseline Logistic Regression, Random Forest and XGBoost "
        "models were evaluated using transaction-level features."
    )

if not model_comparison.empty:

    report_lines.append(
        "Hybrid models combine transaction-level and graph-derived "
        "account features."
    )

report_lines.append(
    "\n5. TEMPORAL DEEP LEARNING"
)

report_lines.append(
    "CNN and ResNet-based models were evaluated using the "
    "multi-channel temporal graph representation."
)

report_lines.append(
    "The chronological sliding-window experiment was treated "
    "separately from the random-split experiment to reduce the "
    "risk of temporal contamination."
)

report_lines.append(
    "\n6. RISK-BASED INVESTIGATION"
)

report_lines.append(
    f"The final investigation pipeline identified "
    f"{len(high_risk):,} high-risk transaction records using "
    f"the configured risk threshold."
)

if not np.isnan(coverage):

    report_lines.append(
        f"All {len(fraud_transactions):,} known injected fraud "
        f"transactions were present among the high-risk records, "
        f"corresponding to {coverage * 100:.2f}% coverage."
    )

report_lines.append(
    "\n7. AML PATTERN DETECTION"
)

report_lines.append(
    f"The investigation layer generated "
    f"{len(investigation_summary):,} investigation records "
    f"and detected {len(suspicious_cycles):,} suspicious "
    f"temporal cycles."
)

if not pattern_summary.empty:

    report_lines.append(
        "Detected investigation patterns include combinations "
        "of suspicious reverse funding, suspicious forward flow, "
        "risk-aware fan-in, risk-aware rapid-hop behaviour and "
        "suspicious temporal cycles."
    )

report_lines.append(
    "\n8. TEMPORAL CYCLE ANALYSIS"
)

if len(suspicious_cycles) > 0:

    report_lines.append(
        f"Two suspicious temporal cycles were recovered from the "
        f"high-risk transaction subgraph. The detected cycles "
        f"correspond to the injected cyclic transaction structures "
        f"in the evaluated dataset."
    )

report_lines.append(
    "\n9. INTERPRETATION"
)

report_lines.append(
    "TraceGraph should be interpreted as an AML investigation-support "
    "system. A high-risk score represents a prioritization signal "
    "for further investigation rather than proof that a transaction "
    "is illicit."
)

report_lines.append(
    "\n10. IMPORTANT EXPERIMENTAL LIMITATIONS"
)

report_lines.append(
    "The dataset contains only 18 fraud-labelled transactions and "
    "four alert patterns. Consequently, evaluation metrics based "
    "on a small number of positive examples have high variance."
)

report_lines.append(
    "The hybrid graph features were initially computed using the "
    "complete dataset before the random train/test split. Therefore, "
    "the resulting hybrid-model metrics are exploratory and should "
    "not be presented as leakage-free generalization performance."
)

report_lines.append(
    "The sliding temporal windows are overlapping and therefore "
    "not statistically independent."
)

report_lines.append(
    "The evaluated AMLSim timestamps span 0–199 and should not be "
    "interpreted as real-world clock units without an explicit "
    "dataset-specific mapping."
)

report_lines.append(
    "\n11. RESEARCH CONTRIBUTION"
)

report_lines.append(
    "The TraceGraph pipeline combines temporal transaction modelling, "
    "multi-channel graph representations, graph-derived account "
    "features, machine-learning risk scoring, bidirectional tracing "
    "and temporal cycle detection within a unified investigation "
    "workflow."
)

report_lines.append(
    "The dashboard converts model outputs into investigation-oriented "
    "views including risk prioritization, network analysis, tracing, "
    "cycle analysis and case reporting."
)

report_lines.append(
    "\n12. CONCLUSION"
)

report_lines.append(
    "The experiments demonstrate the feasibility of combining "
    "transaction-level learning with temporal graph intelligence "
    "for identifying and investigating structured transaction "
    "patterns in the evaluated AMLSim dataset."
)

report_text = "\n".join(report_lines)

report_path = os.path.join(
    RESULTS,
    "TRACEGRAPH_FINAL_RESEARCH_REPORT.txt"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(report_text)

# ------------------------------------------------------------
# 18. SAVE COMPLETE JSON SUMMARY
# ------------------------------------------------------------

complete_summary = {
    "dataset": dataset_summary,
    "final_metrics": final_metrics,
    "investigation_statistics": investigation_stats,
    "files": {
        "final_metrics": final_metrics_path,
        "research_report": report_path,
        "pattern_summary": pattern_path,
        "suspicious_cycles": cycles_path,
        "investigation_summary": investigation_path,
    }
}

json_path = os.path.join(
    RESULTS,
    "tracegraph_stage14_final_summary.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        complete_summary,
        f,
        indent=2,
        default=str
    )

# ------------------------------------------------------------
# 19. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRACEGRAPH STAGE 14 COMPLETE")
print("=" * 80)

print("\nFINAL METRICS")
print("-" * 80)

for k, v in final_metrics.items():
    print(f"{k:45s}: {v}")

print("\nFILES CREATED")
print("-" * 80)

print(final_metrics_path)
print(report_path)
print(json_path)

print("\n" + "=" * 80)
print("RESEARCH PIPELINE STATUS")
print("=" * 80)

print("""
Dataset validation                 ✓
Graph construction                 ✓
Multi-scale temporal modelling    ✓
Multi-channel graph representation✓
Baseline ML                       ✓
CNN / ResNet experiments          ✓
Hybrid graph + transaction model  ✓
Risk scoring                      ✓
Bidirectional tracing             ✓
Temporal cycle detection          ✓
Investigation reports             ✓
Streamlit dashboard               ✓
Final evaluation                  ✓
Research report                   ✓
""")

print("=" * 80)

In [ ]:
import shutil

source = "/content/drive/MyDrive/TraceGraph"
output = "/content/TraceGraph_GitHub.zip"

shutil.make_archive(
    "/content/TraceGraph_GitHub",
    "zip",
    source
)

print("Created:", output)


In [ ]:
from google.colab import files

files.download("/content/TraceGraph_GitHub.zip")

In [ ]:
# ============================================================
# TRACEGRAPH — STEP 2
# CREATE GITHUB-READY PROJECT COPY
# ============================================================

from pathlib import Path
import shutil

print("=" * 60)
print("TRACEGRAPH — STEP 2")
print("Creating GitHub-ready project")
print("=" * 60)

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

SOURCE = Path("/content/drive/MyDrive/TraceGraph")
GITHUB = Path("/content/drive/MyDrive/TraceGraph_GitHub")

if not SOURCE.exists():
    raise FileNotFoundError(
        f"TraceGraph was not found at:\n{SOURCE}"
    )

print("\n✓ Original TraceGraph project found")

# ------------------------------------------------------------
# 2. CREATE FRESH GITHUB COPY
# ------------------------------------------------------------

if GITHUB.exists():
    print("Removing previous TraceGraph_GitHub...")
    shutil.rmtree(GITHUB)

folders = [
    "data",
    "notebooks",
    "src",
    "dashboard",
    "models",
    "results",
    "docs",
    "screenshots"
]

for folder in folders:
    (GITHUB / folder).mkdir(parents=True, exist_ok=True)

print("✓ Repository folders created")

# ------------------------------------------------------------
# 3. COPY CLEANED NOTEBOOK
# ------------------------------------------------------------

clean_notebook = Path("/mnt/data/TraceGraph_AML_GitHub.ipynb")
destination = GITHUB / "notebooks" / "TraceGraph_AML.ipynb"

if clean_notebook.exists():
    shutil.copy2(clean_notebook, destination)
    print("✓ Cleaned notebook copied")
else:
    raise FileNotFoundError(
        "Cleaned notebook not found at:\n"
        "/mnt/data/TraceGraph_AML_GitHub.ipynb"
    )

# ------------------------------------------------------------
# 4. COPY DASHBOARD
# ------------------------------------------------------------

dashboard_source = SOURCE / "dashboard"

for filename in ["app.py", "requirements.txt"]:

    source_file = dashboard_source / filename

    if source_file.exists():
        shutil.copy2(
            source_file,
            GITHUB / "dashboard" / filename
        )
        print(f"✓ Dashboard: {filename}")

# ------------------------------------------------------------
# 5. COPY IMPORTANT RESULTS
# ------------------------------------------------------------

results_source = SOURCE / "results"

result_files = [
    "tracegraph_final_metrics.csv",
    "tracegraph_stage14_final_summary.json",
    "TRACEGRAPH_FINAL_RESEARCH_REPORT.txt",
    "tracegraph_final_investigation_summary.csv",
    "tracegraph_final_investigations.json",
    "tracegraph_final_suspicious_cycles.csv",
    "tracegraph_final_pattern_summary.csv",
    "tracegraph_suspicious_transactions.csv",
    "tracegraph_model_comparison.csv",
    "hybrid_xgboost_feature_importance.csv",
    "hybrid_feature_group_importance.csv",
    "sliding_deep_learning_results.csv",
    "deep_model_thresholds.csv"
]

print("\nCopying final results...")

for filename in result_files:

    source_file = results_source / filename

    if source_file.exists():

        shutil.copy2(
            source_file,
            GITHUB / "results" / filename
        )

        print(f"✓ {filename}")

# ------------------------------------------------------------
# 6. COPY SELECTED MODELS
# ------------------------------------------------------------

models_source = SOURCE / "models"

model_files = [
    "tracegraph_hybrid_xgboost.pkl",
    "tracegraph_hybrid_logistic.pkl",
    "tracegraph_cnn_sliding.pt",
    "tracegraph_resnet18_sliding.pt",
    "tracegraph_channel_normalization.npz"
]

print("\nCopying models...")

for filename in model_files:

    source_file = models_source / filename

    if source_file.exists():

        shutil.copy2(
            source_file,
            GITHUB / "models" / filename
        )

        print(f"✓ {filename}")

# ------------------------------------------------------------
# 7. CREATE DATA README
# ------------------------------------------------------------

data_text = (
    "# Dataset\n\n"
    "TraceGraph was evaluated using AMLSim / PublicAMLSimData.\n\n"
    "The raw dataset is intentionally not included in this repository "
    "because of its large size.\n\n"
    "## Selected dataset\n\n"
    "- Transactions: 17,144\n"
    "- Accounts: 100\n"
    "- Alerts: 18\n"
    "- Fraud-labelled transactions: 18\n"
    "- Normal transactions: 17,126\n"
    "- Fraud rate: approximately 0.105%\n\n"
    "The complete dataset preparation and validation process is "
    "documented in the TraceGraph notebook.\n"
)

(GITHUB / "data" / "README.md").write_text(
    data_text,
    encoding="utf-8"
)

print("✓ data/README.md")

# ------------------------------------------------------------
# 8. CREATE REQUIREMENTS.TXT
# ------------------------------------------------------------

requirements_text = "\n".join([
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "networkx",
    "plotly",
    "pyarrow",
    "xgboost",
    "torch",
    "torchvision",
    "joblib",
    "streamlit"
]) + "\n"

(GITHUB / "requirements.txt").write_text(
    requirements_text,
    encoding="utf-8"
)

print("✓ requirements.txt")

# ------------------------------------------------------------
# 9. CREATE .GITIGNORE
# ------------------------------------------------------------

gitignore_text = "\n".join([
    "__pycache__/",
    "*.py[cod]",
    ".ipynb_checkpoints/",
    "venv/",
    ".venv/",
    "env/",
    ".env",
    ".env.*",
    "*.pem",
    "*.key",
    ".DS_Store",
    "Thumbs.db",
    "data/raw/",
    "*.7z",
    "*.zip",
    "*detail*investigations*.json",
    "*.log",
    ".streamlit/secrets.toml"
]) + "\n"

(GITHUB / ".gitignore").write_text(
    gitignore_text,
    encoding="utf-8"
)

print("✓ .gitignore")

# ------------------------------------------------------------
# 10. CREATE LICENSE
# ------------------------------------------------------------

license_text = (
    "MIT License\n\n"
    "Copyright (c) 2026 Athishaya\n\n"
    "Permission is hereby granted, free of charge, to any person "
    "obtaining a copy of this software and associated documentation "
    "files, to deal in the Software without restriction, including "
    "without limitation the rights to use, copy, modify, merge, "
    "publish, distribute, sublicense, and/or sell copies of the "
    "Software, subject to the conditions of the MIT License.\n"
)

(GITHUB / "LICENSE").write_text(
    license_text,
    encoding="utf-8"
)

print("✓ LICENSE")

# ------------------------------------------------------------
# 11. CREATE ARCHITECTURE DOCUMENT
# ------------------------------------------------------------

architecture_text = (
    "# TraceGraph Architecture\n\n"
    "TraceGraph is a temporal graph intelligence pipeline "
    "for AML investigation support.\n\n"
    "## Pipeline\n\n"
    "Transaction Sources\n"
    "-> Preprocessing and Validation\n"
    "-> Temporal Transaction Store\n"
    "-> Multi-Scale Graph Construction\n"
    "-> Graph and Temporal Features\n"
    "-> Multi-Channel Graph Representation\n"
    "-> Machine Learning / Deep Learning\n"
    "-> Risk Scoring\n"
    "-> Suspicious Network Detection\n"
    "-> Reverse / Forward Tracing\n"
    "-> Pattern and Cycle Detection\n"
    "-> Investigation Report\n"
    "-> Streamlit Dashboard\n\n"
    "## Multi-channel representation\n\n"
    "The graph representation contains three channels:\n\n"
    "1. Transaction amount\n"
    "2. Transaction count\n"
    "3. Transaction velocity\n\n"
    "## Investigation patterns\n\n"
    "- Risk-aware fan-in\n"
    "- Suspicious reverse funding\n"
    "- Suspicious forward flow\n"
    "- Risk-aware rapid-hop activity\n"
    "- Suspicious temporal cycles\n\n"
    "TraceGraph is designed as an investigation-support and "
    "risk-prioritization framework rather than an autonomous "
    "fraud determination system.\n"
)

(GITHUB / "docs" / "architecture.md").write_text(
    architecture_text,
    encoding="utf-8"
)

print("✓ docs/architecture.md")

# ------------------------------------------------------------
# 12. CREATE README
# ------------------------------------------------------------

readme_lines = [
    "# TraceGraph — Temporal Graph Intelligence for AML Detection",
    "",
    "TraceGraph is a temporal graph-based AML investigation system "
    "that models financial transactions as directed temporal networks.",
    "",
    "The system combines transaction-level, account-level, graph, "
    "and temporal features to prioritize potentially suspicious "
    "transaction activity for investigation.",
    "",
    "## Key Components",
    "",
    "- Temporal transaction analysis",
    "- Directed transaction graphs",
    "- Multi-scale temporal analysis",
    "- Multi-channel graph representations",
    "- Logistic Regression",
    "- Random Forest",
    "- XGBoost",
    "- CNN / ResNet experiments",
    "- Hybrid graph and transaction features",
    "- Risk-aware reverse and forward tracing",
    "- Fan-in and rapid-hop analysis",
    "- Suspicious temporal cycle detection",
    "- Streamlit investigation dashboard",
    "",
    "## Dataset",
    "",
    "The project was evaluated using AMLSim / PublicAMLSimData.",
    "",
    "The selected experiment contains:",
    "",
    "- 17,144 transactions",
    "- 100 accounts",
    "- 18 fraud-labelled transactions",
    "- 18 alerts",
    "",
    "The raw dataset is not included because of its size.",
    "",
    "See `data/README.md`.",
    "",
    "## Experimental Result",
    "",
    "In the evaluated AMLSim dataset, TraceGraph identified all "
    "18 fraud-labelled transactions as high-risk.",
    "",
    "This is a dataset-specific experimental result and should "
    "not be interpreted as proof of real-world AML detection "
    "performance.",
    "",
    "The final investigation stage also identified two "
    "suspicious temporal cycles.",
    "",
    "## Repository Structure",
    "",
    "```text",
    "TraceGraph/",
    "├── README.md",
    "├── LICENSE",
    "├── .gitignore",
    "├── requirements.txt",
    "├── data/",
    "├── notebooks/",
    "├── src/",
    "├── dashboard/",
    "├── models/",
    "├── results/",
    "├── docs/",
    "└── screenshots/",
    "```",
    "",
    "## Notebook",
    "",
    "The complete experimental pipeline is available at:",
    "",
    "`notebooks/TraceGraph_AML.ipynb`",
    "",
    "## Dashboard",
    "",
    "Run the Streamlit dashboard using:",
    "",
    "```bash",
    "streamlit run dashboard/app.py",
    "```",
    "",
    "## Research Scope",
    "",
    "TraceGraph is an AML investigation-support and "
    "risk-prioritization framework.",
    "",
    "Risk scores and detected patterns are intended to support "
    "further investigation and human/compliance review rather "
    "than independently determine whether a transaction "
    "constitutes financial crime.",
    "",
    "## License",
    "",
    "MIT License"
]

(GITHUB / "README.md").write_text(
    "\n".join(readme_lines) + "\n",
    encoding="utf-8"
)

print("✓ README.md")

# ------------------------------------------------------------
# 13. PLACEHOLDER FILES
# ------------------------------------------------------------

(GITHUB / "src" / ".gitkeep").write_text("", encoding="utf-8")
(GITHUB / "screenshots" / ".gitkeep").write_text("", encoding="utf-8")

# ------------------------------------------------------------
# 14. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 2 COMPLETE")
print("=" * 60)

file_count = 0
total_size = 0

for file in GITHUB.rglob("*"):

    if file.is_file():

        file_count += 1
        total_size += file.stat().st_size

print(f"\nGitHub-ready project:")
print(GITHUB)

print(f"\nFiles created/copied: {file_count}")
print(f"Total size: {total_size / (1024 * 1024):.2f} MB")

print("\nOriginal project remains untouched:")
print(SOURCE)

print("\n✓ Cleaned notebook included")
print("✓ Results included")
print("✓ Selected models included")
print("✓ Dashboard included")
print("✓ README created")
print("✓ requirements.txt created")
print("✓ .gitignore created")
print("✓ Dataset README created")
print("✓ Architecture documentation created")
print("✓ Original project NOT modified")

print("\nDO NOT PUSH TO GITHUB YET.")
print("Wait for Step 3.")